# Powderday flux catalogs — quenched galaxies in the high-res 25 Mpc box

**Goal.** Multi-aperture, dusty vs dust-free photometric catalogs (fluxes **with errors**) for
**quenched** galaxies in SIMBA high-res **m25n512** (`cis25`) at the `TARGET_REDSHIFTS`
anchors **z ≈ 0.3, 0.7, 1.0, 1.5** (snapshots 134/116/105/078), and a CIGALE fit of those
mocks in which every stellar-population parameter is **pinned per object** so that the dust
attenuation $A_V$ is the measurand rather than one free parameter among thousands.

**Sample (per anchor snapshot).** `log10 M* > 10`, **passive** by the 0.2/τ criterion
(sSFR < 0.2/t_H at the anchor), and **> 20 gas particles** (plus the usual ≥ 20 star-particle floor).
The sample is split by **weak vs strong AGN feedback over the quench window**: the AGN–ISM coupling
strength `xcoup_hist` (jet-mode strength gated by gas-poorness, §8j physics) averaged between each
galaxy's **SFT and QT** (1/t and 0.2/t crossings from `find_quenching_times`); classes: **strong** = fully coupled (`xstr_quench` = 1) through the window, **weak** = bottom tercile of the remainder, **intermediate** = the rest.

**Pipeline** (same skeleton as `test_powderday.ipynb`, selection machinery from
`quench_mode_vs_sigma_gas.ipynb`):

| Part | What | Where |
|---|---|---|
| 1 | anchors + gated `BUILD_MULTI_Z` / `BUILD_BH` history builds | cluster |
| 2–3 | selection, SFT/QT, AGN split, **sample statistics** | anywhere (needs the HDF5s) |
| 4 | Stage 0 — per-galaxy particle files | cluster |
| 4b | annulus sampling QC — star/gas/dust counts per projected annulus × sightline | anywhere (needs Stage 0) |
| 5 | Stage 1 — selection HDF5 + Slurm masters (dust_on / dust_off / agn_on) → run RT | cluster |
| 6 | aperture QC on the first `.rtout.sed` | cluster |
| 7 | Stage 2 — per-aperture flux extraction → **one catalog per aperture per RT arm** | cluster |
| 7a | the **true** $A_V = -2.5\log_{10}(F_{\rm on}/F_{\rm off})$ vs the ISM — no CIGALE | anywhere |
| 7b | observed-frame CIGALE input catalogs | anywhere |
| 7c | aperture-matched **formed-mass** SFH archive (the injected prior) | cluster |
| 7d | **one CIGALE run per object** (SFH + $Z$ + age pinned) + the chunked job array | cluster |
| 7e | aperture-matched SIMBA truth | cluster |
| 7f | merged fits vs truth: **is $A_V$ recovered?** | anywhere |

**Apertures & sightlines (mock observation).** Stage 0 cuts a **100 pkpc spherical region**
around each galaxy (everything: CGM, satellites, projected neighbours — a true mock aperture,
not just member particles); the RT grid spans ±100 kpc (`zoom_box_len`). Hyperion log-spaces
`N_AP = 5` projected apertures 1→100 kpc — the 10^(k/2) ladder **1, 3.16, 10, 31.6, 100 kpc**
(central → outskirts), all extracted. Each SED is peeled along **4 sightlines**
(θ,φ) = (0,0), (45,90), (90,180), (135,270) deg — one catalog per (RT arm, aperture,
inclination). `N_AP/AP_MIN_KPC/AP_MAX_KPC` and `THETA_DEG/PHI_DEG` below must match the
parameter masters that the RT jobs copy. **Requires the one-time powderday patch documented
before Stage 1 (already applied on this cluster's install).**

**Flux errors.** Hyperion's Monte-Carlo SED uncertainty, read with
`get_sed(..., uncertainties=True)` and propagated through the filter convolution
(`<filter>_err` columns; NaN if a run stored no uncertainties).

# Part 0 — Setup & configuration

In [ ]:
import os
import gc
import glob
import json
import re
import subprocess
import warnings
import numpy as np
import h5py
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.table import Table, vstack, join
from astropy import units as u
from astropy.cosmology import Planck15 as COSMO   # matches the quenching machinery

from simbanator.io.simba import Simulation
from simbanator.analysis import HDF5BuildHistory, caesar_read_progen
from simbanator.analysis.quenching import find_quenching_times
from simbanator.utils.geometry import sightline_unit_vectors, projected_radius

# ── simulation ────────────────────────────────────────────────────────────────
SIM_NAME = "cis25"        # SIMBA high-res 25 Mpc/h box (m25n512); must exist in ~/.simbanator/config.json
try:
    sim = Simulation(SIM_NAME)
except KeyError as e:
    raise KeyError(
        f"'{SIM_NAME}' is not registered in ~/.simbanator/config.json on this machine.\n"
        "Register it once (adjust paths to where the 25 Mpc snapshots+catalogs live):\n"
        "  from simbanator.io.config import add_simulation\n"
        "  add_simulation('cis25', data_dir='<...>/SIMBA_25/s25',\n"
        "                 catalog_dir='<...>/SIMBA_25/s25/Groups',\n"
        "                 file_format='m25n512_{snap:03d}.hdf5')\n"
        "then add \"snap_z_map\": \"zsnap_map_caesar_box100.txt\" to that entry "
        "(SIMBA boxes share the snapshot schedule)."
    ) from e
if sim.scale_factors is None:
    raise ValueError(f"'{SIM_NAME}' config has no snap_z_map — add "
                     '"snap_z_map": "zsnap_map_caesar_box100.txt" to its entry in ~/.simbanator/config.json')

# filtered-particle filename prefix (Stage 0 == Stage 1, never let them drift)
PARTICLE_PREFIX = sim.file_format.split("_{")[0]        # 'm25n512'

# ── selection: quenched + massive + realistically gas-populated ───────────────
TARGET_REDSHIFTS = [0.3, 0.7, 1.0, 1.5, 2]   # z=2 (snap 78) dropped 2026-08-03: 8-gal sample too small for the class split
MASS_FLOOR       = 10.0        # log10(M*/Msun) > 10
PASSIVE_FACTOR   = 0.2         # passive if sSFR < 0.2 / t_H  (== the QT threshold of find_quenching_times)
NGAS_MIN         = 21          # STRICTLY > 20 gas particles at the anchor
NSTAR_MIN        = 20          # star-particle floor (same as quench_mode_vs_sigma_gas)
DUST_TO_H2_MIN   = 1e-4        # keep only M_dust/M_H2 >= 1e-4 at the anchor (drops the dust-poor half; 2026-08-05)

# ── AGN / coupling constants (identical to quench_mode_vs_sigma_gas §0) ──────
JET_LOGMBH    = 7.5            # jet mode: log10(M_BH) > 7.5 ...
JET_FEDD      = 0.2            #           ... AND f_Edd < 0.2
XRAY_FEDD_MAX = 0.02           # (kept for reference; xcoup uses the f_gas gate)
XRAY_FGAS_MAX = 0.2            # coupling gate: f_gas = Mgas/M* < 0.2
GYR = 1e9

# ── history tracking ──────────────────────────────────────────────────────────
TRACK_AGE_FRAC      = 0.09     # track back to ~this fraction of the cosmic age at selection
ANCHOR_END_OVERRIDE = {}
CORRUPT_SNAPS       = set()

# ── heavy-build gates (set True on the cluster, then reuse the cached HDF5s) ──
BUILD_MULTI_Z = False          # per-anchor progenitor FITS + property history HDF5
BUILD_BH      = False          # per-anchor BH (mass / mdot / f_Edd) history HDF5

# ── apertures (MUST match SED_APERTURE_* in simbanator/sed/parameters_master*.py) ──
N_AP       = 5           # SED_APERTURE_NAP: 10^(k/2) ladder -> 1, 3.16, 10, 31.6, 100 kpc
AP_MIN_KPC = 1.0
AP_MAX_KPC = 100.0
APERTURE_RADII_KPC = np.geomspace(AP_MIN_KPC, AP_MAX_KPC, N_AP)
# central -> outskirts; ALL rungs are extracted (nominal labels, true radii above)
TARGET_AP_KPC   = [1, 3, 10, 32, 100]
WANTED_AP_IDX   = list(range(N_AP))
APERTURE_LABELS = [f"ap{t:g}kpc" for t in TARGET_AP_KPC]
# annulus edges between consecutive rungs; the OUTER rung names the annulus
# (ann1kpc = the 0->1 kpc disc == ap1kpc) — shared by Parts 4b/4c/7a/7c/7f
R_EDGES        = np.concatenate([[0.0], APERTURE_RADII_KPC])   # [pkpc]
ANNULUS_LABELS = [l.replace("ap", "ann") for l in APERTURE_LABELS]

# ── viewing angles (MUST match THETA/PHI in the parameter masters) ──
THETA_DEG   = [0, 45, 90, 135]
PHI_DEG     = [0, 90, 180, 270]
N_INCL      = len(THETA_DEG)
INCL_LABELS = [f"i{t:g}p{p:g}" for t, p in zip(THETA_DEG, PHI_DEG)]   # i0p0, i45p90, ...
NHAT        = sightline_unit_vectors(THETA_DEG, PHI_DEG)      # LOS unit vectors

# ── Stage-0 region cutout: EVERYTHING (CGM, satellites) within this proper radius ──
# sphere radius = zoom_box_len = largest aperture (100 kpc): the grid's inscribed sphere
# is fully populated; only the outermost aperture is slightly depth-truncated at its edge
R_CUTOUT_KPC = 100.0

# ── powderday run layout (same conventions as test_powderday.ipynb) ──────────
GVFS_BASE   = ''
# '+' not os.path.join: with GVFS_BASE='' this must stay ABSOLUTE (see test_powderday)
REMOTE_HOME = GVFS_BASE + "/mnt/home/glorenzon/analize_simba_cgm"

hydro_dir_base = os.path.join(os.getcwd(), 'output', sim.name, 'filtered_particles')
selection_file = 'selection_m25_quenched'                  # MakeSED appends '.h5'
sed_output_dir = os.path.join(REMOTE_HOME, 'output', sim.name, 'sed_quenched_regions')

RUNS = {
    'dust_on':  dict(run_tag='dusty_simdust', paramf='parameters_master.py'),
    'dust_off': dict(run_tag='nodust_1e-12',  paramf='parameters_master-nodust.py'),
    # dust_on + AGN point sources (BH_SED=True, Hopkins+2007 template, BH_var=False:
    # L_bol = 0.1*BH_Mdot*c^2 from the SIMBA accretion rates). Needs PartType5 in the
    # Stage-0 cutouts -> re-run Part 4 (EXTRACT_OVERWRITE=True) before Stage 1.
    'agn_on':   dict(run_tag='dusty_simdust_agn', paramf='parameters_master-agn.py'),
}

# ── local output tree ─────────────────────────────────────────────────────────
OUT      = os.path.join(os.getcwd(), "output", SIM_NAME)
SFHDIR   = os.path.join(OUT, "caesar_sfh")
TABLEDIR = os.path.join(OUT, "tables")
PLOTDIR  = os.path.join(OUT, "plots", "powderday_quenched")
CATDIR   = os.path.join(OUT, "sed_aperture_catalogs")
for _d in (SFHDIR, TABLEDIR, PLOTDIR, CATDIR):
    os.makedirs(_d, exist_ok=True)
SELECTION_FITS = os.path.join(TABLEDIR, "powderday_quenched_selection.fits")
AV_DUSTY = 0.1     # global A_V above which a galaxy counts as 'dusty' (Parts 4b/7a/7g)

# ── CIGALE tree (Parts 7b-7f) ──
CIGALE_DIR   = os.path.join(CATDIR, "cigale")   # CIGALE input files (Part 7b)
# Part 7d writes ONE run dir per object under RUN_BASE_PIN; RUN_BASE is the
# pre-2026-08-10 tree (SFH families / Z groups), kept on disk for comparison.
RUN_BASE     = os.path.join(OUT, "cigale_runs")
RUN_BASE_PIN = os.path.join(OUT, "cigale_runs_pinned")

def _ztag(z):
    return ("z%g" % z).replace(".", "p")

print(f"sim={sim.name}  data_dir={sim.data_dir}")
print(f"prefix={PARTICLE_PREFIX}  anchors z={TARGET_REDSHIFTS}")
print("aperture ladder [kpc]:", np.round(APERTURE_RADII_KPC, 2))
print("extracted rungs:", {l: f"{APERTURE_RADII_KPC[i]:.3g} kpc (idx {i})"
                           for l, i in zip(APERTURE_LABELS, WANTED_AP_IDX)})
print("sightlines:", INCL_LABELS, "  region cutout:", R_CUTOUT_KPC, "pkpc")
print("SED output:", sed_output_dir)

# Part 0b — shared helpers

Small loaders used by several parts, so each part stays runnable in a fresh session after
Parts 0/0b: the selection catalog, `(snap, gal_id)`-keyed alignment, the RT-grid centres
(Stage-1 selection HDF5, caesar fallback — identical values), the Stage-0 cutout reader
(code units → proper kpc about a given centre), anchor-epoch (row 0) history values, and
the Part 7a dusty flag.


In [ ]:
# ── shared helpers: selection, alignment, centres, cutouts, row-0 histories ──
def load_selection():
    """SELECTION_FITS (written by Part 3) -> (table, snap array, gal_id array)."""
    sel = Table.read(SELECTION_FITS)
    return sel, np.asarray(sel["snap"], int), np.asarray(sel["gal_id"], int)

def by_snap_gal(db, snaps, gids, default=np.nan):
    """Align a {(snap, gal_id): value} dict to (snaps, gids) rows -> array."""
    return np.array([db.get((int(s), int(g)), default)
                     for s, g in zip(snaps, gids)])

def rt_centers(snaps, gids):
    """RT-grid centres (code units): Stage-1 selection h5, else caesar (identical values)."""
    from simbanator.sed.makesed import read_selection_centers
    selh5 = os.path.join(sed_output_dir, RUNS["dust_on"]["run_tag"],
                         "target_selection", selection_file + ".h5")
    cen = read_selection_centers(selh5)
    if cen:
        print(f"grid centres from the Stage-1 selection h5 ({len(cen)} galaxies)")
        return cen
    print(f"[fallback] {selh5} missing -> reading centres from the caesar catalogs")
    for s in np.unique(snaps):
        cs = sim.load_catalog(snap=int(s))
        for g in np.unique(np.asarray(gids)[np.asarray(snaps) == s]):
            cen[(int(s), int(g))] = cs.galaxies[int(g)].pos.in_units("code_length").value
        del cs
        gc.collect()
    return cen

def cutout_file(snap, gid):
    """Path of one Stage-0 per-galaxy particle cutout."""
    return os.path.join(hydro_dir_base, f"snap_{int(snap):03d}",
                        f"{PARTICLE_PREFIX}_snap{int(snap):03d}_gal{int(gid):06d}.h5")

def read_cutout(snap, gid, center_code, ptype, fields=()):
    """One Stage-0 cutout particle type -> dict(pos [proper kpc], a, h, <fields> raw).

    `center_code` (code units, ckpc/h) is subtracted before the a/h conversion,
    so `pos` is proper kpc about that centre. Requested `fields` are returned
    raw (code units); a field absent from the file comes back as None.
    Returns None if the centre, the cutout file or the particle type is missing.
    """
    pf = cutout_file(snap, gid)
    if center_code is None or not os.path.exists(pf):
        return None
    with h5py.File(pf, "r") as f:
        if ptype not in f:
            return None
        a  = float(f["Header"].attrs["Time"])
        hh = float(f["Header"].attrs["HubbleParam"])
        out = dict(a=a, h=hh,
                   pos=(np.asarray(f[f"{ptype}/Coordinates"][:], float)
                        - center_code) * a / hh)
        for fld in fields:
            out[fld] = np.asarray(f[f"{ptype}/{fld}"][:]) if fld in f[ptype] else None
    return out

def anchor_row0(keys):
    """Anchor-epoch (row 0) values from every history under SFHDIR.

    -> {(anchor_snap, gal_id): {key: value, 'z0': anchor redshift}};
    keys absent from a history are simply missing from its dicts.
    """
    db = {}
    for hf in sorted(glob.glob(os.path.join(SFHDIR, "history_anchor_*.hdf5"))):
        with h5py.File(hf, "r") as f:
            snap0 = int(f["metadata/snapshots"][0])
            gid   = np.asarray(f["metadata/galaxy_ids"][:], int)
            z0    = float(f["redshift/Redshift"][0])
            row0  = {k: f[f"properties/{k}"][0] for k in keys
                     if f"properties/{k}" in f}
        for j, g in enumerate(gid):
            db[(snap0, int(g))] = {k: float(v[j]) for k, v in row0.items()}
            db[(snap0, int(g))]["z0"] = z0
    return db

def row0_arr(db, snaps, gids, key):
    """anchor_row0 value `key` aligned to (snaps, gids) rows (NaN where absent)."""
    return np.array([db.get((int(s), int(g)), {}).get(key, np.nan)
                     for s, g in zip(snaps, gids)])

def dusty_flags(snaps, gids):
    """Part 7a global A_V aligned to (snaps, gids) -> (A_V array, dusty flag array).

    dusty: 1 (A_V > AV_DUSTY), 0 (transparent), -1 (not measured yet — run
    Part 7a, then re-run the caller to get the dusty/non-dusty split).

    This is Part 7a's ONE-SIGHTLINE global A_V (ATTEN_INCL = INCL_LABELS[0], the
    fiducial aperture) — a coarse per-galaxy label for splitting figures. Part 7f
    builds its own A_V per (aperture, sightline) from the same catalogs; do not
    confuse the two.
    """
    avf = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")
    db = {}
    if os.path.exists(avf):
        at = Table.read(avf)
        db = {(int(s), int(g)): float(a)
              for s, g, a in zip(at["snap"], at["gal_id"], at["A_V"])}
    else:
        print(f"[dusty split] {os.path.basename(avf)} not found — run Part 7a first")
    av = by_snap_gal(db, snaps, gids)
    return av, np.where(np.isnan(av), -1, (av > AV_DUSTY).astype(int))


# Part 1 — Anchors & gated cluster builds

Each anchor (z ≈ 0.3, 0.6, 0.7, 1.0, 2.0 → nearest snapshot) gets its **own** progenitor table +
property history with that snapshot as row 0, and a BH history aligned to the same rows — exactly
the `quench_mode_vs_sigma_gas.ipynb` machinery, pointed at `cis25`. Histories are pre-selected to
**massive + passive** at the anchor (the gas/star floors are applied later so the statistics can
count them).

In [ ]:
# ── anchor table: snapshot, track end, per-anchor product paths ──
_sall, _zall = [], []
for _s in range(0, 152):
    try:
        _zv = float(sim.get_z_from_snap(_s))
    except Exception:
        continue
    if np.isfinite(_zv) and _zv >= 0:
        _sall.append(_s); _zall.append(_zv)
_sall, _zall = np.asarray(_sall), np.asarray(_zall)
_aall = COSMO.age(_zall).value

ANCHORS = {}
for _zt in TARGET_REDSHIFTS:
    _snap = int(_sall[np.argmin(np.abs(_zall - _zt))])
    _age_end = TRACK_AGE_FRAC * float(_aall[_sall == _snap][0])
    _end = int(ANCHOR_END_OVERRIDE.get(_zt, int(_sall[np.searchsorted(_aall, _age_end)])))
    _tag = _ztag(_zt)
    ANCHORS[_zt] = dict(z_target=_zt, tag=_tag, snap=_snap,
                        z=float(sim.get_z_from_snap(_snap)), end_snap=_end,
                        prog_file=f"progenitors_anchor_{_tag}.fits",
                        hist_path=os.path.join(SFHDIR, f"history_anchor_{_tag}.hdf5"),
                        bh_path=os.path.join(SFHDIR, f"bh_history_anchor_{_tag}.hdf5"))

print(f"{'z_tgt':>6s} {'snap':>5s} {'z':>7s} {'end':>5s} {'hist':>6s} {'BH':>4s}")
for _zt, A in ANCHORS.items():
    print(f"{_zt:6.1f} {A['snap']:5d} {A['z']:7.3f} {A['end_snap']:5d} "
          f"{'ok' if os.path.exists(A['hist_path']) else '--':>6s} "
          f"{'ok' if os.path.exists(A['bh_path']) else '--':>4s}")

In [ ]:
# ── property list tracked per anchor (superset of what selection + coupling need) ──
PROPS = {
    "galaxy_data": [
        "masses.stellar", "sfr", "masses.gas", "masses.dust", "masses.H2", "masses.HI",
        "radii.stellar_half_mass", "radii.gas_half_mass",
        "pos", "ngas", "nstar", "ages.mass_weighted",
    ],
    "halo_data": ["masses.total"],
}

# ── GATED (cluster): per-anchor progenitor table + property history ──
# Verbatim port of quench_mode_vs_sigma_gas 1z·build, with the (stricter) M*>10 pre-selection.
if BUILD_MULTI_Z:
    for _zt, A in ANCHORS.items():
        if os.path.exists(A["hist_path"]):
            print(f"[{A['tag']}] cached -> {os.path.basename(A['hist_path'])}"); continue
        end = int(A["end_snap"])
        while end < A["snap"] and (end in CORRUPT_SNAPS or not os.path.exists(sim.get_caesar_file(end))):
            end += 1
        A["end_snap"] = end
        print(f"[{A['tag']}] anchor snap {A['snap']} (z={A['z']:.2f}) <- {end}: progenitor table ...")
        cs_a = sim.load_catalog(snap=A["snap"])
        caesar_read_progen([g.GroupID for g in cs_a.galaxies], A["prog_file"],
                           range(end, A["snap"] + 1), sim, output_dir=None)
        hist = HDF5BuildHistory(sim, cs_a, progfilename=A["prog_file"])
        with fits.open(hist.progen_file) as hdul:
            valid_ids = np.asarray(hdul[1].data["GroupID"])
            _tHa = COSMO.age(float(A["z"])).value * 1e9
            _gid = np.array([g.GroupID for g in cs_a.galaxies])
            _ms  = np.array([float(g.masses["stellar"]) for g in cs_a.galaxies])
            _sf  = np.array([float(g.sfr) for g in cs_a.galaxies])
            with np.errstate(all="ignore"):
                _ss = np.where(_ms > 0, _sf / _ms, np.nan)
                _ok = (np.log10(np.where(_ms > 0, _ms, np.nan)) > MASS_FLOOR) & (_ss < PASSIVE_FACTOR / _tHa)
            _keep = {int(g) for g in _gid[_ok]}
            valid_ids = np.asarray([i for i in valid_ids if int(i) in _keep], dtype=valid_ids.dtype)
            print(f"  [pre-select] {len(valid_ids)}/{len(_gid)} massive+passive at z={A['z']:.2f}")
        hist.get_history_indx(valid_ids, A["snap"], end)
        props_try = {k: list(v) for k, v in PROPS.items()}
        while True:   # drop-and-retry: some catalog versions miss some fields
            try:
                hist.get_property_history(props_try, verbose=0); break
            except KeyError as e:
                msg = str(e); dropped = False
                for fam, plist in props_try.items():
                    for pr in list(plist):
                        if pr in msg or pr.split("/")[-1] in msg:
                            plist.remove(pr); print("  [drop]", pr); dropped = True
                if not dropped:
                    raise
        hist.save_history_to_hdf5(os.path.basename(A["hist_path"]))
        del cs_a, hist; gc.collect()
        print(f"[{A['tag']}] history -> {A['hist_path']}")
else:
    print("BUILD_MULTI_Z=False -> expecting per-anchor histories under", SFHDIR)

In [ ]:
# ── loaders (verbatim from quench_mode_vs_sigma_gas): row 0 = the anchor epoch ──
def load_anchor_history(A):
    """Load one anchor's history -> dict(galaxy_ids, snaps_arr, redshift, t_cosmic_yr, P)."""
    H = {"P": {}}
    with h5py.File(A["hist_path"], "r") as f:
        H["galaxy_ids"] = f["metadata/galaxy_ids"][:]
        H["snaps_arr"]  = f["metadata/snapshots"][:]
        H["redshift"]   = f["redshift/Redshift"][:]
        f["properties"].visititems(
            lambda name, obj: H["P"].__setitem__(name, obj[:]) if isinstance(obj, h5py.Dataset) else None)
    H["t_cosmic_yr"] = COSMO.age(H["redshift"]).value * 1e9
    return H

def build_prog_index(A, galaxy_ids, snaps_arr):
    """(n_snap, n_gal) catalogue group-index matrix aligned to the anchor history rows."""
    cs0 = sim.load_catalog(snap=A["snap"])
    hP = HDF5BuildHistory(sim, cs0, progfilename=A["prog_file"])
    hP.get_history_indx(galaxy_ids, int(np.max(snaps_arr)), int(np.min(snaps_arr)))
    M = np.vstack([hP.history_indx[str(s)] for s in snaps_arr])
    del cs0, hP; gc.collect()
    return M

In [ ]:
# ── BH history: per-anchor build (GATED) + loader (verbatim quench_mode §4b) ──
BH_CANDIDATES = {"bh_mass": ["masses.bh", "masses.bh_mass", "bhmass"],
                 "bh_mdot": ["bhmdot", "bh_mdot"],
                 "bh_fedd": ["bh_fedd", "bhfedd", "fedd"]}

def _resolve_bh_path(f, cands):
    for c in cands:
        for p in (f"galaxy_data/dicts/{c}", f"galaxy_data/{c}"):
            if p in f:
                return p
    return None

def build_bh_for_anchor(A, galaxy_ids, snaps_arr, n_gal):
    pidx = build_prog_index(A, galaxy_ids, snaps_arr)
    n_snap = len(snaps_arr)
    BH = {k: np.full((n_snap, n_gal), np.nan) for k in BH_CANDIDATES}
    for ri, snap in enumerate(snaps_arr):
        snap = int(snap)
        if snap in CORRUPT_SNAPS:
            continue
        try:
            with h5py.File(sim.get_caesar_file(snap), "r") as f:
                valid = np.isfinite(pidx[ri]); cv = np.where(valid)[0]
                vi = pidx[ri][valid].astype(int)
                for k, cands in BH_CANDIDATES.items():
                    p = _resolve_bh_path(f, cands)
                    if p is not None:
                        BH[k][ri, cv] = f[p][:][vi]
        except (OSError, KeyError) as e:
            print(f"  [skip] snap {snap}: {type(e).__name__}"); CORRUPT_SNAPS.add(snap)
    with h5py.File(A["bh_path"], "w") as f:
        for k, arr in BH.items():
            f.create_dataset(k, data=arr)
    print(f"[{A['tag']}] BH history -> {A['bh_path']}")
    return BH

def load_bh(bh_hist_path):
    with h5py.File(bh_hist_path, "r") as f:
        return {k: f[k][:] for k in f.keys()}

if BUILD_BH:
    for _zt, A in ANCHORS.items():
        if os.path.exists(A["bh_path"]):
            print(f"[{A['tag']}] cached -> {os.path.basename(A['bh_path'])}"); continue
        if not os.path.exists(A["hist_path"]):
            print(f"[{A['tag']}] no history yet -> run BUILD_MULTI_Z first"); continue
        _H = load_anchor_history(A)
        build_bh_for_anchor(A, _H["galaxy_ids"], _H["snaps_arr"], len(_H["galaxy_ids"]))
        del _H; gc.collect()
else:
    print("BUILD_BH=False -> expecting per-anchor BH histories under", SFHDIR)

# Part 2 — Selection, quench events (SFT/QT) & the weak/strong AGN split

- **Selection** (at row 0 = the anchor): `log10 M* > 10`, passive (`sSFR < 0.2/t_H`), `ngas > 20`,
  `nstar ≥ 20`.
- **SFT/QT** per galaxy from `find_quenching_times` on the tracked sSFR history (SFT = crossing
  below 1/t, QT = subsequent crossing below 0.2/t with persistence); the **last** event is kept.
- **AGN split**: `xstr_quench` = mean of `xcoup_hist` (jet strength `clip(log10(0.2/f_Edd),0,1)`
  for `log M_BH > 7.5`, gated by `f_gas < 0.2`) over snapshots with `t_SFT ≤ t ≤ t_QT`; if the
  window is narrower than the snapshot spacing, the finite snapshot nearest SFT is used.
  **strong = fully coupled** (`xstr_quench` = 1 through the window; plain terciles degenerate
  into this tie-clump — 15–20 gals per anchor sit exactly at 1); **weak** = bottom tercile of the
  non-saturated remainder (per anchor); `intermediate` = the rest; no finite coupling = `no_AGN`;
  no detected quench event = `no_event`.

In [ ]:
# ── selection mask at the anchor epoch (row 0) ──
def selection_mask(P, t_cosmic_yr):
    mstar0 = P["masses.stellar"][0]
    sfr0   = P["sfr"][0]
    ngas0  = P["ngas"][0]
    nstar0 = P["nstar"][0] if "nstar" in P else np.full_like(mstar0, np.inf)
    _has_dh2 = ("masses.dust" in P) and ("masses.H2" in P)
    if not _has_dh2:
        print("[selection_mask] WARNING: masses.dust/masses.H2 missing from history -> dust/H2 cut skipped")
    mdust0 = P["masses.dust"][0] if _has_dh2 else None
    mh2_0  = P["masses.H2"][0]   if _has_dh2 else None
    with np.errstate(all="ignore"):
        ssfr0 = np.where(mstar0 > 0, sfr0 / mstar0, np.nan)
        cuts = {
            "massive":  np.log10(np.where(mstar0 > 0, mstar0, np.nan)) > MASS_FLOOR,
            "passive":  ssfr0 < (PASSIVE_FACTOR / t_cosmic_yr[0]),
            "gas>20":   ngas0 >= NGAS_MIN,
            "star>=20": nstar0 >= NSTAR_MIN,
            # multiplicative form so M_H2=0 rows pass instead of dividing by zero
            "dust/H2":  (mdust0 >= DUST_TO_H2_MIN * mh2_0) if _has_dh2
                        else np.ones_like(mstar0, dtype=bool),
        }
    m = (cuts["massive"] & cuts["passive"] & cuts["gas>20"] & cuts["star>=20"]
         & cuts["dust/H2"])
    return m, cuts

# ── SFT/QT per selected galaxy (trimmed from quench_mode build_records) ──
def quench_records(P, t_cosmic_yr, redshift, galaxy_ids, cols):
    """One record per selected column; galaxies without a detected quench event keep NaN times."""
    records = []
    for col in np.asarray(cols, int):
        gid = galaxy_ids[col]
        mstar = P["masses.stellar"][:, col]; sfr = P["sfr"][:, col]
        with np.errstate(all="ignore"):
            ssfr = np.where(mstar > 0, sfr / mstar, np.nan)
        valid = np.isfinite(ssfr) & (ssfr > 0) & np.isfinite(t_cosmic_yr)
        rec = dict(gid=int(gid), col=int(col), t_sft=np.nan, t_qt=np.nan,
                   tau_q=np.nan, tau_q_over_tH=np.nan, z_qt=np.nan)
        if valid.sum() >= 5:
            t = t_cosmic_yr[valid]; s = ssfr[valid]
            o = np.argsort(t); t, s = t[o], s[o]
            tu, ui = np.unique(t, return_index=True); su = s[ui]
            if len(tu) >= 5:
                qts, sfts, _, dbg = find_quenching_times(
                    tu, su, galaxy_id=int(gid), plot=False, save_fits_path=None, return_debug=True)
                if len(qts):
                    k = int(np.argmax(qts))                     # last (surviving) quench event
                    rec["t_qt"], rec["t_sft"] = float(qts[k]), float(sfts[k])
                    rec["tau_q"] = rec["t_qt"] - rec["t_sft"]
                    z_qt = float(np.interp(rec["t_qt"], t_cosmic_yr[::-1], redshift[::-1]))
                    rec["z_qt"] = z_qt
                    rec["tau_q_over_tH"] = rec["tau_q"] / (COSMO.age(z_qt).value * 1e9)
        records.append(rec)
    return records

In [ ]:
# ── AGN–ISM coupling over the quench window [SFT, QT] (physics verbatim from §8j build_coupling) ──
def coupling_quench_window(BH, P, records, t_cosmic_yr):
    _ord = np.argsort(t_cosmic_yr); t_inc = t_cosmic_yr[_ord]
    with np.errstate(all="ignore"):
        fgas_hist = np.where(P["masses.stellar"] > 0, P["masses.gas"] / P["masses.stellar"], np.nan)
        _bh_ok  = np.isfinite(BH["bh_mass"]) & np.isfinite(BH["bh_fedd"])
        _mbh_ok = BH["bh_mass"] > 10 ** JET_LOGMBH
        wjet_hist = np.where(_bh_ok, np.where(_mbh_ok,
                             np.clip(np.log10(JET_FEDD / np.clip(BH["bh_fedd"], 1e-12, None)), 0.0, 1.0),
                             0.0), np.nan)
        xcoup_hist = np.where(np.isfinite(wjet_hist) & np.isfinite(fgas_hist),
                              wjet_hist * (fgas_hist < XRAY_FGAS_MAX).astype(float), np.nan)
    n = len(records)
    xstr_q = np.full(n, np.nan)
    for i, r in enumerate(records):
        if not (np.isfinite(r["t_sft"]) and np.isfinite(r["t_qt"])):
            continue                                   # no quench event -> stays NaN ('no_event')
        cs = xcoup_hist[_ord, r["col"]].astype(float)
        fin = np.isfinite(cs)
        win = (t_inc >= r["t_sft"]) & (t_inc <= r["t_qt"]) & fin
        if not win.any() and fin.any():
            # quench window narrower than the snapshot spacing -> nearest finite snapshot to SFT
            j = np.where(fin)[0]
            win = np.zeros_like(fin); win[j[np.argmin(np.abs(t_inc[j] - r["t_sft"]))]] = True
        if win.any():
            xstr_q[i] = np.nanmean(cs[win])
    bx = np.isfinite(xstr_q)
    # Physical classes (2026-08-03): xstr_quench piles up at exactly 1.0 (fully
    # coupled through the whole quench window; 15-20 gals per anchor), so plain
    # terciles degenerate into the tie-clump at 1. Instead: strong = the fully
    # coupled clump, weak = bottom tercile of the non-saturated remainder,
    # intermediate = the rest. xstr_quench stays continuous in the catalogs.
    XSTR_FULL_EPS = 1e-6
    strong = bx & (xstr_q >= 1.0 - XSTR_FULL_EPS)
    weak = np.zeros(n, bool); lo_q = np.nan; hi_q = 1.0
    _rest = bx & ~strong
    if _rest.sum() >= 3:
        lo_q = float(np.nanquantile(xstr_q[_rest], 1.0 / 3.0))
        weak = _rest & (xstr_q <= lo_q)
    inter = bx & ~strong & ~weak
    no_fb = ~bx
    return dict(xstr_quench=xstr_q, strong=strong, weak=weak, inter=inter, no_fb=no_fb,
                tercile=(lo_q, hi_q))

def agn_class_labels(CO, records):
    """Per-record string label; galaxies without a quench event are 'no_event'."""
    n = len(records)
    has_event = np.array([np.isfinite(r["t_sft"]) and np.isfinite(r["t_qt"]) for r in records])
    lab = np.array(["unclassified"] * n, dtype=object)
    if CO is not None:
        lab[CO["no_fb"]] = "no_AGN"
        lab[CO["inter"]] = "intermediate"
        lab[CO["weak"]]  = "weak"
        lab[CO["strong"]] = "strong"
    lab[~has_event] = "no_event"
    return lab

In [ ]:
# ── driver: per anchor -> selection, records, coupling, labels ──
RESULTS = {}
for _zt, A in ANCHORS.items():
    if not os.path.exists(A["hist_path"]):
        print(f"[{A['tag']}] MISSING history -> run BUILD_MULTI_Z on the cluster; skipped")
        continue
    H = load_anchor_history(A)
    m, cuts = selection_mask(H["P"], H["t_cosmic_yr"])
    cols = np.where(m)[0]
    recs = quench_records(H["P"], H["t_cosmic_yr"], H["redshift"], H["galaxy_ids"], cols)
    BH = load_bh(A["bh_path"]) if os.path.exists(A["bh_path"]) else None
    CO = coupling_quench_window(BH, H["P"], recs, H["t_cosmic_yr"]) if BH is not None else None
    labels = agn_class_labels(CO, recs)
    if BH is None:
        print(f"[{A['tag']}] WARNING: no BH history -> AGN split = 'unclassified' (run BUILD_BH)")
    RESULTS[_zt] = dict(A=A, H=H, mask=m, cuts=cuts, cols=cols, records=recs, CO=CO, labels=labels)
    n_ev = int(np.isfinite([r["t_qt"] for r in recs]).sum())
    print(f"[{A['tag']}] snap {A['snap']} (z={A['z']:.3f}): pool={m.size} "
          f"selected={len(cols)} with_event={n_ev} "
          f"classes={dict(zip(*np.unique(labels, return_counts=True))) if len(labels) else {}}")

# Part 3 — Sample statistics & the selection catalog

How many galaxies survive each cut per snapshot, how many have gas at all, and how the AGN classes
populate. **Note:** the pool is the history's build-time pre-selection (massive + passive at the
anchor), not the full galaxy catalog — the funnel starts there. Also writes the per-galaxy
selection table (`powderday_quenched_selection.fits`) that Stages 0–2 read, so the RT stages never
depend on this session's memory.

In [ ]:
# ── funnel table + per-galaxy selection FITS ──
_rows, _sel_rows = [], []
for _zt, R in RESULTS.items():
    A, H, cuts = R["A"], R["H"], R["cuts"]
    ngas0 = H["P"]["ngas"][0]
    n_pool = int(np.isfinite(H["P"]["masses.stellar"][0]).sum())
    lab = R["labels"]
    _rows.append(dict(
        z_target=_zt, snap=A["snap"], z_snap=round(A["z"], 4),
        pool_massive_passive=n_pool,
        with_any_gas=int((ngas0 > 0).sum()),
        gas_gt20=int(cuts["gas>20"].sum()),
        massive=int(cuts["massive"].sum()),
        passive=int(cuts["passive"].sum()),
        star_ge20=int(cuts["star>=20"].sum()),
        dust_h2_ok=int(cuts["dust/H2"].sum()),
        selected=len(R["cols"]),
        with_event=int(np.isfinite([r["t_qt"] for r in R["records"]]).sum()),
        strong=int((lab == "strong").sum()), weak=int((lab == "weak").sum()),
        intermediate=int((lab == "intermediate").sum()), no_AGN=int((lab == "no_AGN").sum()),
        no_event=int((lab == "no_event").sum()), unclassified=int((lab == "unclassified").sum()),
    ))
    # per-galaxy rows
    P0 = H["P"]
    for i, (r, l) in enumerate(zip(R["records"], lab)):
        c = r["col"]
        with np.errstate(all="ignore"):
            _ms = float(P0["masses.stellar"][0, c])
            _sf = float(P0["sfr"][0, c])
            _md  = float(P0["masses.dust"][0, c]) if "masses.dust" in P0 else np.nan
            _mh2 = float(P0["masses.H2"][0, c])   if "masses.H2"   in P0 else np.nan
            xs = R["CO"]["xstr_quench"][i] if R["CO"] is not None else np.nan
        _sel_rows.append(dict(
            snap=int(A["snap"]), z_snap=float(A["z"]), z_target=float(_zt),
            gal_id=int(r["gid"]),
            log_mstar=float(np.log10(_ms)) if _ms > 0 else np.nan,
            ssfr=float(_sf / _ms) if _ms > 0 else np.nan,
            ngas=int(P0["ngas"][0, c]), nstar=int(P0["nstar"][0, c]) if "nstar" in P0 else -1,
            t_sft=r["t_sft"], t_qt=r["t_qt"], tau_q=r["tau_q"],
            tau_q_over_tH=r["tau_q_over_tH"], z_qt=r["z_qt"],
            xstr_quench=float(xs), agn_class=str(l),
            mdust=_md, mh2=_mh2,
            dust_to_h2=(_md / _mh2) if (np.isfinite(_md) and np.isfinite(_mh2) and _mh2 > 0) else np.nan,
        ))

STATS = Table(_rows)
STATS.write(os.path.join(TABLEDIR, "powderday_quenched_stats.fits"), overwrite=True)
STATS.pprint(max_width=-1)

SEL = Table(_sel_rows)
SEL.write(SELECTION_FITS, overwrite=True)
print(f"\nselection table: {len(SEL)} galaxies over {len(np.unique(SEL['snap']))} snapshots "
      f"-> {SELECTION_FITS}")

In [ ]:
# ── figures: selection funnel + gas-particle content + AGN classes ──
_zs   = list(RESULTS.keys())
_tags = [RESULTS[z]["A"]["tag"] for z in _zs]

fig, axes = plt.subplots(1, 3, figsize=(22, 6.5))

# funnel per anchor
_steps = ["pool_massive_passive", "with_any_gas", "gas_gt20", "selected", "with_event"]
_slbl  = ["massive+passive", "any gas", "gas>20", "all cuts", "SFT/QT found"]
_x = np.arange(len(_zs)); _w = 0.16
for j, (st, sl) in enumerate(zip(_steps, _slbl)):
    axes[0].bar(_x + (j - 2) * _w, [STATS[st][i] for i in range(len(STATS))], width=_w, label=sl)
axes[0].set_xticks(_x); axes[0].set_xticklabels(_tags)
axes[0].set_ylabel("N galaxies"); axes[0].set_title("selection funnel")
axes[0].legend(fontsize=10, frameon=False)

# gas-particle histograms (pool), with the >20 floor
for z in _zs:
    ng = RESULTS[z]["H"]["P"]["ngas"][0]
    ng = ng[np.isfinite(ng) & (ng > 0)]
    if ng.size:
        axes[1].hist(np.log10(ng), bins=25, histtype="step", lw=2, label=RESULTS[z]["A"]["tag"])
axes[1].axvline(np.log10(NGAS_MIN), color="k", ls=":", label=f"ngas={NGAS_MIN}")
axes[1].set_xlabel("log10 ngas (anchor)"); axes[1].set_ylabel("N")
axes[1].set_title("pool gas content"); axes[1].legend(fontsize=10, frameon=False)

# AGN classes among the selected
_classes = ["strong", "intermediate", "weak", "no_AGN", "no_event", "unclassified"]
_bot = np.zeros(len(_zs))
for cl in _classes:
    v = np.array([STATS[cl][i] for i in range(len(STATS))], float)
    axes[2].bar(_x, v, bottom=_bot, label=cl)
    _bot += v
axes[2].set_xticks(_x); axes[2].set_xticklabels(_tags)
axes[2].set_ylabel("N selected"); axes[2].set_title("AGN-coupling classes")
axes[2].legend(fontsize=10, frameon=False)

fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "sample_statistics.png"), dpi=150, bbox_inches="tight")
plt.show()

# Part 3b — mass–size QC: flag sources too large (or too small) for the apertures

Fixed **physical** apertures implicitly assume every source has a similar size — the mass–size
relation is the check on that. Anchor-epoch CAESAR radii come from the histories (row 0;
`radii.*` are **comoving kpc** — verified `unit: 'kpccm'` in the m25n512 catalogs — converted
with $1/(1+z)$). CAESAR $R_{50}$ is the 3D half-**mass** radius; the van der Wel+2014 quiescent
relations (projected half-light $R_e$) are drawn for context only.

Flags (written back into `SELECTION_FITS`; Part 7 carries them into every catalog):

- **`flag_too_large`** — `SIZE_FACTOR·R50 > R_CUTOUT_KPC`: the 100 pkpc cutout/grid truncates
  the stellar envelope → the "≈ total" 100 kpc aperture (and any CIGALE mass) biases low;
- **`flag_unresolved`** — `R50 < N_EPS_MIN·ε` (softening): the size is not trusted and the
  1 kpc "central" aperture is not meaningfully sub-galactic.

Nothing is dropped — the flags are one boolean away in any downstream cut. The per-rung print
shows for how much of the sample each aperture is sub-galactic (< R50) vs effectively total
(> 3 R50).


In [ ]:
# ── Part 3b — mass-size QC: flag too-large / unresolved sources for the aperture ladder ──
# Self-contained after Parts 0/0b: reads SELECTION_FITS + the anchor histories in SFHDIR.
SIZE_FACTOR    = 5.0     # envelope proxy: SIZE_FACTOR*R50 beyond the cutout -> truncated
N_EPS_MIN      = 2.0     # resolved if R50 >= N_EPS_MIN * softening
EPS_MIN_CKPC_H = 0.25    # m25n512 minimum gravitational softening [comoving kpc/h]
SIMBA_H        = 0.68

SEL, SNAPS, IDS = load_selection()
_rdb = anchor_row0(("radii.stellar_half_mass", "radii.gas_half_mass"))
_zz0 = row0_arr(_rdb, SNAPS, IDS, "z0")                        # row 0 = anchor epoch
# radii are comoving in the catalogs (kpccm) -> proper kpc with each anchor's own z
_r50s = row0_arr(_rdb, SNAPS, IDS, "radii.stellar_half_mass") / (1.0 + _zz0)
_r50g = row0_arr(_rdb, SNAPS, IDS, "radii.gas_half_mass") / (1.0 + _zz0)

_zsel = np.asarray(SEL["z_snap"], float)
_eps_pkpc = EPS_MIN_CKPC_H / SIMBA_H / (1.0 + _zsel)           # softening, proper kpc
flag_too_large  = SIZE_FACTOR * _r50s > R_CUTOUT_KPC
flag_unresolved = _r50s < N_EPS_MIN * _eps_pkpc

SEL["r50_star_kpc"]    = _r50s
SEL["r50_gas_kpc"]     = _r50g
SEL["flag_too_large"]  = flag_too_large.astype(int)
SEL["flag_unresolved"] = flag_unresolved.astype(int)
SEL.write(SELECTION_FITS, overwrite=True)

_nok = int(np.isfinite(_r50s).sum())
print(f"R50 matched: {_nok}/{len(SEL)} | median R50 = {np.nanmedian(_r50s):.2f} pkpc | "
      f"too large (R50 > {R_CUTOUT_KPC/SIZE_FACTOR:.0f} kpc): {int(flag_too_large.sum())} | "
      f"unresolved (R50 < {N_EPS_MIN:g} eps): {int(flag_unresolved.sum())}")
print("aperture rung vs the sample sizes:")
_rungs = APERTURE_RADII_KPC[np.asarray(WANTED_AP_IDX)]
for _t, _rr in zip(APERTURE_LABELS, _rungs):
    _sub = 100 * np.nanmean(_rr < _r50s); _tot = 100 * np.nanmean(_rr > 3 * _r50s)
    print(f"  {_t:>9s} ({_rr:6.2f} kpc): sub-galactic (<R50) for {_sub:4.0f}%  |  "
          f"~total (>3 R50) for {_tot:4.0f}%")
for _k in np.where(flag_too_large | flag_unresolved)[0]:
    _why = "TOO LARGE" if flag_too_large[_k] else "unresolved"
    print(f"  [{_why}] snap {int(SEL['snap'][_k])} gal {int(SEL['gal_id'][_k])}: "
          f"R50={_r50s[_k]:.2f} pkpc, logM*={float(SEL['log_mstar'][_k]):.2f}, "
          f"z={_zsel[_k]:.2f}")

# ── mass-size relation vs the aperture ladder ──
_fig, _ax = plt.subplots(figsize=(11, 7.5))
_zt_colors = plt.cm.viridis(np.linspace(0, 0.9, len(TARGET_REDSHIFTS)))
# van der Wel+2014 Table 5, early types: R_e = A*(M*/5e10)^alpha [kpc] (context only)
_VDW = {0.25: (10**0.60, 0.75), 0.75: (10**0.42, 0.71), 1.25: (10**0.22, 0.76),
        1.75: (10**0.09, 0.76), 2.25: (10**-0.05, 0.79)}
_lm = np.asarray(SEL["log_mstar"], float)
_xmax = max(11.4, np.nanmax(_lm) + 0.15)
_mm = np.logspace(10, _xmax, 40)
for _c, _zt in zip(_zt_colors, TARGET_REDSHIFTS):
    _m = np.isclose(np.asarray(SEL["z_target"], float), _zt)
    if not _m.any():
        continue
    _ax.scatter(_lm[_m], _r50s[_m], s=22, color=_c, label=f"z\u2248{_zt:g}", zorder=3)
    _A, _al = _VDW[min(_VDW, key=lambda z: abs(z - _zt))]
    _ax.plot(np.log10(_mm), _A * (_mm / 5e10) ** _al, "--", color=_c, lw=1.1, alpha=0.7)
for _k in np.where(flag_too_large | flag_unresolved)[0]:
    _ax.scatter([_lm[_k]], [_r50s[_k]], s=95, facecolor="none",
                edgecolor="crimson", lw=1.4, zorder=4)
for _rr, _t in zip(_rungs, APERTURE_LABELS):
    _ax.axhline(_rr, color="0.78", lw=0.7, zorder=1)
    _ax.text(_xmax - 0.03, _rr * 1.04, _t, fontsize=7, va="bottom", ha="right", color="0.45")
_ax.axhline(R_CUTOUT_KPC / SIZE_FACTOR, color="crimson", ls=":", lw=1.3)
_ax.text(10.02, R_CUTOUT_KPC / SIZE_FACTOR * 1.05,
         f"too large ({SIZE_FACTOR:g}\u00b7R50 > {R_CUTOUT_KPC:.0f} kpc cutout)",
         fontsize=8, color="crimson", va="bottom")
_ax.set_yscale("log"); _ax.set_xlim(9.98, _xmax)
_ax.set_xlabel(r"$\log_{10}\,M_*/M_\odot$")
_ax.set_ylabel(r"stellar $R_{50}$ [proper kpc]")
_ax.set_title("mass\u2013size QC")
_ax.plot([], [], "--", color="0.5", lw=1.1, label="vdW+14 quiescent $R_e$")
_ax.legend(fontsize=10, frameon=False, loc="lower right")
plt.savefig(os.path.join(PLOTDIR, "mass_size_aperture_qc.png"), dpi=140, bbox_inches="tight")
plt.show()


# Part 4 — Stage 0: extract the per-galaxy particle files (cluster)

One HDF5 per galaxy (gas + stars + BHs; gas keeps `Dust_Masses`, BHs carry
`BH_Mass`/`BH_Mdot` for the `agn_on` run) under
`hydro_dir_base/snap_NNN/<PREFIX>_snap<NNN>_gal<ID>.h5` — now a **100 pkpc spherical region
cutout** around each galaxy centre (CGM + satellites included; periodic-wrap safe), NOT just
the caesar member particles. Identical for all runs (`dust_on` / `dust_off` / `agn_on` — the dust treatment and the
AGN sources live in the parameter masters). Cutouts extracted before 2026-07-28 have no
`PartType5` group — re-run this part (`EXTRACT_OVERWRITE = True`) before launching `agn_on`. Reads `SELECTION_FITS`, so it can run in a fresh session once Part 3
has been executed.

⚠ Region files reuse the plist filenames, so `EXTRACT_OVERWRITE = True` below **replaces** any
old galaxy-member-only files — intended, since mixed hydro inputs would corrupt the sample.

In [ ]:
from simbanator.analysis import extract_particles

SEL, SNAPS, IDS = load_selection()
print(f"{len(SNAPS)} sources over snapshots {sorted(set(SNAPS.tolist()))}")

EXTRACT_OVERWRITE = True    # region cutouts REPLACE the old plist files (same names)
EXTRACT_PTYPES    = ("PartType0", "PartType4", "PartType5")   # gas (Dust_Masses) + stars + BHs (agn_on)

bad_snaps = []
for _snap in np.unique(SNAPS):
    _snap = int(_snap)
    _ids_here = np.unique(IDS[SNAPS == _snap])
    _simfile = sim.get_snapshot_file(_snap)
    print(f"snap {_snap:3d}: extracting {len(_ids_here)} galaxies from {os.path.basename(_simfile)}")
    try:
        _cs = sim.load_catalog(snap=_snap)
        extract_particles(_cs, _simfile, _snap, galaxy_ids=_ids_here, radius=R_CUTOUT_KPC,
                          ptypes=EXTRACT_PTYPES, sim_name=sim.name, prefix=PARTICLE_PREFIX,
                          overwrite=EXTRACT_OVERWRITE, verbose=1)
        del _cs
    except (OSError, KeyError) as e:
        print(f"  [SKIP] snap {_snap}: {type(e).__name__}: {str(e).splitlines()[0]}")
        bad_snaps.append((_snap, len(_ids_here)))

print("\nparticle extraction complete ->", hydro_dir_base)
if bad_snaps:
    print(f"{len(bad_snaps)} snapshot(s) unreadable: {bad_snaps} — re-stage those files and re-run.")

# Part 4b — annulus sampling QC: particle counts per projected annulus

How well can powderday sample an **annular** SED? Each Hyperion aperture is a circle in the
**image plane**, so for every sightline the star (emission sources) and gas (dust carriers)
particles of each 100 pkpc cutout are projected perpendicular to the viewing direction and
counted in the annuli between consecutive rungs (`ann1kpc` = the 0→1 kpc disc, then 1→3.16,
3.16→10, 10→31.6, 31.6→100 kpc; as in Part 7a, the **outer** rung names the annulus).
Counts span the whole LOS depth through the sphere — exactly the geometry the RT sees (the
outermost annulus is depth-truncated like its aperture). Centres are the **exact RT grid
centres** (`code_coods` from the Stage-1 selection HDF5; caesar fallback if Part 5 has not
run yet).

Reading the numbers:

- **stars = intrinsic emitters.** `nstar = 0` → the annular flux is scattered/re-emitted
  light only and a CIGALE fit of it is meaningless; a few tens of star particles → the
  annular SED is shot-noise dominated (a handful of SSP ages/metallicities).
- **gas → dust grid.** Gas is smoothed onto the octree, so counts are indicative; `ndust`
  (gas with `Dust_Masses > 0`) counts the particles actually carrying dust.

One row per (galaxy, sightline) → `tables/annulus_particle_counts.fits`
(`nstar_/ngas_/ndust_/ntot_<annulus>` + `A_V_glob`/`dusty`); Part 7a's radial profile uses it to flag
star-free annular catalogs.

**Dusty vs non-dusty split.** If Part 7a's `attenuation_vs_ism.fits` exists, each galaxy is
flagged **dusty** (global $A_V > 0.1$, same threshold as 7a Fig 3, fiducial aperture +
sightline) or non-dusty, the figure highlights the two subsamples (red vs gray lines, separate
medians) and the summary prints their per-annulus median counts — the dusty galaxies are the
ones whose annular attenuation/CIGALE fits matter, so their sampling is the QC that counts.
Part 7a needs the RT fluxes, so on a fresh pipeline this cell first runs without the split
(`dusty = -1`) — **re-run it after Part 7a** to get the highlighted version.

In [ ]:
# ── Part 4b — annulus sampling QC: star/gas counts per projected annulus & sightline ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts (Part 4).
_R_MID = np.where(R_EDGES[:-1] > 0, np.sqrt(R_EDGES[:-1] * R_EDGES[1:]), R_EDGES[1:] / 2.0)

SEL, SNAPS, IDS = load_selection()
_centers = rt_centers(SNAPS, IDS)

_rows, _skipped = [], []
for _s, _g in zip(SNAPS, IDS):
    _c  = _centers.get((int(_s), int(_g)))
    _st = read_cutout(_s, _g, _c, "PartType4")
    _gs = read_cutout(_s, _g, _c, "PartType0", fields=("Dust_Masses",))
    if _st is None or _gs is None:
        _skipped.append((int(_s), int(_g)))
        continue
    _dm = (_gs["Dust_Masses"] if _gs["Dust_Masses"] is not None
           else np.zeros(len(_gs["pos"])))
    _d = {"star": _st["pos"], "gas": _gs["pos"],
          "dust": _gs["pos"][np.asarray(_dm) > 0]}                # dust-carrying gas
    for _j, _il in enumerate(INCL_LABELS):
        _row = {"snap": int(_s), "gal_id": int(_g), "incl": _il}
        for _pt, _pos in _d.items():                              # projected radius wrt LOS
            _cnt, _ = np.histogram(projected_radius(_pos, NHAT[_j]), R_EDGES)
            for _k, _al in enumerate(ANNULUS_LABELS):
                _row[f"n{_pt}_{_al}"] = int(_cnt[_k])
        for _al in ANNULUS_LABELS:
            _row[f"ntot_{_al}"] = _row[f"nstar_{_al}"] + _row[f"ngas_{_al}"]
        _rows.append(_row)

QC_COUNTS = Table(_rows)
QC_COUNTS.meta["R_EDGES"] = list(np.round(R_EDGES, 3))             # proper kpc

# dusty split from Part 7a (global A_V, fiducial aperture/sightline):
# dusty = 1 (A_V > AV_DUSTY), 0 (transparent), -1 (no A_V yet -> re-run after Part 7a)
_avg, QC_COUNTS["dusty"] = dusty_flags(QC_COUNTS["snap"], QC_COUNTS["gal_id"])
QC_COUNTS["A_V_glob"] = _avg
_have_av = bool(np.isfinite(_avg).any())

_out = os.path.join(TABLEDIR, "annulus_particle_counts.fits")
QC_COUNTS.write(_out, overwrite=True)
print(f"{len(QC_COUNTS)} rows ({len(QC_COUNTS)//N_INCL} galaxies x {N_INCL} sightlines) -> {_out}")
if _skipped:
    print(f"[WARN] {len(_skipped)} galaxies without cutout/centre, skipped: {_skipped}")
_dg = np.asarray(QC_COUNTS["dusty"], int)[::N_INCL]   # per galaxy (same on all sightlines)
if _have_av:
    print(f"dusty split (Part 7a global A_V > {AV_DUSTY:g}): {int((_dg == 1).sum())} dusty / "
          f"{int((_dg == 0).sum())} non-dusty / {int((_dg == -1).sum())} unmatched galaxies")

# ── summary: how well is each annulus sampled? ──
print(f"\n{'annulus':>10s} {'r [pkpc]':>13s} | {'nstar p16/50/84':>17s} {'=0':>4s} {'<10':>4s} "
      f"{'<100':>5s} | {'ngas p50':>8s} {'=0':>4s} | {'ndust p50':>9s} {'=0':>4s}")
for _k, _al in enumerate(ANNULUS_LABELS):
    _ns = np.asarray(QC_COUNTS[f"nstar_{_al}"], int)
    _ng = np.asarray(QC_COUNTS[f"ngas_{_al}"],  int)
    _nd = np.asarray(QC_COUNTS[f"ndust_{_al}"], int)
    _p  = np.percentile(_ns, [16, 50, 84]).astype(int)
    print(f"{_al:>10s} {R_EDGES[_k]:5.1f}-{R_EDGES[_k+1]:6.1f} | "
          f"{_p[0]:5d}/{_p[1]:5d}/{_p[2]:5d} {np.mean(_ns == 0)*100:3.0f}% {np.mean(_ns < 10)*100:3.0f}% "
          f"{np.mean(_ns < 100)*100:4.0f}% | {int(np.median(_ng)):8d} {np.mean(_ng == 0)*100:3.0f}% | "
          f"{int(np.median(_nd)):9d} {np.mean(_nd == 0)*100:3.0f}%")
_nfree = int(sum((np.asarray(QC_COUNTS[f"nstar_{_al}"], int) == 0).sum()
                 for _al in ANNULUS_LABELS))
print(f"\nstar-free (annulus, galaxy, sightline) triples: {_nfree} "
      f"/ {len(QC_COUNTS) * len(ANNULUS_LABELS)} — those annular SEDs have NO intrinsic emitters")

if _have_av:                     # median counts split by the Part 7a dusty flag
    _dm_all = np.asarray(QC_COUNTS["dusty"], int)
    print(f"\nmedian counts, dusty (D, n={int((_dg == 1).sum())} gals) "
          f"vs non-dusty (N, n={int((_dg == 0).sum())}):")
    print(f"{'annulus':>10s} | {'nstar D':>8s} {'nstar N':>8s} | {'ngas D':>8s} {'ngas N':>8s} "
          f"| {'ndust D':>8s} {'ndust N':>8s}")
    for _al in ANNULUS_LABELS:
        _vals = []
        for _cc in ("nstar", "ngas", "ndust"):
            _v = np.asarray(QC_COUNTS[f"{_cc}_{_al}"], int)
            for _dd in (1, 0):
                _m = _dm_all == _dd
                _vals.append(int(np.median(_v[_m])) if _m.any() else -1)
        print(f"{_al:>10s} | {_vals[0]:8d} {_vals[1]:8d} | {_vals[2]:8d} {_vals[3]:8d} "
              f"| {_vals[4]:8d} {_vals[5]:8d}")

# ── figure: count distributions per annulus, dusty vs non-dusty highlighted ──
_dm_all = np.asarray(QC_COUNTS["dusty"], int)
_have_split = _have_av and (_dm_all >= 0).any()
_fig, _axs = plt.subplots(1, 3, figsize=(20, 6.5), sharey=True)
for _ax, _pt, _ttl in zip(_axs, ("star", "gas", "dust"),
                          ("stars", "gas", "dust-carrying gas")):
    _M = np.column_stack([np.asarray(QC_COUNTS[f"n{_pt}_{_al}"], int)
                          for _al in ANNULUS_LABELS])
    if _have_split:                # per-(gal,sightline) lines colored by the Part 7a split
        for _rowv, _dd in zip(_M, _dm_all):
            _ax.plot(_R_MID, _rowv, color={1: "#c0392b", 0: "0.75"}.get(_dd, "0.88"),
                     lw=0.5, alpha=0.45, zorder=1)
        for _dd, _col, _mk, _lab in ((1, "#c0392b", "o-", f"dusty ($A_V>{AV_DUSTY:g}$)"),
                                     (0, "#2980b9", "s--", "non-dusty")):
            _mrows = _dm_all == _dd
            if _mrows.any():
                _ax.plot(_R_MID, np.median(_M[_mrows], axis=0), _mk, color=_col, lw=2,
                         zorder=3, label=f"{_lab} median (n={int(_mrows.sum()) // N_INCL} gals)")
    else:
        for _rowv in _M:                                          # one line per (gal, sightline)
            _ax.plot(_R_MID, _rowv, color="0.75", lw=0.5, alpha=0.5, zorder=1)
        _ax.fill_between(_R_MID, np.percentile(_M, 16, axis=0), np.percentile(_M, 84, axis=0),
                         color="#2980b9", alpha=0.25, zorder=2, label="16–84%")
        _ax.plot(_R_MID, np.median(_M, axis=0), "o-", color="#2980b9", lw=2, zorder=3,
                 label="median")
    for _thr, _ls in ((10, ":"), (100, "--")):
        _ax.axhline(_thr, color="0.3", ls=_ls, lw=0.9)
        _ax.text(_R_MID[0] * 0.9, _thr * 1.15, f"N={_thr}", color="0.3", fontsize=7)
    _ax.set_xscale("log"); _ax.set_yscale("symlog", linthresh=1)
    _ax.set_xticks(_R_MID); _ax.set_xticklabels([l[3:] for l in ANNULUS_LABELS], fontsize=10)
    _ax.set_xlabel("annulus"); _ax.set_title(_ttl)
    _ax.set_ylim(bottom=-0.5)
_axs[0].set_ylabel("particles per projected annulus (full LOS depth)")
_axs[0].legend(fontsize=10, frameon=False, loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(PLOTDIR, "annulus_particle_counts.png"), dpi=140, bbox_inches="tight")
plt.show()


# Part 4c — SIMBA metallicities per aperture & annulus (CIGALE priors)

Mass-weighted **stellar** and **gas** metallicities (total metal mass fraction,
`Metallicity[:, 0]`) of each cutout, measured in the **same projected geometry as the SEDs**:
per sightline, cumulative within each aperture rung (`ap1kpc…ap100kpc`) and in each annulus
between rungs (`ann3kpc…ann100kpc`; `ann1kpc`≡`ap1kpc`). Same centres/projection as Part 4b.

These are the **metallicity pins for the CIGALE runs** (Part 7d): CIGALE's bc03
`metallicity` and nebular `zgas` are strict grids, so each galaxy's SIMBA value is snapped to
the **nearest allowed grid value in log space** (bc03: 0.0001, 0.0004, 0.004, 0.008, 0.02,
0.05 — verified against the cluster's CIGALE 2025.1 sources) and the catalog is split into
per-metallicity sub-runs: one per bc03 node, each fitted with that single stellar Z and a
`zgas` grid restricted to the group members' snapped values. The summary below shows how
the sample maps onto the bc03 nodes per aperture — since Part 7d pins ONE node per object,
this is also the quantisation floor on the metallicity prior. It reports how the sample
would have split into sub-runs under the old grouped scheme, which Part 7d
create. Empty apertures/annuli (no particles) → NaN → the `Zsfree` group (default Z grid).

One row per (galaxy, sightline) → `tables/aperture_metallicities.fits`
(`Zstar_<label>`, `Zgas_<label>` for the 9 labels).

In [ ]:
# ── Part 4c — mass-weighted Z_star / Z_gas per projected aperture & annulus ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts; same geometry as Part 4b.
from simbanator.sed.cigale import grid_options, nearest_option

Z_LABELS = list(APERTURE_LABELS) + ANNULUS_LABELS[1:]   # cumulative rungs + true annuli

SEL, SNAPS, IDS = load_selection()
_centers = rt_centers(SNAPS, IDS)

def _mwz(z, m, sel):
    # mass-weighted metallicity over a particle selection (NaN if empty)
    if not sel.any():
        return np.nan
    mm = m[sel]
    return float(np.sum(mm * z[sel]) / np.sum(mm)) if mm.sum() > 0 else np.nan

_rows, _skipped = [], []
for _s, _g in zip(SNAPS, IDS):
    _c = _centers.get((int(_s), int(_g)))
    _P = {}
    for _pt, _nm in (("PartType4", "star"), ("PartType0", "gas")):
        _cut = read_cutout(_s, _g, _c, _pt, fields=("Masses", "Metallicity"))
        if _cut is None:
            _P = None
            break
        _met = _cut["Metallicity"]
        _z = np.asarray(_met[:, 0] if _met.ndim == 2 else _met, float)  # col 0 = total Z
        _P[_nm] = (_cut["pos"], np.asarray(_cut["Masses"], float), _z)
    if _P is None:
        _skipped.append((int(_s), int(_g)))
        continue
    for _j, _il in enumerate(INCL_LABELS):
        _row = {"snap": int(_s), "gal_id": int(_g), "incl": _il}
        for _nm, (_pos, _m, _z) in _P.items():
            _R = projected_radius(_pos, NHAT[_j])
            for _k, _lab in enumerate(APERTURE_LABELS):              # cumulative
                _row[f"Z{_nm}_{_lab}"] = _mwz(_z, _m, _R <= R_EDGES[_k + 1])
            for _k, _lab in enumerate(ANNULUS_LABELS[1:], start=1):  # annular
                _row[f"Z{_nm}_{_lab}"] = _mwz(_z, _m,
                                              (_R > R_EDGES[_k]) & (_R <= R_EDGES[_k + 1]))
        _rows.append(_row)

ZTAB = Table(_rows)
ZTAB.meta["R_EDGES"] = list(np.round(R_EDGES, 3))
_out = os.path.join(TABLEDIR, "aperture_metallicities.fits")
ZTAB.write(_out, overwrite=True)
print(f"{len(ZTAB)} rows ({len(ZTAB)//N_INCL} galaxies x {N_INCL} sightlines) -> {_out}")
if _skipped:
    print(f"[WARN] {len(_skipped)} galaxies without cutout/centre, skipped: {_skipped}")

# ── summary: sample vs the CIGALE grids (Part 7d pins ONE node per object) ──
ZSUN = 0.0134
_zs_grid = grid_options("bc03", "metallicity")
print(f"\nbc03 metallicity nodes: {_zs_grid}")
print(f"{'label':>10s} | {'med Z*/Zsun':>11s} {'med Zgas/Zsun':>13s} | bc03 node counts (stars)")
for _lab in Z_LABELS:
    _zsv = np.asarray(ZTAB[f"Zstar_{_lab}"], float)
    _zgv = np.asarray(ZTAB[f"Zgas_{_lab}"], float)
    _near = nearest_option(_zsv, _zs_grid)
    _cnt = {f"{_n:g}": int(np.sum(_near == _n)) for _n in _zs_grid
            if np.sum(_near == _n)}
    _nnan = int(np.sum(~np.isfinite(_near)))
    if _nnan:
        _cnt["free"] = _nnan
    print(f"{_lab:>10s} | {np.nanmedian(_zsv)/ZSUN:11.2f} {np.nanmedian(_zgv)/ZSUN:13.2f} | {_cnt}")


# Part 5 — Stage 1: selection HDF5 + Slurm masters (all three runs)

## ⚠ REQUIRED once per powderday install: the multi-aperture patch

Stock powderday gives the peeled SED a **single infinite aperture**; the parameter masters in this
repo now carry `SED_APERTURE_NAP / SED_APERTURE_MIN_KPC / SED_APERTURE_MAX_KPC`, but powderday must
be taught to read them (**already applied** in this cluster's `powderday/front_end_tools.py`,
both SED branches — redo only on a fresh install). On the cluster, locate the peeled-image setup:

```bash
grep -rn "add_peeled_images" $(python -c "import powderday, os; print(os.path.dirname(powderday.__file__))")
```

and immediately after the image-configuration lines (`set_viewing_angles` / `set_track_origin` /
`set_uncertainties`), insert (adapt `image` / `cfg.par` to the local variable names in that file):

```python
# --- multi-aperture SEDs (analize_simba_cgm patch) ---
try:
    from hyperion.util.constants import kpc as _kpc
    _nap = int(getattr(cfg.par, 'SED_APERTURE_NAP', 0))
    if _nap > 0:
        image.set_aperture_range(_nap,
                                 float(cfg.par.SED_APERTURE_MIN_KPC) * _kpc,
                                 float(cfg.par.SED_APERTURE_MAX_KPC) * _kpc)
        image.set_uncertainties(True)   # Monte-Carlo SED errors -> <filter>_err columns
except Exception as _e:
    print('[aperture patch] skipped:', _e)
```

Hyperion **log-spaces** the apertures between min and max: 1→100 kpc with `NAP=5` gives the
10^(k/2) ladder **1, 3.16, 10, 31.6, 100 kpc** (central → outskirts).
Viewing angles need **no extra patch**: stock powderday already reads
`MANUAL_ORIENTATION / THETA / PHI` from the parameter master. Verify with the Part 6 QC cell
after the first galaxy finishes.

Then run this cell, launch `submit_all_snaps.sh` under each run's `powderday_sed_out/`, and come
back to Part 6/7 when the `.rtout.sed` files exist.

In [ ]:
# ── MakeSED handles (constructor only — cheap; Parts 6–7 need just this cell, not the next) ──
from simbanator.sed.makesed import MakeSED

makeseds = {
    key: MakeSED(sim, nnodes=1, model_run_name=cfg['run_tag'],
                 hydro_dir_base=hydro_dir_base, selection_file=selection_file,
                 output_dir=sed_output_dir, run_tag=cfg['run_tag'])
    for key, cfg in RUNS.items()
}
for key, ms in makeseds.items():
    print(f"{key:9s} -> run_tag='{ms.run_tag}', master='{RUNS[key]['paramf']}'")

In [ ]:
# ── write the selection HDF5 + generate the Slurm masters (run once per sample change) ──
SEL, SNAPS, IDS = load_selection()

for key, cfg in RUNS.items():
    ms = makeseds[key]
    print(f"\n=== {key} (run_tag='{ms.run_tag}', master='{cfg['paramf']}') ===")
    ms.selection_gals(snaps=SNAPS, galaxyID=IDS)                 # same sources for both runs
    ms.create_master('cluster', 'region', radius=R_CUTOUT_KPC,
                     partition='INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL',
                     prefix=PARTICLE_PREFIX, paramf=cfg['paramf'], snaps_to_run=None)

# Part 6 — Aperture QC (run after the first `.rtout.sed` exists)

Confirms the powderday patch took effect **before** burning time on the full extraction:
reads the aperture layout stored in one output file, probes every aperture index, and prints
the expected index → radius mapping.

In [ ]:
from simbanator.sed.makesed import list_sed_apertures, _read_sed

_pat = os.path.join(makeseds['dust_on'].model_dir_base, 'snap_*', 'gal_*', '*.rtout.sed')
_cands = sorted(glob.glob(_pat))
if not _cands:
    raise FileNotFoundError(f"no .rtout.sed yet under {makeseds['dust_on'].model_dir_base} — "
                            "run the RT jobs first")
_probe = _cands[0]
print("probing:", _probe, "\n")

for gname, entry in list_sed_apertures(_probe).items():
    print(f"[{gname}] seds shape = {entry.get('seds_shape')}")
    for k, v in entry.get('seds_attrs', {}).items():
        print(f"    seds.attrs[{k!r}] = {v}")
    for k, v in entry['group_attrs'].items():
        print(f"    group.attrs[{k!r}] = {v}")

print("\nexpected mapping (log-spaced, from the parameter master):")
for i, r in enumerate(APERTURE_RADII_KPC):
    mark = (f"   <- {APERTURE_LABELS[WANTED_AP_IDX.index(i)]}"
            if i in WANTED_AP_IDX else "")
    print(f"  aperture={i:2d} -> {r:7.2f} kpc{mark}")

n_ok = 0
for i in range(N_AP):
    try:
        wav, flx, unc = _read_sed(_probe, aperture=i, uncertainties=True)
        assert np.shape(flx)[0] == N_INCL, (
            f"{np.shape(flx)[0]} inclination(s) in the rtout but N_INCL={N_INCL} — "
            "THETA/PHI here disagree with the parameter master the jobs copied")
        has_unc = unc is not None and np.isfinite(np.asarray(unc)).any()
        print(f"  aperture={i}: OK  flux shape={np.shape(flx)}  MC uncertainties={'yes' if has_unc else 'NO'}")
        n_ok += 1
    except Exception as e:
        print(f"  aperture={i}: FAILED ({type(e).__name__}: {e})")
assert n_ok == N_AP, (
    f"only {n_ok}/{N_AP} apertures readable — the powderday aperture patch is NOT active "
    "(or N_AP here disagrees with SED_APERTURE_NAP in the parameter master the jobs copied)")
print(f"\nOK — {N_AP} apertures present.")

# Part 7 — Stage 2: per-aperture flux extraction → catalogs

Filter set (2026-07-10): **Subaru/HSC, CFHT/MegaCam, HST/WFC3, JWST/NIRCam+MIRI,
Spitzer/MIPS (24/70/160 µm) + Herschel/PACS+SPIRE (70–500 µm — the dust-emission
peak), VISTA/VIRCAM, JCMT/SCUBA-2, ALMA band 6** (custom 211–275 GHz top-hat,
`ALMA_band6.res`) + the Johnson V/U & 2MASS J locals kept for Part 7a.

For each dust run × aperture: convolve the SED (and its Monte-Carlo uncertainty) with the filter
set, then join the sample metadata. **One catalog per aperture per RT arm** under
`output/cis25/sed_aperture_catalogs/`, columns: `gal_id, snap, redshift`, sample metadata
(`agn_class, log_mstar, ngas, t_sft, t_qt, tau_q, xstr_quench`, …) and per-filter
`<filter>` / `<filter>_err` fluxes (mJy, rest-frame convolution as in `test_powderday.ipynb`).

In [ ]:
# ── filter set (user-chosen mock-observation instruments, 2026-07-09) ──
# optical: Subaru/HSC, CFHT/MegaCam, HST/WFC3; near-IR: JWST/NIRCam, VISTA
# (SVO lists it as Paranal/VIRCAM; CIGALE names it paranal.vircam.*);
# mid-IR: JWST/MIRI; far-IR: Spitzer/MIPS 24/70/160 um (samples the dust
# emission peak that MIRI + the sub-mm bands only straddle); sub-mm/mm:
# Herschel/PACS (70/100/160 um) + SPIRE (250/350/500 um) bracket the cold-dust
# peak (added 2026-07-10, user request); sub-mm/mm: JCMT/SCUBA-2 + ALMA band 6
# (local top-hat, 211-275 GHz — the SED grid is 0.001-1000 um REST, so
# observed-frame band 6 is covered at every anchor; at z=0.3 its red edge is
# truncated at 1.31 mm).
FACILITIES  = ['Subaru', 'CFHT', 'HST', 'JWST', 'JWST', 'Spitzer',
               'Herschel', 'Herschel', 'Paranal', 'JCMT']
INSTRUMENTS = ['HSC', 'MegaCam', 'WFC3', 'NIRCam', 'MIRI', 'MIPS',
               'PACS', 'SPIRE', 'VIRCAM', 'SCUBA2']

local_filters = {
    # Johnson V/U + 2MASS J stay for the Part 7a/7f true A_V (never fitted)
    '2MASS':   {'J': {'J': REMOTE_HOME + '/2MASS_J.res'}},
    'Johnson': {'V': {'V': REMOTE_HOME + '/maiz-apellaniz_Johnson_V.res'}},
    # separate top-level key required: dict cannot hold two entries under 'Johnson'
    'Johnson2': {'U': {'U': REMOTE_HOME + '/maiz-apellaniz_Johnson_U.res'}},
    # custom: not in SVO; the matching ALMA_band6_cigale.dat must be added to
    # the CIGALE env once: pcigale-filters add ALMA_band6_cigale.dat
    'ALMA': {'ALMA': {'band6': REMOTE_HOME + '/ALMA_band6.res'}},
}

SEL, SNAPS, IDS = load_selection()

def extract_flux_set(redshift, prefix):
    """extract_flux_batch over every (RT arm, aperture rung, sightline).

    redshift=False -> rest-frame (this part's catalogs); True -> observed
    frame (the Part 7b CIGALE inputs). One pass = 2 x 5 x 4 = 40 extractions;
    returns {(dust key, aperture label, incl label): flux-table path}.
    """
    files = {}
    for key in RUNS:
        ms = makeseds[key]
        for i, label in zip(WANTED_AP_IDX, APERTURE_LABELS):
            for j, ilab in enumerate(INCL_LABELS):
                print(f"\n=== extract: {key} / {label} / {ilab} (aperture {i}, "
                      f"inclination {j}, {'observed' if redshift else 'rest'} frame) ===")
                flux_file, _ = ms.extract_flux_batch(
                    SNAPS, IDS, FACILITIES, INSTRUMENTS,
                    filters=None, local_filters=local_filters, wave_unit='micron',
                    findx=j, aperture=i, uncertainties=True, redshift=redshift,
                    outname=f"{prefix}_{key}_{label}_{ilab}.fits")
                files[(key, label, ilab)] = flux_file
    return files

FLUX_FILES = extract_flux_set(redshift=False, prefix="fluxes")   # rest frame


In [ ]:
# ── final catalogs: fluxes+errors ⨝ sample metadata; one file per (RT arm, aperture) ──
_META = ["gal_id", "snap", "z_snap", "z_target", "agn_class", "xstr_quench",
         "log_mstar", "ngas", "nstar", "ssfr", "t_sft", "t_qt", "tau_q", "tau_q_over_tH",
         "r50_star_kpc", "flag_too_large", "flag_unresolved"]
SEL, SNAPS, IDS = load_selection()
_META = [c for c in _META if c in SEL.colnames]   # size-QC columns exist after Part 3b

CATALOGS = {}
for (key, label, ilab), ff in FLUX_FILES.items():
    t = Table.read(ff)
    if len(t) == 0:
        print(f"[{key}/{label}/{ilab}] EMPTY flux table — skipped"); continue
    t.rename_column('gal_id_at_snap', 'gal_id')
    cat = join(t, SEL[_META], keys=['snap', 'gal_id'], join_type='left')
    flux_cols = [c for c in t.colnames if c not in ('gal_id', 'snap', 'redshift')]
    cat = cat[['gal_id', 'snap', 'redshift'] + [c for c in _META if c not in ('gal_id', 'snap')]
              + flux_cols]
    out = os.path.join(CATDIR, f"catalog_{key}_{label}_{ilab}.fits")
    _k = APERTURE_LABELS.index(label)
    cat.meta['APERTURE'] = label
    cat.meta['APIDX'] = WANTED_AP_IDX[_k]
    cat.meta['APKPC'] = float(APERTURE_RADII_KPC[WANTED_AP_IDX[_k]])   # true rung radius
    cat.meta['INCL'] = ilab
    cat.meta['THETA'] = THETA_DEG[INCL_LABELS.index(ilab)]
    cat.meta['PHI'] = PHI_DEG[INCL_LABELS.index(ilab)]
    cat.meta['DUSTRUN'] = key
    cat.write(out, overwrite=True)
    CATALOGS[(key, label, ilab)] = out
    n_err = sum(1 for c in cat.colnames if c.endswith('_err'))
    print(f"[{key}/{label}/{ilab}] {len(cat)} galaxies, {n_err} error columns -> {out}")

# ── cross-check: per aperture, every run must contain the same sources as dust_on ──
print()
for label in APERTURE_LABELS:
    for ilab in INCL_LABELS:
        fon = FLUX_FILES.get(('dust_on', label, ilab))
        if fon is None:
            continue
        t_on = Table.read(fon)
        s_on = set(zip(np.asarray(t_on['snap'], int), np.asarray(t_on['gal_id_at_snap'], int)))
        for key in (k for k in RUNS if k != 'dust_on'):
            fk = FLUX_FILES.get((key, label, ilab))
            if fk is None:
                continue
            t_k = Table.read(fk)
            s_k = set(zip(np.asarray(t_k['snap'], int), np.asarray(t_k['gal_id_at_snap'], int)))
            status = "OK" if s_on == s_k else f"MISMATCH on={sorted(s_on - s_k)} {key}={sorted(s_k - s_on)}"
            print(f"{label:>10s}/{ilab}: dust_on={len(s_on)} {key}={len(s_k)} -> {status}")

# Part 7a — Dust attenuation $A_V$ from the matched dust_on / dust_off fluxes

We already have dust_on **and** dust_off fluxes for the *same* galaxies, so the rest-frame
band attenuation is a direct differential measurement — no SED fit needed:

$$A_\lambda = -2.5\,\log_{10}\!\left(\frac{F_{\rm dust\_on}}{F_{\rm dust\_off}}\right)\ \ [\mathrm{mag}]$$

measured per aperture from the Part-7 catalogs (rest-frame `Johnson.V.V` for $A_V$, plus
`Johnson2.U.U`/`2MASS.J.J` for the curve slope $A_U\!-\!A_V$). The **radial** attenuation
is built the observational way: annular fluxes $F(<r_{\rm out})-F(<r_{\rm in})$ between
consecutive aperture rungs give $A_V$ per annulus (annuli whose differential flux goes
non-positive from MC noise are masked).

$A_V$ is correlated against

- the **anchor-epoch ISM**: $f_{\rm mol}$, $M_{H_2}/M_\star$, $f_{\rm gas}$, dust-to-gas and
  $f_{\rm dust}=M_{\rm dust}/M_\star$ (histories, row 0), plus $\kappa_{\rm rot}$ of the H$_2$
  gas disk — the H$_2$-mass-weighted Sales+2012 $\kappa_{\rm rot}$ inside 20 pkpc, computed
  from the Stage-0 region cutouts (caesar's all-gas $\kappa_{\rm rot}$ kept for comparison);
- the **quench diagnostics + stellar structure at the observation epoch**: sSFR, $M_\star$,
  mass-weighted age, $\log Z_\star/Z_\odot$, stellar $B/T$ (caesar `rotation.stellar_BoverT`),
  $\tau_q$, $\tau_q/t_H$ and the AGN class.

Outputs: `tables/attenuation_vs_ism.fits` (per-galaxy $A_\lambda$ + ISM + structure +
quench/AGN, with $A_V$ per aperture **and** per annulus), a Spearman-ranked correlation
table, and three figures (`attenuation_vs_ism.png`, `attenuation_vs_quench.png`,
`attenuation_aperture_curve.png`).

*Needs Parts 0–3 (histories under `SFHDIR`), the Part-4 region cutouts (for
$\kappa_{\rm rot}^{H_2}$), the anchor caesar catalogs (for $Z$, $B/T$, $\kappa_{\rm rot}$)
and Part 7 (flux catalogs); CIGALE is **not** required.*

In [ ]:
# ── Part 7a — Dust attenuation (A_λ) from dust_on/dust_off vs ISM & quenching ──
# Since we already have matched dust_on / dust_off fluxes for the SAME galaxies,
# the rest-frame band attenuation follows directly (no SED fit needed):
#       A_λ = -2.5 log10( F_dust_on / F_dust_off )   [mag]
# measured per aperture from the Part-7 catalogs. The RADIAL profile is built the
# observational way: annular fluxes F(<r_out) - F(<r_in) between consecutive
# aperture rungs -> A_V per annulus. A_V is then correlated against the
# anchor-epoch ISM content (H2/HI/gas/dust from the histories + kappa_rot of the
# H2 disk from the Stage-0 region cutouts) and the quench diagnostics + stellar
# structure (mass, age, metallicity, B/T) at the observation epoch.
from scipy.stats import spearmanr
from simbanator.sed.flux_extraction import attenuation_mag as _atten

ATTEN_BANDS = {"A_U": "Johnson2.U.U", "A_V": "Johnson.V.V", "A_J": "2MASS.J.J"}
AGN_COLORS  = {"strong": "#c0392b", "intermediate": "#e67e22", "weak": "#2980b9",
               "no_AGN": "#27ae60", "no_event": "#7f8c8d", "unclassified": "#bdc3c7"}
ZSUN       = 0.0134     # Asplund+2009 total-Z scale (SIMBA's Solar reference)
R_KROT_KPC = 20.0       # H2-disk kappa_rot measured inside this proper radius

# ── 1. anchor-epoch ISM masses + stellar age (row 0 of each history) ──
GAS_KEYS = ["masses.H2", "masses.HI", "masses.gas", "masses.dust", "masses.stellar",
            "ages.mass_weighted"]
_gasdb = anchor_row0(GAS_KEYS)                                   # row 0 = anchor epoch
_missing_gas = [k for k in GAS_KEYS if not any(k in v for v in _gasdb.values())]
if _missing_gas:
    print("WARNING: ISM fields absent from histories (dropped at build):", _missing_gas)
print(f"ISM masses: {len(_gasdb)} (snap,gal) rows from the anchor histories")

def _gp_arr(tab, key):
    """anchor-epoch mass `key` aligned to a catalog table's (snap, gal_id) rows."""
    return row0_arr(_gasdb, tab["snap"], tab["gal_id"], key)

# ── 1b. anchor-epoch structure from the caesar catalogs (direct h5py read) ──
# metallicities / stellar B/T / gas kappa_rot are not tracked in the histories;
# GroupID == row index in the caesar files, so a plain dataset read suffices.
STRUCT_KEYS = {"Z_star": "metallicities.stellar", "Z_gas": "metallicities.mass_weighted",
               "BT_star": "rotation.stellar_BoverT", "kappa_gas": "rotation.gas_kappa_rot"}
_structdb, _posdb = {}, {}            # (snap,gid) -> {props} / (pos[kpccm], a, h)
_need = {}
for _s, _g in _gasdb:
    _need.setdefault(_s, set()).add(_g)
for _snap, _gids in sorted(_need.items()):
    _cf = sim.get_caesar_file(_snap)
    if not os.path.exists(_cf):
        print(f"WARNING: no caesar file for snap {_snap} -> structure props stay NaN")
        continue
    with h5py.File(_cf, "r") as f:
        _dcts = f["galaxy_data/dicts"]
        _vals = {k: _dcts[v][:] for k, v in STRUCT_KEYS.items() if v in _dcts}
        _pos  = f["galaxy_data/pos"][:]                          # kpccm
        _sa   = f["simulation_attributes"].attrs
        _a, _h = float(_sa["scale_factor"]), float(_sa["hubble_constant"])
    _absent = [v for k, v in STRUCT_KEYS.items() if k not in _vals]
    if _absent:
        print(f"WARNING: snap {_snap} caesar file lacks {_absent}")
    for _g in _gids:
        if _g < len(_pos):
            _structdb[(_snap, _g)] = {k: float(_arr[_g]) for k, _arr in _vals.items()}
            _posdb[(_snap, _g)]    = (np.asarray(_pos[_g], float), _a, _h)
print(f"structure props: {len(_structdb)} (snap,gal) rows from the anchor caesar catalogs")

def _sp_arr(tab, key):
    """anchor-epoch structure prop `key` aligned to a catalog table's rows."""
    return np.array([_structdb.get((int(s), int(g)), {}).get(key, np.nan)
                     for s, g in zip(tab["snap"], tab["gal_id"])])

# ── 1c. kappa_rot of the H2 gas disk from the Stage-0 region cutouts ──
def _kappa_rot_h2(snap, gid):
    """Sales+12 kappa_rot of the H2-mass-weighted gas within R_KROT_KPC (proper).

    Cutout Coordinates are code units (ckpc/h) unwrapped around the caesar centre,
    so pos_kpccm*h recovers that centre exactly; the uniform sqrt(a) factor of the
    code velocities cancels in the K_rot/K ratio.
    """
    rec = _posdb.get((int(snap), int(gid)))
    if rec is None:
        return np.nan
    pos_kpccm, a_scale, hub = rec
    cut = read_cutout(snap, gid, pos_kpccm * hub, "PartType0",
                      fields=("Velocities", "Masses", "FractionH2"))
    if cut is None or cut["FractionH2"] is None:
        return np.nan
    r   = cut["pos"]                                            # proper kpc, gal frame
    v   = np.asarray(cut["Velocities"], float)
    mh2 = np.asarray(cut["Masses"], float) * np.asarray(cut["FractionH2"], float)
    sel = (np.sqrt(np.sum(r**2, axis=1)) < R_KROT_KPC) & (mh2 > 0) & np.isfinite(mh2)
    if sel.sum() < 10:
        return np.nan
    r, vv, w = r[sel], v[sel], mh2[sel]
    r  = r  - np.average(r,  axis=0, weights=w)                 # recentre on the H2 body
    vv = vv - np.average(vv, axis=0, weights=w)
    j  = np.cross(r, vv)
    L  = np.sum(w[:, None] * j, axis=0)
    if not np.isfinite(L).all() or np.linalg.norm(L) == 0:
        return np.nan
    zhat = L / np.linalg.norm(L)
    jz   = j @ zhat
    Rcyl = np.sqrt(np.maximum(np.sum(r**2, axis=1) - (r @ zhat)**2, 0.0))
    ok   = Rcyl > 1e-3
    Krot = 0.5 * np.sum(w[ok] * (jz[ok] / Rcyl[ok])**2)
    Ktot = 0.5 * np.sum(w * np.sum(vv**2, axis=1))
    return float(Krot / Ktot) if Ktot > 0 else np.nan

# ── 2. per-aperture attenuation table (dust_on ⨝ dust_off on snap+gal_id) ──
ATTEN_INCL = INCL_LABELS[0]     # fiducial sightline for the A_λ analysis
def _load_atten(label, incl=None):
    incl = ATTEN_INCL if incl is None else incl
    fon  = os.path.join(CATDIR, f"catalog_dust_on_{label}_{incl}.fits")
    foff = os.path.join(CATDIR, f"catalog_dust_off_{label}_{incl}.fits")
    if not (os.path.exists(fon) and os.path.exists(foff)):
        return None
    on, off = Table.read(fon), Table.read(foff)
    off_cols = ["snap", "gal_id"] + [c for c in ATTEN_BANDS.values() if c in off.colnames]
    m = join(on, off[off_cols], keys=["snap", "gal_id"],
             table_names=["on", "off"], metadata_conflicts="silent")
    for aname, col in ATTEN_BANDS.items():
        m[aname] = (_atten(m[f"{col}_on"], m[f"{col}_off"])
                    if f"{col}_on" in m.colnames and f"{col}_off" in m.colnames
                    else np.full(len(m), np.nan))
    return m

_atab = {lab: _load_atten(lab) for lab in APERTURE_LABELS}
_atab = {k: v for k, v in _atab.items() if v is not None and len(v)}
if not _atab:
    raise FileNotFoundError(f"no dust_on/dust_off catalogs in {CATDIR}; run Part 7 first")
FID_AP = APERTURE_LABELS[-1] if APERTURE_LABELS[-1] in _atab else list(_atab)[-1]
print(f"apertures with catalogs: {list(_atab)}  |  fiducial (global A_V) = {FID_AP}"
      f"  |  sightline = {ATTEN_INCL}")

# ── 3. fiducial-aperture analysis table: A_λ + ISM tracers + structure + quench ──
base  = _atab[FID_AP]
_MH2  = _gp_arr(base, "masses.H2");   _MHI = _gp_arr(base, "masses.HI")
_Mgas = _gp_arr(base, "masses.gas");  _Md  = _gp_arr(base, "masses.dust")
_Mst  = _gp_arr(base, "masses.stellar")
with np.errstate(all="ignore"):
    f_mol       = _MH2 / (_MH2 + _MHI)          # molecular fraction of neutral gas
    f_H2_star   = _MH2 / _Mst                    # specific molecular content
    f_gas       = _Mgas / (_Mgas + _Mst)         # gas fraction
    DGR         = _Md / _Mgas                     # dust-to-gas ratio
    f_dust_star = _Md / _Mst                      # f_dust: specific dust content

ATTEN = Table()
ATTEN["snap"]     = np.asarray(base["snap"], int)
ATTEN["gal_id"]   = np.asarray(base["gal_id"], int)
ATTEN["z_target"] = np.asarray(base["z_target"], float)
for aname in ATTEN_BANDS:
    ATTEN[aname] = np.asarray(base[aname], float)
ATTEN["S_UV"] = ATTEN["A_U"] - ATTEN["A_V"]      # attenuation-curve slope proxy (mag)
for lab, tt in _atab.items():                    # enclosed A_V at every aperture
    idx = {(int(s), int(g)): k for k, (s, g) in enumerate(zip(tt["snap"], tt["gal_id"]))}
    col = np.full(len(base), np.nan)
    for k, (s, g) in enumerate(zip(base["snap"], base["gal_id"])):
        j = idx.get((int(s), int(g)))
        if j is not None:
            col[k] = tt["A_V"][j]
    ATTEN[f"A_V_{lab}"] = col
with np.errstate(all="ignore"):
    ATTEN["log_MH2"]   = np.log10(np.where(_MH2 > 0, _MH2, np.nan))
    ATTEN["log_Mgas"]  = np.log10(np.where(_Mgas > 0, _Mgas, np.nan))
    ATTEN["log_Mdust"] = np.log10(np.where(_Md > 0, _Md, np.nan))
ATTEN["f_mol"] = f_mol; ATTEN["f_H2_star"] = f_H2_star; ATTEN["f_gas"] = f_gas
ATTEN["DGR"] = DGR; ATTEN["f_dust_star"] = f_dust_star
for c in ("log_mstar", "ssfr", "tau_q", "tau_q_over_tH", "xstr_quench"):
    ATTEN[c] = np.asarray(base[c], float)
_agn = np.asarray(base["agn_class"])
ATTEN["agn_class"] = np.array([a.decode() if isinstance(a, (bytes, np.bytes_)) else str(a)
                               for a in _agn])

# anchor-epoch stellar structure + gas-disk rotation (observation time)
ATTEN["age_mw"] = _gp_arr(base, "ages.mass_weighted")          # Gyr, mass-weighted
with np.errstate(all="ignore"):
    ATTEN["logZ_star"] = np.log10(_sp_arr(base, "Z_star") / ZSUN)
    ATTEN["logZ_gas"]  = np.log10(_sp_arr(base, "Z_gas") / ZSUN)
ATTEN["BT_star"]   = _sp_arr(base, "BT_star")
ATTEN["kappa_gas"] = _sp_arr(base, "kappa_gas")
ATTEN["kappa_H2"]  = np.array([_kappa_rot_h2(s, g)
                               for s, g in zip(base["snap"], base["gal_id"])])
print(f"kappa_H2 (<{R_KROT_KPC:g} pkpc): measured for "
      f"{int(np.isfinite(np.asarray(ATTEN['kappa_H2'])).sum())}/{len(ATTEN)} galaxies "
      f"(needs the Stage-0 cutouts + >=10 H2-bearing gas particles)")

# ── 3b. annular A_V — the observational radial profile: F(<r_out) - F(<r_in) ──
_labs_all = [l for l in APERTURE_LABELS if l in _atab]
_r_out    = np.array([APERTURE_RADII_KPC[WANTED_AP_IDX[APERTURE_LABELS.index(l)]]
                      for l in _labs_all])                     # TRUE rung radii [pkpc]
_r_in     = np.concatenate([[0.0], _r_out[:-1]])
_r_mid    = np.where(_r_in > 0, np.sqrt(_r_in * _r_out), _r_out / 2.0)

def _band_matrix(col):
    """(n_gal, n_ap) matrix of catalog column `col`, aligned to the base rows."""
    M = np.full((len(base), len(_labs_all)), np.nan)
    for k, lab in enumerate(_labs_all):
        tt = _atab[lab]
        if col not in tt.colnames:
            continue
        idx = {(int(s), int(g)): j for j, (s, g) in enumerate(zip(tt["snap"], tt["gal_id"]))}
        for i, (s, g) in enumerate(zip(base["snap"], base["gal_id"])):
            j = idx.get((int(s), int(g)))
            if j is not None:
                M[i, k] = tt[col][j]
    return M

_Von, _Voff = _band_matrix("Johnson.V.V_on"), _band_matrix("Johnson.V.V_off")
_dVon  = np.column_stack([_Von[:, :1],  np.diff(_Von,  axis=1)])   # annular fluxes
_dVoff = np.column_stack([_Voff[:, :1], np.diff(_Voff, axis=1)])
AV_ANN = _atten(_dVon, _dVoff)
for k, lab in enumerate(_labs_all):
    ATTEN[f"A_V_ann_{lab}"] = AV_ANN[:, k]
_nneg = int(np.sum(((_dVon <= 0) | (_dVoff <= 0)) & np.isfinite(_Von) & np.isfinite(_Voff)))
print(f"annular A_V: {len(_labs_all)} annuli/galaxy; {_nneg} annuli with non-positive "
      f"differential flux (MC noise / empty annulus) -> NaN")

_out = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")
ATTEN.write(_out, overwrite=True)
_av = np.asarray(ATTEN["A_V"], float)
print(f"attenuation table: {len(ATTEN)} galaxies ({FID_AP}) -> {_out}")
print(f"A_V [{FID_AP}]  median={np.nanmedian(_av):.3f}  90th={np.nanpercentile(_av,90):.3f}  "
      f"max={np.nanmax(_av):.3f}   (A_V>0.1 mag: {int(np.nansum(_av>0.1))}/{len(ATTEN)})")

# ── 4. Spearman correlations of the global A_V with everything ──
_targets = [("f_mol","f_mol"), ("f_H2_star","M_H2/M*"), ("f_gas","f_gas"),
            ("DGR","dust/gas"), ("f_dust_star","f_dust"), ("log_MH2","log M_H2"),
            ("log_Mgas","log M_gas"), ("kappa_H2","kappa_H2"), ("kappa_gas","kappa_gas"),
            ("log_mstar","log M*"), ("age_mw","age_mw"), ("logZ_star","log Z*/Zsun"),
            ("logZ_gas","log Zg/Zsun"), ("BT_star","B/T"), ("ssfr","sSFR"),
            ("tau_q","tau_q"), ("tau_q_over_tH","tau_q/t_H"), ("S_UV","A_U-A_V")]
_ranked = []
for col, lbl in _targets:
    x = np.asarray(ATTEN[col], float); ok = np.isfinite(_av) & np.isfinite(x)
    if ok.sum() >= 5:
        rho, p = spearmanr(_av[ok], x[ok]); _ranked.append((lbl, rho, p, int(ok.sum())))
_ranked.sort(key=lambda r: -abs(r[1]))
print("\nSpearman  A_V  vs …   (fiducial aperture, sorted by |rho|)")
print(f"  {'quantity':12s} {'rho':>7s} {'p':>10s} {'n':>4s}")
for lbl, rho, p, n in _ranked:
    flag = "***" if p < 0.01 else "** " if p < 0.05 else "*  " if p < 0.1 else ""
    print(f"  {lbl:12s} {rho:+7.3f} {p:10.2e} {n:4d}  {flag}")

# per-aperture robustness of the two headline ISM correlations
print("\nrobustness across apertures  (rho[p]):")
for lab in _labs_all:
    tt = _atab[lab]; av = np.asarray(tt["A_V"], float)
    with np.errstate(all="ignore"):
        fh2 = _gp_arr(tt, "masses.H2") / _gp_arr(tt, "masses.stellar")
        dgr = _gp_arr(tt, "masses.dust") / _gp_arr(tt, "masses.gas")
    def _rp(x):
        ok = np.isfinite(av) & np.isfinite(x)
        return spearmanr(av[ok], x[ok]) if ok.sum() >= 5 else (np.nan, np.nan)
    (r1, p1), (r2, p2) = _rp(fh2), _rp(dgr)
    print(f"  {lab:16s}  M_H2/M*: {r1:+.2f}[{p1:.2g}]   dust/gas: {r2:+.2f}[{p2:.2g}]")

# ── 5. figures ──
_cls_present = [c for c in ["strong","intermediate","weak","no_AGN","no_event","unclassified"]
                if c in set(ATTEN["agn_class"])]
def _scatter(ax, xcol, xlabel, xlog=False):
    x = np.asarray(ATTEN[xcol], float); y = _av
    for cls in _cls_present:
        s = ATTEN["agn_class"] == cls
        ax.scatter(x[s], y[s], s=28, c=AGN_COLORS.get(cls, "#333"), label=cls,
                   edgecolor="k", linewidth=0.3, alpha=0.85)
    ok = np.isfinite(x) & np.isfinite(y)
    if xlog: ok &= x > 0
    if ok.sum() >= 5:
        rho, p = spearmanr(x[ok], y[ok])
        ax.text(0.04, 0.95, f"$\\rho$={rho:+.2f}\np={p:.2g}", transform=ax.transAxes,
                va="top", fontsize=8.5, bbox=dict(fc="white", ec="0.7", alpha=0.85, pad=1.6))
    if xlog:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel); ax.set_ylabel(r"$A_V$ [mag]")

# Fig 1 — attenuation vs ISM / dust content (the H2 connection)
_p1 = [("f_mol", r"$f_{\rm mol}=M_{H_2}/(M_{H_2}\!+\!M_{HI})$", False),
       ("f_H2_star", r"$M_{H_2}/M_\star$", True),
       ("f_gas", r"$f_{\rm gas}=M_{\rm gas}/(M_{\rm gas}\!+\!M_\star)$", False),
       ("DGR", r"dust-to-gas $M_{\rm dust}/M_{\rm gas}$", True),
       ("f_dust_star", r"$f_{\rm dust}=M_{\rm dust}/M_\star$", True),
       ("kappa_H2", r"$\kappa_{\rm rot}^{H_2}$ ($<$%g pkpc)" % R_KROT_KPC, False)]
fig, axes = plt.subplots(2, 3, figsize=(21, 12))
for ax, (c, xl, xlog) in zip(axes.flat, _p1):
    _scatter(ax, c, xl, xlog)
axes.flat[0].legend(fontsize=10, loc="upper right", framealpha=0.9)
fig.suptitle(f"$A_V$ vs ISM ({FID_AP})", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
_f1 = os.path.join(PLOTDIR, "attenuation_vs_ism.png")
fig.savefig(_f1, dpi=130, bbox_inches="tight"); plt.show()
print("saved", _f1)

# Fig 2 — attenuation vs quenching + stellar structure at the observation epoch
_p2 = [("ssfr", "sSFR [yr$^{-1}$]", True),
       ("log_mstar", r"$\log_{10} M_\star\,[M_\odot]$", False),
       ("age_mw", "mass-weighted age [Gyr]", False),
       ("logZ_star", r"$\log_{10} Z_\star/Z_\odot$", False),
       ("BT_star", r"stellar $B/T$", False),
       ("tau_q", r"$\tau_q$ [yr]", False),
       ("tau_q_over_tH", r"$\tau_q/t_H$", False)]
fig, axes = plt.subplots(2, 4, figsize=(26, 12))
for ax, (c, xl, xlog) in zip(axes.flat[:7], _p2):
    _scatter(ax, c, xl, xlog)
ax = axes.flat[7]
for i, cls in enumerate(_cls_present):
    s = ATTEN["agn_class"] == cls; yv = _av[np.asarray(s)]
    xj = i + np.random.uniform(-0.16, 0.16, size=int(np.sum(s)))
    ax.scatter(xj, yv, c=AGN_COLORS.get(cls, "#333"), edgecolor="k", linewidth=0.3, s=28)
    if np.isfinite(yv).any():
        ax.hlines(np.nanmedian(yv), i - 0.3, i + 0.3, color="k", lw=2)
ax.set_xticks(range(len(_cls_present)))
ax.set_xticklabels(_cls_present, rotation=30, ha="right", fontsize=10)
ax.set_ylabel(r"$A_V$ [mag]"); ax.set_title("by AGN class")
fig.suptitle(f"$A_V$ vs quenching ({FID_AP})", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
_f2 = os.path.join(PLOTDIR, "attenuation_vs_quench.png")
fig.savefig(_f2, dpi=130, bbox_inches="tight"); plt.show()
print("saved", _f2)

# Fig 3 — radial A_V from annular fluxes + attenuation-curve slope
fig, (axL, axR) = plt.subplots(1, 2, figsize=(17, 7))
_AVap  = np.vstack([np.asarray(ATTEN[f"A_V_{l}"], float) for l in _labs_all]).T
_AVann = np.vstack([np.asarray(ATTEN[f"A_V_ann_{l}"], float) for l in _labs_all]).T
_dusty = _av > AV_DUSTY                                   # most quenched gals are transparent;
for row in _AVann[_dusty]:                           # the radial trend only matters there
    axL.plot(_r_mid, row, "-", color="0.7", lw=0.8, alpha=0.7, zorder=1)
axL.plot(_r_mid, np.nanmedian(_AVann[_dusty], axis=0), "o-", color="#c0392b", lw=2,
         label=f"annular median, $A_V\\!>\\!0.1$ (n={int(_dusty.sum())})", zorder=3)
axL.plot(_r_mid, np.nanmedian(_AVann, axis=0), "s--", color="#2980b9",
         label=f"annular median, all (n={len(_av)})", zorder=2)
axL.plot(_r_out, np.nanmedian(_AVap[_dusty], axis=0), ":", color="0.35", lw=1.5,
         label=r"enclosed $A_V(<r)$, $A_V\!>\!0.1$", zorder=2)
axL.set_xscale("log"); axL.set_xlabel("radius [pkpc]")
axL.set_ylabel(r"$A_V$ [mag]")
axL.set_title(r"radial $A_V$ (annuli)")
axL.legend(fontsize=10, frameon=False)
for cls in _cls_present:
    s = ATTEN["agn_class"] == cls
    axR.scatter(_av[np.asarray(s)], np.asarray(ATTEN["S_UV"])[np.asarray(s)], s=28,
                c=AGN_COLORS.get(cls, "#333"), edgecolor="k", linewidth=0.3, alpha=0.85, label=cls)
axR.axhline(0, color="0.6", lw=0.8, ls="--")
axR.set_xlabel(r"$A_V$ [mag]"); axR.set_ylabel(r"$A_U-A_V$ [mag]  (curve slope)")
axR.legend(fontsize=10, frameon=False)
fig.tight_layout()
_f3 = os.path.join(PLOTDIR, "attenuation_aperture_curve.png")
fig.savefig(_f3, dpi=130, bbox_inches="tight"); plt.show()
print("saved", _f3)

# Part 7b — CIGALE input files (one per RT arm × aperture × sightline)

Writes `output/cis25/sed_aperture_catalogs/cigale/cigale_{dust_on|dust_off}_{ap…}.fits` in the
exact input format of **CIGALE 2025.0**. All the format/mapping logic lives in
**`simbanator.sed.cigale`** (band names verified against the 2025.0 filter database):

- columns `id` (`snapNNN_galID`), `redshift`, `distance` (Mpc, Planck13 — the same D_L used to
  normalize the fluxes), then per band the flux **in mJy** + its `<band>_err`;
- band names match the CIGALE DB exactly (`jwst.nircam.F200W`, `hst.wfc3.ir.F160W`,
  `spitzer.irac.I1`, `herschel.pacs.green`, `2mass.J`, `generic.johnson.U/V`, …); bands with no
  CIGALE counterpart (grisms, quad filters) are dropped and reported;
- missing fluxes are NaN; make an error negative by hand for upper-limit treatment.

Two deliberate choices:

1. **Observed frame.** CIGALE compares redshifted models to observed photometry, so the
   extraction reruns with `redshift=True` (the Part 7 catalogs stay rest-frame).
2. **Raw MC errors (`err_floor=0`).** CIGALE itself adds `additionalerror` (10 % by default,
   set in Part 7es `prepare_run`) in quadrature at fit time — a floor here too would be
   double-counted. The Hyperion MC error alone is just RT convergence noise.

In [ ]:
# ── Part 7b: CIGALE 2025.0 input files — observed-frame fluxes+errors, one per (RT arm, aperture, sightline) ──
# All three arms are written; Part 7d fits dust_on and agn_on (plus a small
# dust_off zero-point control). dust_off is otherwise the A_V reference only.
# Format + band mapping live in simbanator.sed.cigale (verified against the 2025.0 filter DB).
# Needs the Part 5 MakeSED handles + the Part 7 filter/extractor cell in this session.
from simbanator.sed.cigale import write_cigale_input

os.makedirs(CIGALE_DIR, exist_ok=True)

# CIGALE compares redshifted models to observed photometry -> observed frame
# (the Part 7 catalogs stay rest-frame)
CIGALE_FLUX_FILES = extract_flux_set(redshift=True, prefix="cigale_fluxes")

CIGALE_FILES = {}
for (key, label, ilab), flux_file in CIGALE_FLUX_FILES.items():
    # err_floor=0: CIGALE adds its own 10% 'additionalerror' in quadrature at fit time
    CIGALE_FILES[(key, label, ilab)] = write_cigale_input(
        flux_file, os.path.join(CIGALE_DIR, f"cigale_{key}_{label}_{ilab}.fits"),
        err_floor=0.0)

print(f"\n{len(CIGALE_FILES)} CIGALE input files -> {CIGALE_DIR}")


# Part 7c — aperture-matched **formed-mass** SFH archive (the injected prior)

Part 7d fits every galaxy with **its own** star-formation history: one CIGALE run per object,
`sfhfromfile` with a single column. This cell builds the archive those runs read.

For each selected galaxy it takes the **archaeological** SFH — mass formed per 100 Myr from the
star-particle formation times in the Stage-0 cutout, i.e. the exact particles the RT saw — smooths
it (`smooth_resample_sfh`, 25 Myr grid, 150 Myr Gaussian kernel) and stores it as
`cigale/sfh_smoothed_aperture.h5`, keyed `snapNNN_galID/<sightline>/<aperture>`.

**Why per aperture and per sightline.** The photometry Part 7b extracts is aperture- and
sightline-resolved, so the stellar population inside `ap3kpc` along `i0p0` is *not* the one the
global history track describes — inner apertures are older and more quenched. Injecting the global
SFH into an aperture fit would import exactly the age–dust degeneracy this exercise removes. The
binning reuses the Part 7e geometry (`read_cutout` → `projected_radius` → cumulative rungs), so
archive and `aperture_truth.fits` describe the same particles.

## 2026-08-10 — formed mass, not surviving mass

The archive now stores the **formed**-mass SFR, and this is not cosmetic. Powderday scales each
star particle's FSPS spectrum by

```python
lum = trapz(fnu, nu) * star.mass / mfrac        # source_creation.py:63
mfrac = sp.stellar_mass                          # SED_gen.py:407
```

— FSPS SSPs are normalised to **1 M<sub>⊙</sub> formed**, so powderday divides by the surviving
fraction to recover the initial mass. A histogram of the *current* particle masses therefore sits
low by `mfrac ≈ 0.55` at the old end and ≈ 1.0 at the young end: an **~80 % tilt** across the
curve. `normalise=True` throws the amplitude away, so that tilt is *all* that survives into the
fit — and with the SFH pinned there is nothing left to absorb it except $A_V$. An intrinsically
too-blue model is paid for with extra dust, biasing the measurand high.

CIGALE wants the same correction independently: `sfhfromfile`'s SFR feeds `bc03.convolve`, which
applies BC03's *own* mass loss to produce `stellar.m_star`. Feeding it surviving-mass rates
double-counts the mass loss. Matching the mock and being physically right are the same edit here.

`build_mfrac_lookup()` reproduces powderday's surviving fraction from **`simbanator/sed/parameters_master.py`** — the file
`makesed.py` copies into every model dir, *not* `~/powderday/parameters_master.py` (which ships
`imf_type=2` and is never used by this campaign): Chabrier `imf_type=1`, `pagb=1`,
`add_agb_dust_model=True`, and `add_stellar_remnants` left at the FSPS default so remnants are
included. One `get_spectrum(tage=0)` per metallicity returns the whole SSP age grid, so
`sp.stellar_mass` comes back as an array — 12 FSPS calls (~3 min once, then cached to
`tables/fsps_mfrac_lookup.npz`), not one per particle. Metallicity is snapped in **linear** space
to match powderday's `find_nearest_zmet`.

The HDF5 records `sfh_mass_kind` / `mfrac_source` in its attrs, and **Part 7d refuses to build
against an archive that is not `formed`** — a stale file must never be mistakable for a corrected
one.

**Caveats.** Apertures holding fewer than `SFH_NSTAR_MIN` star particles are left out; under a
per-object pin that means those (galaxy, aperture, sightline) combinations simply get **no run**,
so the coverage table below is a sample cut and is reported as one — `ap1kpc` spans only ~3–4
softening lengths at m25n512 and will be the worst hit. The smoothed grid spans the histogram bin
*centres*, so the last ~50 Myr is edge-held by the interpolation in Part 7d; `sfr_hold_frac` there
records how much of the recent SFR that affects. Times are on the notebook's `COSMO` (Planck15);
Part 7d re-caps the age against pcigale's own Planck18 ceiling.

In [ ]:
# ── Part 7c — aperture-matched FORMED-mass SFH archive (the injected prior) ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts (cluster). Writes one
# smoothed archaeological SFH per (galaxy, sightline, cumulative aperture) —
# the SAME projected geometry Part 7b extracted the photometry in, and the same
# particles Part 7e measures its truth from. Part 7d injects exactly one of
# these per CIGALE run via sfhfromfile (shape only, normalise=True).
#
# 2026-08-10: masses are divided by the FSPS surviving fraction before the
# histogram, so the stored SFR is the FORMED mass per year — what powderday
# actually rendered (source_creation.py:63 scales each SSP by mass/mfrac) and
# what bc03 expects (it applies its own mass loss on top). See the markdown.
from simbanator.analysis.sfh_utils import smooth_resample_sfh

SFH_APERTURE_H5       = os.path.join(CIGALE_DIR, "sfh_smoothed_aperture.h5")
OVERWRITE_SFH_ARCHIVE = False
SFH_ARCH_BIN_MYR      = 100.0    # archaeological bin width before smoothing
SFH_ARCH_DT_MYR       = 25.0     # smoothed output grid step
SFH_ARCH_KERNEL_MYR   = 150.0    # Gaussian kernel sigma
SFH_NSTAR_MIN         = 20       # fewer star particles -> shot-noise, not an SFH

# ── powderday's FSPS settings: simbanator/sed/parameters_master.py, the file
#    makesed.py copies into every model dir. NOT ~/powderday/parameters_master.py
#    (imf_type=2, add_agb_dust_model=False) — that one is never used here.
#    Only imf_type and add_stellar_remnants actually move sp.stellar_mass;
#    the rest are set anyway so the lookup is provably the same population.
PD_IMF_TYPE    = 1            # Chabrier
PD_PAGB        = 1
PD_AGB_DUST    = True
# add_stellar_remnants is never set in the master -> FSPS default 1, i.e. mfrac
# INCLUDES remnants. Keep the default so the lookup matches powderday exactly.
FSPS_MFRAC_NPZ = os.path.join(TABLEDIR, "fsps_mfrac_lookup.npz")
APPLY_MFRAC    = True
MFRAC_FALLBACK = "none"       # 'analytic' -> Chabrier R(t); 'none' -> refuse

os.makedirs(CIGALE_DIR, exist_ok=True)


def build_mfrac_lookup(path=FSPS_MFRAC_NPZ, overwrite=False):
    """(zlegend, log_age_yr, mfrac[nz, nage], source) — FSPS surviving fraction.

    get_spectrum(tage=0) returns the WHOLE SSP age grid, so sp.stellar_mass
    comes back as an (nage,) array: one FSPS call per metallicity instead of
    one per particle. Cached (loading the SSPs costs ~15 s per zmet).
    """
    if os.path.exists(path) and not overwrite:
        d = np.load(path)
        return (d["zlegend"], d["log_age_yr"], d["mfrac"], str(d["source"]))
    import fsps
    sp = fsps.StellarPopulation(imf_type=PD_IMF_TYPE, pagb=PD_PAGB, sfh=0,
                                add_agb_dust_model=PD_AGB_DUST,
                                add_neb_emission=False)  # no effect on m_star
    zleg, rows, log_age = np.asarray(sp.zlegend, float), [], None
    for iz in range(1, len(zleg) + 1):            # FSPS zmet is 1-based
        sp.params["zmet"] = iz
        sp.get_spectrum(tage=0)
        log_age = np.asarray(sp.log_age, float)          # log10(yr)
        rows.append(np.asarray(sp.stellar_mass, float))  # (nage,)
    mfrac = np.vstack(rows)
    src = (f"fsps {fsps.__version__} libs={sp.libraries} "
           f"imf_type={PD_IMF_TYPE} pagb={PD_PAGB} agb_dust={PD_AGB_DUST} "
           f"remnants={sp.params['add_stellar_remnants']}")
    np.savez(path, zlegend=zleg, log_age_yr=log_age, mfrac=mfrac, source=src)
    return zleg, log_age, mfrac, src


def mfrac_of(age_gyr, zstar, zleg, log_age_yr, mfrac):
    """Surviving-mass fraction powderday used for these particles.

    Nearest metallicity node in LINEAR space — powderday's find_nearest_zmet
    (SED_gen.py) is argmin(|zlegend - Z|), not a log-space snap.
    """
    iz = np.argmin(np.abs(np.asarray(zstar, float)[:, None] - zleg[None, :]),
                   axis=1)
    la = np.log10(np.clip(np.asarray(age_gyr, float), 1e-4, None) * 1e9)
    la = np.clip(la, log_age_yr[0], log_age_yr[-1])
    out = np.empty(la.shape, float)
    for k in np.unique(iz):
        m = iz == k
        out[m] = np.interp(la[m], log_age_yr, mfrac[k])
    return np.clip(out, 0.05, 1.0)


def load_sfh_archive(path=None):
    """{id: {incl: {aperture: (t_gyr, sfr, t_obs_gyr)}}} from the archive."""
    path = path or SFH_APERTURE_H5
    out = {}
    with h5py.File(path, "r") as f:
        for sid in f:
            for il in f[sid]:
                for ap in f[sid][il]:
                    d = f[sid][il][ap]
                    arr = np.asarray(d[:], float)
                    out.setdefault(str(sid), {}).setdefault(str(il), {})[str(ap)] = (
                        arr[:, 0], arr[:, 1], float(d.attrs["t_obs_gyr"]))
    return out


def sfh_archive_attrs(path=None):
    """Provenance attrs of the archive (empty dict if it does not exist)."""
    path = path or SFH_APERTURE_H5
    if not os.path.exists(path):
        return {}
    with h5py.File(path, "r") as f:
        return {k: (v.decode() if isinstance(v, bytes) else v)
                for k, v in f.attrs.items()}


if os.path.exists(SFH_APERTURE_H5) and not OVERWRITE_SFH_ARCHIVE:
    _at = sfh_archive_attrs()
    _arch = load_sfh_archive()
    _n = sum(len(a) for g in _arch.values() for a in g.values())
    print(f"cached: {_n} aperture SFHs for {len(_arch)} galaxies -> "
          f"{SFH_APERTURE_H5}  (OVERWRITE_SFH_ARCHIVE=True rebuilds)")
    print(f"   sfh_mass_kind={_at.get('sfh_mass_kind', '?')}  "
          f"mfrac_source={_at.get('mfrac_source', '?')}")
    if str(_at.get("sfh_mass_kind", "")) != "formed":
        print("   *** this archive predates the formed-mass fix — Part 7d will "
              "refuse it. Set OVERWRITE_SFH_ARCHIVE=True and re-run. ***")
else:
    # ── the surviving-mass fraction, exactly as powderday computed it ──
    _MF_TAG = "UNCORRECTED"
    _ZLEG = _LAGE = _MFRAC = None
    if APPLY_MFRAC:
        try:
            _ZLEG, _LAGE, _MFRAC, _MF_TAG = build_mfrac_lookup()
            print(f"[mfrac] {_MF_TAG}")
            print(f"[mfrac] {len(_ZLEG)} Z nodes x {len(_LAGE)} ages; "
                  f"mfrac(1 Gyr, Zsol)={np.interp(9.0, _LAGE, _MFRAC[np.argmin(np.abs(_ZLEG - 0.019))]):.3f}  "
                  f"mfrac(10 Gyr, Zsol)={np.interp(10.0, _LAGE, _MFRAC[np.argmin(np.abs(_ZLEG - 0.019))]):.3f}")
        except ImportError as _exc:
            if MFRAC_FALLBACK != "analytic":
                raise RuntimeError(
                    f"fsps is not importable in this kernel ({_exc}) and "
                    "MFRAC_FALLBACK='none'. Without it the archive would hold "
                    "CURRENT-mass SFHs, tilted ~80% against the oldest bins "
                    "relative to what powderday rendered — and with the SFH "
                    "pinned that tilt lands directly in A_V. Run this cell in "
                    "the pd39 kernel, or set MFRAC_FALLBACK='analytic'.") from _exc
            # Chabrier return fraction R(t) = C ln(t/lam + 1) (Jungwiert+2001):
            # within ~5% of FSPS/MIST beyond 1 Gyr, and normalise=True only
            # ever sees the SHAPE.
            _MF_TAG = "analytic-chabrier"
            print(f"[mfrac] fsps unavailable ({_exc}) — {_MF_TAG} fallback")

    def _mfrac(age_gyr, zstar):
        if not APPLY_MFRAC:
            return np.ones_like(np.asarray(age_gyr, float))
        if _MFRAC is not None:
            return mfrac_of(age_gyr, zstar, _ZLEG, _LAGE, _MFRAC)
        return np.clip(1.0 - 0.05 * np.log(np.asarray(age_gyr, float) * 1e3 / 0.4
                                           + 1.0), 0.05, 1.0)

    SEL, SNAPS, IDS = load_selection()
    _cen = rt_centers(SNAPS, IDS)
    _ag = np.linspace(0.02, 1.0, 4096)               # a -> cosmic time grid
    _tg = COSMO.age(1.0 / _ag - 1.0).value           # Gyr
    _apr = APERTURE_RADII_KPC[np.asarray(WANTED_AP_IDX)]

    _nw = _ngal_ok = _nskip = 0
    _demo = {}                                       # QC figure payload
    with h5py.File(SFH_APERTURE_H5, "w") as f5:
        f5.attrs["bin_myr"] = SFH_ARCH_BIN_MYR
        f5.attrs["dt_myr"] = SFH_ARCH_DT_MYR
        f5.attrs["kernel_myr"] = SFH_ARCH_KERNEL_MYR
        f5.attrs["nstar_min"] = SFH_NSTAR_MIN
        f5.attrs["cosmology"] = COSMO.name
        f5.attrs["mfrac_applied"] = bool(APPLY_MFRAC)
        f5.attrs["mfrac_source"] = _MF_TAG
        f5.attrs["sfh_mass_kind"] = ("formed" if APPLY_MFRAC and
                                     _MF_TAG != "UNCORRECTED" else "current")
        for _snap in np.unique(SNAPS):
            _z = float(sim.get_z_from_snap(int(_snap)))
            _t_obs = float(COSMO.age(_z).value)
            _edges = np.arange(0.0, _t_obs + SFH_ARCH_BIN_MYR / 1e3,
                               SFH_ARCH_BIN_MYR / 1e3)
            _tc = 0.5 * (_edges[:-1] + _edges[1:])
            _ns = 0
            _mfs, _mcur, _mform = [], 0.0, 0.0
            for _gid in np.unique(IDS[SNAPS == _snap]):
                _cut = read_cutout(_snap, _gid,
                                   _cen.get((int(_snap), int(_gid))), "PartType4",
                                   fields=("Masses", "StellarFormationTime",
                                           "Metallicity"))
                if _cut is None or _cut.get("StellarFormationTime") is None:
                    print(f"  [skip] snap {_snap} gal {_gid}: no cutout/centre/stars")
                    _nskip += 1
                    continue
                _a = np.asarray(_cut["StellarFormationTime"], float)
                _ok = np.isfinite(_a) & (_a > 0) & (_a <= 1)
                if int(_ok.sum()) < SFH_NSTAR_MIN:
                    _nskip += 1
                    continue
                _tf = np.interp(np.clip(_a[_ok], _ag[0], 1.0), _ag, _tg)   # Gyr
                _m0 = np.asarray(_cut["Masses"], float)[_ok] * 1e10 / _cut["h"]
                _pos = _cut["pos"][_ok]
                # current -> FORMED mass (powderday's mass/mfrac scaling)
                _Zp = _cut.get("Metallicity")
                if _Zp is None:
                    _Zs = np.full(_m0.shape, 0.019)   # solar-ish fallback
                else:
                    _Zp = np.asarray(_Zp, float)
                    _Zs = (_Zp[:, 0] if _Zp.ndim == 2 else _Zp)[_ok]
                _mf = _mfrac(np.clip(_t_obs - _tf, 1e-4, None), _Zs)
                _mm = _m0 / _mf
                _mfs.append(_mf)
                _mcur += float(_m0.sum())
                _mform += float(_mm.sum())
                _sid = f"snap{int(_snap):03d}_gal{int(_gid)}"
                _ngal_ok += 1
                _ns += 1
                for _il, _nv in zip(INCL_LABELS, NHAT):
                    _rp = projected_radius(_pos, _nv)
                    for _lab, _r in zip(APERTURE_LABELS, _apr):
                        _msk = _rp <= _r
                        _nap = int(_msk.sum())
                        if _nap < SFH_NSTAR_MIN:
                            continue
                        _h, _ = np.histogram(_tf[_msk], bins=_edges,
                                             weights=_mm[_msk])
                        _ts, _ss = smooth_resample_sfh(
                            _tc, _h / (SFH_ARCH_BIN_MYR * 1e6),
                            dt_myr=SFH_ARCH_DT_MYR, kernel_myr=SFH_ARCH_KERNEL_MYR)
                        _d = f5.create_dataset(f"{_sid}/{_il}/{_lab}",
                                               data=np.column_stack([_ts, _ss]))
                        _d.attrs["t_obs_gyr"] = _t_obs
                        _d.attrs["nstar"] = _nap
                        _nw += 1
                        # QC: keep the uncorrected twin for 3 galaxies, ap100kpc
                        if (len(_demo) < 3 and _il == INCL_LABELS[0]
                                and _lab == APERTURE_LABELS[-1]):
                            _h0, _ = np.histogram(_tf[_msk], bins=_edges,
                                                  weights=_m0[_msk])
                            _t0, _s0 = smooth_resample_sfh(
                                _tc, _h0 / (SFH_ARCH_BIN_MYR * 1e6),
                                dt_myr=SFH_ARCH_DT_MYR,
                                kernel_myr=SFH_ARCH_KERNEL_MYR)
                            _demo[_sid] = (_t0, _s0, _ts, _ss)
            _mfa = np.concatenate(_mfs) if _mfs else np.array([np.nan])
            print(f"snap {_snap:3d} (z={_z:.2f}, t_obs={_t_obs:.2f} Gyr): "
                  f"{_ns} galaxies archived | mfrac median {np.median(_mfa):.3f} "
                  f"[{np.min(_mfa):.3f}, {np.max(_mfa):.3f}] | "
                  f"M_formed/M_current = {(_mform / _mcur if _mcur else np.nan):.3f}")
    print(f"\n[SFH archive] {_nw} aperture SFHs for {_ngal_ok} galaxies "
          f"({_nskip} skipped) -> {SFH_APERTURE_H5}")
    print(f"[SFH archive] sfh_mass_kind="
          f"{'formed' if APPLY_MFRAC and _MF_TAG != 'UNCORRECTED' else 'current'}"
          f"  ({_MF_TAG})")
    print("   M_formed/M_current ~1.5-1.8 is expected for these old systems; "
          "~1.0 means the correction did NOT apply")

    # ── QC figure: what the formed-mass correction actually does to the shape ──
    if _demo:
        fig, axs = plt.subplots(1, len(_demo), figsize=(7 * len(_demo), 5.2),
                                squeeze=False)
        for _k, (_sid, (_t0, _s0, _ts, _ss)) in enumerate(_demo.items()):
            ax = axs[0][_k]
            ax.plot(_t0, _s0 / max(_s0.sum(), 1e-30), "-", lw=1.5, color="0.55",
                    label="current mass (what the RT did NOT use)")
            ax.plot(_ts, _ss / max(_ss.sum(), 1e-30), "-", lw=1.8, color="C3",
                    label="formed mass (injected)")
            ax.set(xlabel="cosmic time [Gyr]", ylabel="normalised SFR",
                   title=_sid)
            ax.grid(alpha=0.3)
            if _k == 0:
                ax.legend(fontsize=10, frameon=False)
        fig.tight_layout()
        fig.savefig(os.path.join(PLOTDIR, "sfh_mfrac_correction.png"), dpi=150,
                    bbox_inches="tight")
        plt.show()
    _arch = load_sfh_archive()

# ── coverage: this is a SAMPLE CUT under a per-object pin, not a footnote ──
_cov = {}
for _sid, _g in _arch.items():
    for _il, _aps in _g.items():
        for _lab in _aps:
            _cov[(_il, _lab)] = _cov.get((_il, _lab), 0) + 1
_ngal = len(_arch)
print(f"\naperture SFH coverage — galaxies with an archived SFH (of {_ngal}); "
      "an empty cell means NO Part 7d run for that (galaxy, aperture, sightline):")
print("   " + "".join(f"{l:>10s}" for l in APERTURE_LABELS))
for _il in INCL_LABELS:
    print(f"{_il:>7s}" + "".join(f"{_cov.get((_il, l), 0):>10d}"
                                 for l in APERTURE_LABELS))
print("   deficit vs the full sample:", {
    l: _ngal - _cov.get((INCL_LABELS[0], l), 0) for l in APERTURE_LABELS})

# ── figure: the radial SFH gradient the injection preserves ──
_pick = [s for s in sorted(_arch) if len(_arch[s].get(INCL_LABELS[0], {}))
         == len(APERTURE_LABELS)][:3]
if _pick:
    fig, axs = plt.subplots(1, len(_pick), figsize=(7 * len(_pick), 5.2),
                            squeeze=False)
    _cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(APERTURE_LABELS)))
    for _k, _sid in enumerate(_pick):
        ax = axs[0][_k]
        for _lab, _c in zip(APERTURE_LABELS, _cmap):
            _e = _arch[_sid][INCL_LABELS[0]].get(_lab)
            if _e is None:
                continue
            ax.plot(_e[0], _e[1], "-", lw=1.5, color=_c, label=_lab)
        ax.set(xlabel="cosmic time [Gyr]", ylabel=r"SFR [$M_\odot$/yr]",
               title=_sid)
        ax.grid(alpha=0.3)
        if _k == 0:
            ax.legend(fontsize=10, frameon=False, title="aperture",
                      title_fontsize=10)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTDIR, "sfh_aperture_archive.png"), dpi=150,
                bbox_inches="tight")
    plt.show()

# Part 7c2 — what the galaxy-level SFH pin actually costs

Part 7d injects **one** SFH per galaxy (`SFH_PIN_AP`, default `ap100kpc`) and fits all five
apertures, all four sightlines and both dust arms against it as rows of a single run. That is only
legitimate to the extent that the aperture SFHs differ from the pin in **amplitude**, which
`normalise=True` throws away, rather than in **shape**, which it does not. This cell measures the
difference instead of assuming it.

Two numbers per `(galaxy, sightline, aperture)`, computed on the *normalised* histories:

- $\Delta t_{\rm mw}$ — mass-weighted age of the aperture SFH minus that of the pin. **Positive
  means the aperture population is older than the pin**, so the fit is handed a too-blue template
  and can only pay for it with extra dust: $A_V$ biased **high**. Part 7c's own argument predicts
  this for the inner apertures.
- $\Delta \log f_{300}$ — log ratio of the mass fraction formed in the last 300 Myr, the part of
  the curve that sets the UV and the nebular lines.

plus the Kolmogorov–Smirnov distance between the two cumulative mass histories, as a single
shape-mismatch scalar that does not depend on choosing a summary statistic.

**The sightline check is a proof, not a diagnostic.** `R_CUTOUT_KPC = 100` and the ladder tops out
at 100 kpc, so the projected `ap100kpc` aperture is the entire Stage-0 sphere along every line of
sight. The sightline spread of $t_{\rm mw}$ at `ap100kpc` must therefore come out **identically
zero** — if it does not, the archive and the aperture geometry have drifted apart and Part 7d's
sightline collapse is unsound.

Output: `tables/cigale_sfh_pin_shape.fits`, one row per `(galaxy, sightline, aperture)`.
Read it before trusting the campaign — an inner-aperture $\Delta t_{\rm mw}$ of a few hundred Myr
is an age→$A_V$ leak of the same order as the attenuation being measured in the outer apertures,
and Part 7f must then quote it as a systematic (or `SFH_PIN_LEVEL = "aperture"` in Part 7d buys the
strict per-aperture pin back for ~5× the runs).


In [ ]:
# ── Part 7c2 — shape cost of the galaxy-level SFH pin (Part 7d's licence) ────
# Part 7d pins ONE SFH per galaxy and fits every aperture/sightline/arm against
# it. normalise=True discards the amplitude, so that is exact for any aperture
# whose history differs from the pin by a scale factor and wrong in proportion
# to the SHAPE difference. Measure the shape difference; do not assume it.
# Needs only the Part 7c archive (kernel-restart safe: reloads it if needed).
SFH_PIN_AP = globals().get("SFH_PIN_AP", APERTURE_LABELS[-1])
SFH_PIN_IL = globals().get("SFH_PIN_IL", None) or INCL_LABELS[0]
SFH_SHAPE_FITS = os.path.join(TABLEDIR, "cigale_sfh_pin_shape.fits")
F_RECENT_GYR = 0.3      # 'recent' window for the UV/nebular-driving mass

if "_arch" not in globals() or not _arch:
    _arch = load_sfh_archive()


def _sfh_shape_stats(t_gyr, sfr, t_obs):
    """(mass-weighted age [Myr], f_recent, normalised cumulative mass)."""
    w = np.clip(np.asarray(sfr, float), 0.0, None)
    tot = w.sum()
    if not np.isfinite(tot) or tot <= 0:
        return np.nan, np.nan, None
    age_mw = (t_obs - float((w * t_gyr).sum() / tot)) * 1e3
    f_rec = float(w[t_gyr > t_obs - F_RECENT_GYR].sum() / tot)
    return age_mw, f_rec, np.cumsum(w) / tot


_rows = []
for _sid, _g in sorted(_arch.items()):
    _pin = _g.get(SFH_PIN_IL, {}).get(SFH_PIN_AP)
    if _pin is None:
        print(f"[shape] {_sid}: no {SFH_PIN_AP}/{SFH_PIN_IL} pin — Part 7d "
              "would have no run for this galaxy at all")
        continue
    _pa, _pf, _pc = _sfh_shape_stats(_pin[0], _pin[1], _pin[2])
    if _pc is None:
        continue
    for _il in INCL_LABELS:
        for _lab in APERTURE_LABELS:
            _e = _g.get(_il, {}).get(_lab)
            if _e is None:
                continue
            _a, _f, _c = _sfh_shape_stats(_e[0], _e[1], _e[2])
            if _c is None or _c.shape != _pc.shape:
                continue       # different snapshot grid: cannot be the same gal
            _rows.append(dict(
                id=_sid, snap=int(_sid.split("_gal")[0][4:]),
                gal_id=int(_sid.split("_gal")[1]), incl=_il, aperture=_lab,
                age_mw_myr=_a, f_recent=_f,
                d_age_mw_myr=_a - _pa,
                dlog_f_recent=(np.log10(_f / _pf)
                               if (_f > 0 and _pf > 0) else np.nan),
                ks=float(np.max(np.abs(_c - _pc)))))

if not _rows:
    raise RuntimeError("no aperture SFHs to compare — is the Part 7c archive "
                       "built? (OVERWRITE_SFH_ARCHIVE=True)")
SHAPE = Table(rows=_rows)
SHAPE.write(SFH_SHAPE_FITS, overwrite=True)
print(f"[shape] {len(SHAPE)} (galaxy, sightline, aperture) rows -> "
      f"{SFH_SHAPE_FITS}")

_ap_col = np.char.strip(np.asarray(SHAPE["aperture"], str))
_il_col = np.char.strip(np.asarray(SHAPE["incl"], str))

# ── 1. the amplitude-vs-shape question, per aperture ────────────────────────
print(f"\nnormalised-SFH shape vs the {SFH_PIN_AP} pin "
      "(positive d_age = aperture OLDER than the pin -> the fit pays for a "
      "too-blue template with dust -> Av biased HIGH):")
print(f"{'aperture':>10s} {'N':>5s} {'d_age_mw [Myr]':>22s} "
      f"{'dlog f_300Myr':>18s} {'KS':>12s}")
for _lab in APERTURE_LABELS:
    _m = _ap_col == _lab
    if not _m.any():
        continue
    _d = np.asarray(SHAPE["d_age_mw_myr"], float)[_m]
    _l = np.asarray(SHAPE["dlog_f_recent"], float)[_m]
    _k = np.asarray(SHAPE["ks"], float)[_m]
    print(f"{_lab:>10s} {_m.sum():5d} "
          f"{np.nanmedian(_d):+8.1f} [{np.nanpercentile(_d, 16):+7.1f},"
          f"{np.nanpercentile(_d, 84):+7.1f}] "
          f"{np.nanmedian(_l):+8.2f} [{np.nanpercentile(_l, 84):+6.2f}] "
          f"{np.nanmedian(_k):8.3f} p95 {np.nanpercentile(_k, 95):.3f}")

# ── 2. the sightline collapse: at the pin aperture this MUST be zero ────────
_spread = {}
for _lab in APERTURE_LABELS:
    _v = []
    for _sid in set(np.asarray(SHAPE["id"], str)):
        _m = (np.asarray(SHAPE["id"], str) == _sid) & (_ap_col == _lab)
        if _m.sum() > 1:
            _v.append(np.nanmax(np.asarray(SHAPE["age_mw_myr"], float)[_m])
                      - np.nanmin(np.asarray(SHAPE["age_mw_myr"], float)[_m]))
    _spread[_lab] = np.nanmedian(_v) if _v else np.nan
print("\nsightline-to-sightline spread of the mass-weighted age [Myr] "
      "(median peak-to-peak over the 4 sightlines):")
for _lab in APERTURE_LABELS:
    print(f"   {_lab:>10s}  {_spread[_lab]:8.2f}"
          + ("   <- the pin aperture: MUST be 0, it is the whole cutout "
             "along every sightline" if _lab == SFH_PIN_AP else ""))
_pin_spread = _spread.get(SFH_PIN_AP, np.nan)
if np.isfinite(_pin_spread) and _pin_spread > 1e-6:
    raise RuntimeError(
        f"the {SFH_PIN_AP} SFH differs between sightlines by up to "
        f"{_pin_spread:.3g} Myr in mass-weighted age. It should be the ENTIRE "
        f"{R_CUTOUT_KPC:g} kpc cutout along every line of sight, so Part 7d's "
        "sightline collapse rests on an aperture geometry that no longer holds "
        "— check projected_radius / the cutout radius before fitting.")
print("   -> the sightline axis is exactly degenerate at the pin aperture, so "
      "Part 7d collapsing it costs nothing")

# ── 3. figure ───────────────────────────────────────────────────────────────
fig, axs = plt.subplots(1, 2, figsize=(16, 6.2))
_x = np.arange(len(APERTURE_LABELS))
_dat = [np.asarray(SHAPE["d_age_mw_myr"], float)[_ap_col == l]
        for l in APERTURE_LABELS]
_dat = [d[np.isfinite(d)] for d in _dat]
axs[0].axhline(0, color="0.4", lw=1)
_bp = axs[0].boxplot([d if d.size else [np.nan] for d in _dat], positions=_x,
                     widths=0.6, showfliers=False, patch_artist=True)
for _p in _bp["boxes"]:
    _p.set(facecolor="C0", alpha=0.35)
for _i, _d in enumerate(_dat):
    if _d.size:
        axs[0].plot(np.full(_d.size, _i) + np.random.uniform(-.16, .16, _d.size),
                    _d, ".", ms=2.5, color="C0", alpha=0.4)
axs[0].set(xticks=_x, xticklabels=APERTURE_LABELS,
           ylabel=r"$\Delta t_{\rm mw}$ [Myr]  (aperture $-$ pin)")
axs[0].grid(alpha=0.3, axis="y")

for _lab, _c in zip(APERTURE_LABELS,
                    plt.cm.viridis(np.linspace(0.1, 0.9, len(APERTURE_LABELS)))):
    _k = np.asarray(SHAPE["ks"], float)[_ap_col == _lab]
    _k = np.sort(_k[np.isfinite(_k)])
    if _k.size:
        axs[1].plot(_k, np.arange(1, _k.size + 1) / _k.size, "-", lw=1.6,
                    color=_c, label=_lab)
axs[1].set(xlabel="KS distance vs the pin",
           ylabel="cumulative fraction", ylim=(0, 1))
axs[1].legend(fontsize=10, frameon=False, title="aperture", title_fontsize=10)
axs[1].grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "sfh_pin_shape_cost.png"), dpi=150,
            bbox_inches="tight")
plt.show()


# Part 7d — galaxy-pinned CIGALE runs (cluster)

CIGALE never fits inside the notebook: this cell **prepares** the run directories
(`simbanator.sed.cigale.prepare_run`, which writes a complete, validated `pcigale.ini` + `.spec` —
no `pcigale init`/`genconf` by hand) and writes **one SLURM job array**; you then `sbatch`
it from a login node.

## 2026-08-10 — one galaxy, one SFH, one run per chain

The previous generation fitted every galaxy against a *family* of injected SFHs, which was still a
search over stellar populations, so `bayes.attenuation.Av_ISM` was one loose parameter among many
rather than a measurement. The fix was to pin the SFH — but it was implemented as **one run per
SED**, keyed on the full 5-tuple `(arm, aperture, sightline, snapshot, galaxy)`: ~4141 runs, each
rebuilding a 28 512- (or 114 048-) model grid from scratch to fit a *single* row. ≈ 291 M model SEDs.

The pin survives; the run count does not. What sets the cost is a fact about pcigale, not about
the physics:

> The model grid is built **once per run** and every source in the catalog is fitted against that
> same shared grid. `PdfAnalysis._compute_models` fills one `ModelsManager` of
> `multiprocessing.RawArray`s, and `_compute_bayes` then maps the workers over the *observations*
> (`pdf_analysis/__init__.py:129`, `managers/models.py:50`). **A source is nearly free; a run is
> not.**

So the only question is which SEDs legitimately share a pinned grid — and three of the four axes
turn out to share one already:

| axis | shared? | why |
|---|---|---|
| **RT arm** | `dust_on` + `dust_off` **yes**, `agn_on` no | The three arms are renderings of the *same stars*, so they never had different SFHs — only different module chains. `dust_on`/`dust_off` use the identical chain and become rows of one run; `agn_on` needs `skirtor2016` and keeps its own. |
| **sightline** | **yes** | `R_CUTOUT_KPC == 100` and the ladder tops out at 100 kpc, so the *projected* `ap100kpc` aperture is the whole Stage-0 sphere. The galaxy-level pin is sightline-independent **by construction**, not by approximation. |
| **aperture** | **yes**, by the argument below | `sfhfromfile` runs with `normalise=True`: it divides the SFR by its integral, so $M_\star$ is the fitted normalization and only the SFH **shape** is the pin. Apertures whose histories differ by a scale factor are already the identical pin. |
| **metallicity node** | **no** — rows sub-group | $Z_\star$ reddens the optical exactly like dust, so a mis-pinned node lands straight in the measurand. An aperture whose Part 4c $Z_\star$ snaps to a different `bc03.metallicity` node gets its own run. |

**Result:** ~4141 runs × 1 source → ~200–450 runs × 20–40 sources, ≈ **20× fewer model SEDs for
~1.5× more fits**. The saving pays for fitting the **full `dust_off` set** (5 apertures × 4
sightlines) instead of the old 1-aperture control, so the FSPS-vs-BC03 zero point below is now
measured *per aperture* — where it belongs, since the offset depends on the stellar population.

### What is pinned, and what is free

| axis | how it is constrained | why |
|---|---|---|
| **SFH** | this galaxy's own formed-mass history from Part 7c at `SFH_PIN_AP = ap100kpc`, injected as the single column of `sfh.fits`, shape-only (`normalise=True`) | removes the parametric-form mismatch *and* the family search. $M_*$ stays the **fitted** normalization, so the Part 7f recovery check is still honest |
| **metallicity** | `bc03.metallicity` pinned to the single nearest node to each row's Part 4c $Z_\star$ (rows sub-group by node); `nebular.zgas` from the group's median $Z_{\rm gas}$ | age–metallicity–dust all redden the optical; two of the three are now known |
| **redshift / age** | fixed at the snapshot $z$; `sfh.age` capped at **pcigale's own Planck18 ceiling** at $z$ rounded to `redshift_decimals = 2`, with the `sfh.fits` time axis shifted so the *recent* end survives | CIGALE 2025.1 silently masks every model older than that ceiling ("No suitable model found") — it cost 20/72 objects in the almac11 runs before it was found |
| **dust emission** | **free**: `umin` 11 nodes, `gamma` 4, `qpah` 2 | with the SFH fixed, any FIR mismatch a pinned grid cannot absorb leaks straight into $A_V$ through energy balance. Dust emission never touches the UV/optical, so this reopens no age–dust degeneracy |
| **AGN** | `skirtor2016` **only in the agn chain** (`fracAGN` free incl. 0.0, `i = 30°`, type 1) | `dust_on`/`dust_off` were rendered with no AGN source at all, so a free `fracAGN` there could only absorb MIR flux and torque the energy balance that sets $A_V$. `i` is face-on because the m25 `agn_on` RT injects the BH as a point source obscured only by the galaxy's own dust grid — a torus-obscured geometry would double-count |
| **attenuation** | **the measurand, free**: `Av_ISM` on 18 nodes over 0–3 mag, `slope_ISM` ∈ {−0.7, −0.48, −0.3}, `mu` ∈ {0.2, 0.44, 0.7}, `slope_BC` ∈ {−1.3, −0.7} | `Av_ISM` is normalized at V, so it keeps its V-band meaning even with a freed curve shape. Low `mu` lets the fit power the FIR from birth clouds without reddening the diffuse optical |

### What the aperture collapse gives up — and how it is measured

Part 7c argues, correctly, that inner apertures hold an older and more quenched population. The
normalisation removes the **amplitude** difference between apertures; it does **not** remove a
difference in **shape**, and a shape difference is a genuine change of pin.

That is why the Part 7c QC block exists. It reports, per `(galaxy, aperture, sightline)`, the
mass-weighted age and recent-SFR fraction of the *normalised* aperture SFH minus the same for the
`ap100kpc` pin, to `tables/cigale_sfh_pin_shape.fits`. **Read it before trusting the campaign:**
an inner-aperture age offset of a few hundred Myr is an age→$A_V$ leak of the same order as the
attenuation being measured in the outer apertures, and Part 7f must then quote it as a systematic.

`SFH_PIN_LEVEL = "aperture"` restores the strict per-aperture pin (~1010 runs before the Z split —
still 4× cheaper than the per-SED design, because the sightline and arm collapses are exact and
survive either way). `Z_PIN_LEVEL = "galaxy"` collapses the metallicity split as well, at the cost
of pinning inner apertures at the global $Z_\star$.

### Identity: run name vs row id

What is **constant over a run** is in the directory name —
`pin_{dust|agn}_[apXkpc_]snapNNN_galID_z{node}` (`parse_pin_run`). What **varies row to row** is in
the source `id`, re-keyed by `cg.stack_cigale_inputs` to
`snapNNN_galID__{arm}_{aperture}_{sightline}` (`parse_pin_id`). Part 7f reads the identity from the
id column and takes only the chain/pin/node from the run name — `cg.collect_results` broadcasts
`id_map` values across however many rows a `results.fits` holds.

⚠️ **Downstream:** `sfh.index` is gone from `variables` (with one column it is a zero-width delta),
as are `sfh.tau_main`/`age_main`/`age_bq`/`r_sfr` (there is no parametric SFH).

### Memory and scheduling

Both problems that shaped the previous designs are gone. A run is
$18\times3\times3\times2$ attenuation $\times\,2\times11\times4$ dust $=$ **28 512 models**
(dust chain), or ×4 `fracAGN` $=$ **114 048** (agn chain), at a *single* redshift — worst-case
shared model arrays ≈ 114 048 × ~118 × 8 B ≈ **107 MB**, so `max_block_models` stays at its 5 M
default and `blocks = 1`. Sources add no model memory at all, only χ² work. That removes the
node-RAM exhaustion that SIGKILLed array `12828014` on `INTEL_PHI` (this cluster schedules with
`SelectTypeParameters=CR_Core`, so `--mem-per-cpu` reserves nothing and only sets a cgroup
ceiling). And at a few hundred dirs there is no `MaxArraySize` problem left either, so
`RUNS_PER_TASK` is back to 1.

**`SKIP_IF_DONE = True`** stays the default: a resubmit without it makes pcigale rename each `out/`
to a timestamped backup which it *keeps*.

**One-time env setup.** CIGALE 2025 lives in its **own** conda env — do NOT install it into `pd39`
(powderday pins numpy/astropy):

```bash
conda create -n cigale python=3.12 -y
conda activate cigale
pip install <path to the cigale-v2025 tarball from cigale.lam.fr>
```

The kernel stays `pd39`: only the env's `pcigale` executable is needed, and `cigale.find_pcigale()`
locates it by absolute path (no `conda activate` at runtime, so the same path works inside the SLURM
tasks). `cg.describe_run(run_dir, docs=False)` echoes one prepared run's grids.

**Other knobs.** `BROADBANDS_ONLY` drops `F###N`/`F###M`, HSC/VIRCAM narrowbands and the duplicate
`spire *_ext` curves (93 → ~40 bands); `FIT_BANDS` then pins which of those are actually **fitted** —
everything else is still **predicted** (`bayes.<band>`). `MIN_FIT_BANDS` is now a **per-row** cut, so
a photometrically thin aperture drops that row instead of killing a run. `PILOT` restricts the build
to one snapshot × one galaxy (2 run dirs) for a cheap pre-flight; verify with `cg.check()` and one
interactive `cg.run()` before launching the full array.

Workflow: **Part 7c** (+ its QC) → this cell → `sbatch output/cis25/cigale_runs_pinned/submit_cigale_pin.job`
→ **Part 7e** → **Part 7f**.


In [ ]:
# ── Part 7d: galaxy-pinned CIGALE runs — ONE run per (snapshot, galaxy, chain) ─
# CIGALE never runs in this kernel: the cell only writes run dirs + the job file.
# 2026-08-10 (second pass) — the pin stays, the RUN COUNT collapses. pcigale
# builds its model grid ONCE PER RUN and fits every source in the catalog
# against that same shared grid (PdfAnalysis._compute_models fills one
# ModelsManager of RawArrays; _compute_bayes then maps the workers over the
# OBSERVATIONS). So a source is nearly free and a run is not, and the only
# question that sets the campaign cost is: which SEDs legitimately share a
# pinned grid? Answer here:
#   (1) THE ARMS SHARE STARS. dust_on / dust_off / agn_on are three renderings
#       of the SAME galaxy, so they never had different SFHs — only different
#       module chains. dust_on and dust_off use the identical chain, so they are
#       rows of ONE run; agn_on needs skirtor2016 and keeps its own.
#   (2) normalise=True DISCARDS THE AMPLITUDE. sfhfromfile divides by the
#       integral, so M* is the fitted normalization and two apertures whose
#       histories differ by a scale factor are ALREADY the identical pin. Only
#       a difference in SHAPE is a different pin — see the Part 7c QC, which
#       measures exactly that and is what licenses this collapse.
#   (3) SIGHTLINES ARE FREE AT THE PIN APERTURE. R_CUTOUT_KPC == 100 and the
#       ladder tops out at 100 kpc, so the PROJECTED ap100kpc aperture is the
#       whole Stage-0 sphere for every sightline — the pin is sightline-
#       independent by construction, not by approximation.
#   (4) METALLICITY IS STILL PINNED PER ROW. Rows are sub-grouped by their own
#       Part 4c bc03.metallicity node, so an aperture whose Z_star snaps to a
#       different node gets its own run rather than a wrong pin. The census
#       below reports what that costs (usually 1-2 groups per galaxy).
# Net: ~4141 runs x 1 source -> ~200-450 runs x 20-40 sources, ~20x fewer model
# SEDs for ~1.5x MORE fits (the full dust_off set is now affordable, so the
# FSPS-vs-BC03 zero point is measured per aperture instead of once globally).
# Everything else — the free attenuation grid that IS the measurand, the freed
# dl2014 nuisance, the Planck18 age cap, the formed-mass SFH — is unchanged
# from the per-object design and documented in the markdown above.
from astropy.cosmology import Planck18
from simbanator.sed import cigale as cg

PCIGALE_CMD     = cg.find_pcigale()   # dedicated conda env; see the markdown
CORES_PER_TASK  = 8
PLOT_SEDS       = False   # pcigale-plots on ~6000 SEDs ~doubles the campaign
SKIP_IF_DONE    = True    # resubmits then only run what is missing
USE_Z_PRIORS    = True
BROADBANDS_ONLY = True
PILOT           = False   # True -> one snapshot x one galaxy (2 run dirs)

# ── what is pinned, and at what level ──
SFH_PIN_LEVEL = "galaxy"     # 'galaxy' (one SFH per galaxy) | 'aperture'
SFH_PIN_AP    = "ap100kpc"   # galaxy-level pin = the whole cutout (see note 3)
SFH_PIN_IL    = None         # None -> INCL_LABELS[0]; irrelevant at ap100kpc
Z_PIN_LEVEL   = "row"        # 'row' = sub-group by the row's own Z node
                             # 'galaxy' = one Z node per galaxy (fewer runs)

# every arm is FITTED now: dust_off is no longer a 1-aperture control but the
# per-aperture zero point, and it rides along in the dust chain for free.
FIT_ARMS  = ("dust_on", "dust_off", "agn_on")
ARM_APS   = {a: tuple(APERTURE_LABELS) for a in FIT_ARMS}
ARM_INCLS = {a: tuple(INCL_LABELS) for a in FIT_ARMS}
CHAIN_OF  = {"dust_on": "dust", "dust_off": "dust", "agn_on": "agn"}

# --- scheduling (200-450 dirs: no MaxArraySize problem left) ---
ARRAY_THROTTLE   = 48          # simultaneous tasks
RUNS_PER_TASK    = 1
MAX_ARRAY_TASKS  = 1000        # scontrol show config | grep -i MaxArraySize
PARTITIONS       = "INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL"
WALLTIME         = "0-12:00"   # PHI cores are ~3-4x slower than Cascade

RUN_BASE_PIN = globals().get("RUN_BASE_PIN",
                             os.path.join(OUT, "cigale_runs_pinned"))
os.makedirs(RUN_BASE_PIN, exist_ok=True)

_DROP_BAND = re.compile(r"\.F\d+[NM]$|\.NB\d+$|_ext$|^generic\.|^2mass\.")
# cumulative apertures only (annuli break CIGALE's energy balance)
_TAG_RE = re.compile(r"^(dust_on|dust_off|agn_on)_(ap[0-9]+kpc)_(i\d+p\d+)$")
# run-dir name -> what is CONSTANT over the run (chain + metallicity node).
# Everything that varies row to row lives in the row id — see parse_pin_id.
PIN_RE = re.compile(r"^pin_(dust|agn)_(?:(ap\d+kpc)_)?snap(\d+)_gal(\d+)_z(\d+)$")


def parse_pin_run(name):
    """Run-dir basename -> {chain, pin_ap, snap, gal_id, zs_idx} or None."""
    m = PIN_RE.match(name)
    if m is None:
        return None
    return dict(chain=m.group(1), pin_ap=m.group(2) or "",
                snap=int(m.group(3)), gal_id=int(m.group(4)),
                zs_idx=int(m.group(5)))


def parse_pin_id(row_id):
    """Row id 'snapNNN_galID__arm_ap_incl' -> the identity of ONE fitted SED.

    With many sources per run the arm/aperture/sightline no longer live in the
    directory name — cg.stack_cigale_inputs put them in the id. Returns
    {id, snap, gal_id, arm, aperture, incl} or None.
    """
    sid, tag = cg.parse_stacked_id(str(row_id))
    m = _TAG_RE.match(tag)
    if m is None or "_gal" not in sid:
        return None
    return dict(id=sid, snap=int(sid.split("_gal")[0][4:]),
                gal_id=int(sid.split("_gal")[1]), arm=m.group(1),
                aperture=m.group(2), incl=m.group(3))


# manual fitted-band list (unselected bands are still PREDICTED as bayes.<band>)
FIT_BANDS = [
    "subaru.hsc.g", "subaru.hsc.r", "subaru.hsc.i", "subaru.hsc.z",
    "subaru.hsc.Y",
    "paranal.vircam.Y", "paranal.vircam.J", "paranal.vircam.H",
    "paranal.vircam.Ks",
    "hst.wfc3.uvis1.F606W", "hst.wfc3.uvis1.F814W",
    "jwst.nircam.F070W", "jwst.nircam.F090W", "jwst.nircam.F115W",
    "jwst.nircam.F150W", "jwst.nircam.F150W2", "jwst.nircam.F200W",
    "jwst.nircam.F277W", "jwst.nircam.F322W2", "jwst.nircam.F356W",
    "jwst.nircam.F444W",
    "jwst.miri.F560W", "jwst.miri.F770W", "jwst.miri.F1000W",
    "jwst.miri.F1130W", "jwst.miri.F1280W", "jwst.miri.F1500W",
    "jwst.miri.F1800W", "jwst.miri.F2100W", "jwst.miri.F2550W",
    "spitzer.mips.24mu", "spitzer.mips.70mu", "spitzer.mips.160mu",
    "herschel.pacs.blue", "herschel.pacs.green", "herschel.pacs.red",
    "herschel.spire.PSW", "herschel.spire.PMW",
    "jcmt.scuba2.450GHz", "jcmt.scuba2.850GHz",
    "alma.band6",
]
MIN_FIT_BANDS = 8      # fewer finite fluxes than this -> the ROW is dropped

SED_MODULES_DUST = ("sfhfromfile", "bc03", "nebular", "dustatt_modified_CF00",
                    "dl2014", "restframe_parameters", "redshifting")
SED_MODULES_AGN  = ("sfhfromfile", "bc03", "nebular", "dustatt_modified_CF00",
                    "dl2014", "skirtor2016", "restframe_parameters",
                    "redshifting")

# ── the measurand: attenuation, free ──
AV_ISM_GRID    = [0.0, 0.02, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.55,
                  0.7, 0.9, 1.1, 1.4, 1.7, 2.1, 2.6, 3.0]
SLOPE_ISM_GRID = [-0.7, -0.48, -0.3]      # CF00 default + greyer ISM curves
MU_GRID        = [0.2, 0.44, 0.7]         # low mu = FIR powered by birth clouds
SLOPE_BC_GRID  = [-1.3, -0.7]             # CF00 default + a greyer BC curve
# ── nuisance: dust emission, freed so the IR bands stop torquing Av ──
QPAH_GRID  = [2.50, 5.95]                 # powderday PAH_frac['usg'] + 1 lower
UMIN_GRID  = sorted({float(v) for v in cg.nearest_option(
    [0.1, 0.2, 0.35, 0.6, 1.0, 1.7, 3.0, 5.0, 8.0, 12.0, 25.0],
    cg.grid_options("dl2014", "umin"), log=True)})
GAMMA_GRID = [0.01, 0.02, 0.05, 0.1]
# ── nuisance: AGN (agn chain only). fracAGN keeps 0.0 reachable ──
FRAC_AGN_GRID = [0.0, 0.1, 0.2, 0.4]
SKIRTOR_I     = 30                        # type-1 view (i < 90 - oa = 50 deg)

# per-object metallicity pins
_ZS_GRID   = cg.grid_options("bc03", "metallicity")
_ZG_GRID   = cg.grid_options("nebular", "zgas")
Z_FALLBACK = 0.02      # solar-ish node when Part 4c has no value

# rest-frame line EWs (CIGALE 2025.1 label/blue/line/red format)
EW_SPEC = ("OIII5007/497.7/499.7/499.7/501.7/501.7/503.7 & "
           "Halpha/653.3/655.3/655.3/657.3/657.3/659.3 & "
           "HdeltaA/404.160/407.975/408.350/412.225/412.850/416.100")
# NOTE: no sfh.index (single column -> zero-width delta) and no sfh.tau_main /
# age_main / age_bq / r_sfr (there is no parametric SFH any more).
VARIABLES_BASE = ["stellar.m_star", "stellar.metallicity", "stellar.age_m_star",
                  "sfh.sfr", "sfh.sfr10Myrs", "sfh.sfr100Myrs",
                  "attenuation.Av_ISM", "attenuation.Av_BC",
                  "attenuation.generic.bessell.V",
                  "attenuation.generic.bessell.B",
                  "dust.luminosity", "dust.mass", "dust.umean",
                  "param.Dn4000", "param.EW(OIII5007)", "param.EW(Halpha)",
                  "param.EW(HdeltaA)",
                  "param.restframe_Lnu(galex.FUV)",
                  "param.restframe_Lnu(generic.bessell.V)",
                  "param.restframe_generic.johnson.U-generic.johnson.V",
                  "param.restframe_generic.johnson.V-generic.johnson.J",
                  "param.restframe_galex.FUV-galex.NUV",
                  "param.restframe_galex.NUV-sloan.sdss.r"]
VARIABLES = {"dust": VARIABLES_BASE,
             "agn":  VARIABLES_BASE + ["agn.fracAGN"]}

_n_att  = (len(AV_ISM_GRID) * len(SLOPE_ISM_GRID) * len(MU_GRID)
           * len(SLOPE_BC_GRID))
_n_dust = len(QPAH_GRID) * len(UMIN_GRID) * len(GAMMA_GRID)
print("pcigale:", PCIGALE_CMD)
print(f"[grid] attenuation {_n_att} x dl2014 {_n_dust} = {_n_att * _n_dust:,} "
      f"models/run (dust chain)")
print(f"[grid]   x fracAGN {len(FRAC_AGN_GRID)} = "
      f"{_n_att * _n_dust * len(FRAC_AGN_GRID):,} for the agn chain "
      f"(skirtor2016 @ i={SKIRTOR_I} deg)")
print("[grid] the grid is built ONCE per run and shared by every source in it")

# ── the injected SFHs (Part 7c) — kernel-restart safe, and they MUST be the
#    formed-mass ones or the return-fraction tilt lands straight in Av ──
SFH_APERTURE_H5 = globals().get(
    "SFH_APERTURE_H5", os.path.join(CIGALE_DIR, "sfh_smoothed_aperture.h5"))
if not os.path.exists(SFH_APERTURE_H5):
    raise RuntimeError(f"{SFH_APERTURE_H5} missing — run Part 7c first")
if "load_sfh_archive" not in globals():
    def load_sfh_archive(path=None):
        """{id: {incl: {aperture: (t_gyr, sfr, t_obs_gyr)}}} — see Part 7c."""
        out = {}
        with h5py.File(path or SFH_APERTURE_H5, "r") as f:
            for sid in f:
                for il in f[sid]:
                    for ap in f[sid][il]:
                        d = f[sid][il][ap]
                        arr = np.asarray(d[:], float)
                        out.setdefault(str(sid), {}).setdefault(
                            str(il), {})[str(ap)] = (arr[:, 0], arr[:, 1],
                                                     float(d.attrs["t_obs_gyr"]))
        return out
with h5py.File(SFH_APERTURE_H5, "r") as _f:
    _kind = _f.attrs.get("sfh_mass_kind", b"current")
    _kind = _kind.decode() if isinstance(_kind, bytes) else str(_kind)
    _msrc = _f.attrs.get("mfrac_source", b"?")
    _msrc = _msrc.decode() if isinstance(_msrc, bytes) else str(_msrc)
if _kind != "formed":
    raise RuntimeError(
        f"{SFH_APERTURE_H5} holds '{_kind}'-mass SFHs (mfrac_source={_msrc}). "
        "powderday rendered the FORMED mass (mass/mfrac), so injecting "
        "surviving-mass rates tilts the oldest bins ~80% low — and with the "
        "SFH pinned that tilt goes straight into Av. Re-run Part 7c with "
        "OVERWRITE_SFH_ARCHIVE=True.")
SFH_ARCH = load_sfh_archive()
_PIN_IL = SFH_PIN_IL or INCL_LABELS[0]
print(f"[sfh] archive: {len(SFH_ARCH)} galaxies, mass kind '{_kind}' ({_msrc})")
print(f"[sfh] pin level '{SFH_PIN_LEVEL}'"
      + (f" @ {SFH_PIN_AP} (== the whole {R_CUTOUT_KPC:g} kpc cutout, so it is "
         "the same curve for every sightline)" if SFH_PIN_LEVEL == "galaxy"
         else f", sightline {_PIN_IL}"))


def _sfh_file_one(run_dir, sid, incl, aperture, age_myr, shift_myr=0):
    """Write run_dir/sfh.fits with a SINGLE SFR column for this run.

    col 0 = time [Myr], 0..age_myr in STRICT 1 Myr steps (CIGALE raises
    otherwise); col 1 = the pinned smoothed formed-mass SFH, interpolated from
    the 25 Myr archive grid. SFR = 0 before the first archived sample, edge-held
    after the last (the recent edge drives the UV/nebular fluxes). shift_myr > 0
    maps table time t to cosmic time t + shift_myr, i.e. drops the OLDEST
    shift_myr so that t = age_myr still lands on the snapshot epoch —
    sfhfromfile truncates at the RECENT end (`sfr[time_grid <= age]`), so
    shrinking age alone would cut the newest star formation.

    Returns (path, hold_frac) or (None, reason). hold_frac is the fraction of
    the tabulated time that is past the archive's last sample, i.e. edge-held.
    """
    entry = SFH_ARCH.get(str(sid), {}).get(incl, {}).get(aperture)
    if entry is None:
        return None, f"no archived SFH at {aperture}/{incl}"
    t_gyr, sfr = entry[0], entry[1]
    t1 = np.arange(int(age_myr) + 1, dtype=np.int64)
    tc = t1 + shift_myr                                    # cosmic time [Myr]
    col = np.clip(np.interp(tc, t_gyr * 1e3, sfr, left=0.0), 0.0, None)
    if not np.isfinite(col).all():
        return None, "non-finite SFH"
    if col.sum() <= 0:
        # normalise=True would divide by the zero integral -> all-NaN results
        return None, "all-zero SFH within the age cap"
    os.makedirs(run_dir, exist_ok=True)
    tab = Table()
    tab["time"] = t1
    tab[str(sid)] = col.astype(float)
    path = os.path.join(run_dir, "sfh.fits")
    tab.write(path, overwrite=True)
    return path, float((tc > t_gyr[-1] * 1e3).mean())


def _snap_z(value, grid, fallback=None):
    """SIMBA metallicity -> the single nearest node of a strict CIGALE grid."""
    v = float(value) if np.isfinite(value) else np.nan
    if not np.isfinite(v):
        v = float(fallback) if fallback is not None else Z_FALLBACK
    out = float(cg.nearest_option([v], grid, log=True)[0])
    return out if np.isfinite(out) else float(
        cg.nearest_option([Z_FALLBACK], grid, log=True)[0])


# ── per-galaxy metallicity priors (Part 4c), keyed by (aperture, sightline) ──
_ztabf = os.path.join(TABLEDIR, "aperture_metallicities.fits")
_ztab = Table.read(_ztabf) if (USE_Z_PRIORS and os.path.exists(_ztabf)) else None
if USE_Z_PRIORS and _ztab is None:
    print(f"[Z priors] {_ztabf} missing — run Part 4c first; every object "
          f"falls back to Z_FALLBACK={Z_FALLBACK}")


def _z_maps(label, ilab):
    if _ztab is None:
        return {}, {}
    m = np.char.strip(np.asarray(_ztab["incl"], str)) == ilab
    ids = [f"snap{int(s):03d}_gal{int(g)}" for s, g in
           zip(np.asarray(_ztab["snap"], int)[m],
               np.asarray(_ztab["gal_id"], int)[m])]
    return (dict(zip(ids, np.asarray(_ztab[f"Zstar_{label}"], float)[m])),
            dict(zip(ids, np.asarray(_ztab[f"Zgas_{label}"], float)[m])))


_ZMAPS = {(_ap, _il): _z_maps(_ap, _il)
          for _ap in APERTURE_LABELS for _il in INCL_LABELS}

# ── 1. every Part 7b catalog: cleaned, tagged and stacked ONCE ───────────────
_items, _nocat = [], []
for _arm in FIT_ARMS:
    for _ap in ARM_APS[_arm]:
        for _il in ARM_INCLS[_arm]:
            _f = os.path.join(CIGALE_DIR, f"cigale_{_arm}_{_ap}_{_il}.fits")
            if not os.path.exists(_f):
                _nocat.append(os.path.basename(_f))
                continue
            cg.sanitize_input_errors(_f, verbose=False)   # legacy neg. errors
            _t = Table.read(_f)
            if BROADBANDS_ONLY:
                _t = _t[[c for c in _t.colnames
                         if not _DROP_BAND.search(c.removesuffix("_err"))]]
            # NaN errors -> 10% of the flux (CIGALE adds additionalerror too)
            for _b in [c for c in _t.colnames
                       if c not in ("id", "redshift", "distance")
                       and not c.endswith("_err")]:
                _fl = np.asarray(_t[_b], float)
                _er = np.asarray(_t[f"{_b}_err"], float)
                _bad = np.isfinite(_fl) & ~np.isfinite(_er)
                if _bad.any():
                    _t[f"{_b}_err"][_bad] = 0.1 * np.abs(_fl[_bad])
            _items.append((_t, (_arm, _ap, _il)))
if _nocat:
    print(f"[7b] {len(_nocat)} catalog(s) missing, e.g. {_nocat[:3]}")
if not _items:
    raise RuntimeError(f"no Part 7b catalogs under {CIGALE_DIR} — run 7b first")
ALLCAT = cg.stack_cigale_inputs(_items)
_bandcols = [c for c in ALLCAT.colnames
             if c not in ("id", "redshift", "distance")
             and not c.endswith("_err")]
_fitcols = [c for c in _bandcols if c in FIT_BANDS]
print(f"[7b] {len(_bandcols)} bands kept, {len(_fitcols)} fitted")

# ── 1b. dust_off rows: drop the IR bands from the FIT (2026-08-11) ──────────
# The dust_off SED has no dust emission by construction, but the shared grid
# keeps dl2014, and at IR wavelengths the model-vs-mock residual is the
# BC03-vs-FSPS STELLAR TAIL — factor-level mismatches against additionalerror's
# 10% floor. chi2_min then exceeds ~1.5e3, exp(-chi2/2) underflows to 0 for
# EVERY model and pcigale masks the whole row ("No suitable model found ...
# the chi2 are very large"): the 2026-08-10 campaign lost 16/20 dust_off rows
# per run this way, with all-NaN best_model files. The zero-point is an
# optical/NIR statement, so the IR bands are excluded PER ROW (NaN flux ==
# pcigale's per-observation band drop); dust_on/agn_on rows keep them, and the
# [analysis_params] bands still predict bayes.<band> everywhere.
_ir_fit = [c for c in _bandcols if cg.IR_BAND_RE.match(c)]
_isoff = np.char.find(np.asarray(ALLCAT["id"], str), "__dust_off_") >= 0
if _isoff.any() and _ir_fit:
    for _b in _ir_fit:
        ALLCAT[_b][_isoff] = np.nan
        ALLCAT[f"{_b}_err"][_isoff] = np.nan
    print(f"[dust_off] {len(_ir_fit)} IR fit bands -> NaN on "
          f"{int(_isoff.sum())} dust_off rows (stellar-tail chi2 underflow "
          "guard; see comment)")

# ── 2. one record per candidate SED row ─────────────────────────────────────
_recs, _skips = [], []
for _i, _rid in enumerate(np.asarray(ALLCAT["id"], str)):
    _r = parse_pin_id(_rid)
    if _r is None:
        raise RuntimeError(f"row id {_rid!r} does not parse — parse_pin_id and "
                           "cg.stack_cigale_inputs' tags have drifted apart")
    _r["row"] = _i
    _r["z"] = float(ALLCAT["redshift"][_i])
    _recs.append(_r)

if PILOT:
    _p_sn = min(r["snap"] for r in _recs)
    _p_g = min(r["gal_id"] for r in _recs if r["snap"] == _p_sn)
    _recs = [r for r in _recs if r["snap"] == _p_sn and r["gal_id"] == _p_g]
    print(f"[PILOT] restricted to snap {_p_sn}, galaxy {_p_g} "
          f"({len(_recs)} rows -> 2 run dirs)")

# fitted-band cut, per ROW (a thin row is dropped; the run still happens)
_keep = []
for _r in _recs:
    _n = int(sum(np.isfinite(float(ALLCAT[c][_r["row"]])) for c in _fitcols))
    if _n < MIN_FIT_BANDS:
        _skips.append({**{k: _r[k] for k in ("arm", "aperture", "incl",
                                             "snap", "gal_id")},
                       "reason": f"only {_n} finite fit bands"})
    else:
        _keep.append(_r)
_recs = _keep

# ── 3. the pcigale age cap, per snapshot (note 3 of the markdown) ───────────
_agecap = {}
for _sn in sorted({r["snap"] for r in _recs}):
    _z = float(np.median([r["z"] for r in _recs if r["snap"] == _sn]))
    _age_cos = int(round(COSMO.age(_z).to(u.Myr).value))
    _age_cig = int(np.floor(min(Planck18.age(_z).to(u.Myr).value,
                                Planck18.age(round(_z, 2)).to(u.Myr).value))) - 1
    _age_myr = min(_age_cos, _age_cig)
    _shift = _age_cos - _age_myr
    if _shift > 200:
        raise RuntimeError(
            f"snap {_sn}: the pcigale age cap drops {_shift} Myr of the OLDEST "
            "SFH — that is far beyond the usual few tens of Myr. Check COSMO "
            "vs Planck18 and redshift_decimals before fitting.")
    _agecap[_sn] = (_z, _age_cos, _age_cig, _shift)

# ── 4. group the rows into runs: same SFH + same Z node + same chain ────────
_groups = {}
for _r in _recs:
    _sid, _ap, _il = _r["id"], _r["aperture"], _r["incl"]
    _zs_map, _zg_map = _ZMAPS.get((_ap, _il), ({}, {}))
    if Z_PIN_LEVEL == "galaxy":
        _pm = _ZMAPS.get((SFH_PIN_AP, _PIN_IL), ({}, {}))
        _zs_raw, _zg_raw = _pm[0].get(_sid, np.nan), _pm[1].get(_sid, np.nan)
    else:
        _zs_raw, _zg_raw = _zs_map.get(_sid, np.nan), _zg_map.get(_sid, np.nan)
    _zs = _snap_z(_zs_raw, _ZS_GRID)
    _r["zs_node"], _r["zg_raw"] = _zs, _zg_raw
    _chain = CHAIN_OF[_r["arm"]]
    _pin_ap = _ap if SFH_PIN_LEVEL == "aperture" else ""
    _groups.setdefault((_r["snap"], _r["gal_id"], _chain, _pin_ap, _zs),
                       []).append(_r)

print(f"\n{len(_recs)} fittable SED rows -> {len(_groups)} runs")
_per_gal = {}
for (_sn, _gid, _chain, _pa, _zs) in _groups:
    _per_gal.setdefault((_sn, _gid, _chain, _pa), []).append(_zs)
_nz = np.array([len(v) for v in _per_gal.values()])
_base = len(_per_gal)      # what a single Z node per SFH pin would cost
print(f"[Z pin '{Z_PIN_LEVEL}'] {len(_groups)} runs vs {_base} at one node per "
      f"SFH pin: median {np.median(_nz):.0f}, max {_nz.max()} bc03 node(s) "
      "per pin.")
if len(_groups) > _base:
    print(f"   A galaxy whose apertures straddle a node splits rather than "
          f"take a wrong pin — Z_star reddens the optical exactly like dust, so "
          f"a mis-pinned node lands in the measurand. Set Z_PIN_LEVEL='galaxy' "
          f"to pin every aperture at {SFH_PIN_AP} instead and pay "
          f"{_base} runs (Part 7f can then no longer treat the inner-aperture "
          f"Z as known).")

# ── 5. write one run dir per group ──────────────────────────────────────────
_run_dirs, _holds, _fitted = [], [], []
for _key in sorted(_groups):
    _sn, _gid, _chain, _pin_ap, _zs = _key
    _rows = _groups[_key]
    _sid = f"snap{_sn:03d}_gal{_gid}"
    _z, _age_cos, _age_cig, _shift = _agecap[_sn]
    _age_myr = min(_age_cos, _age_cig)
    _zk = _ZS_GRID.index(_zs)
    _rd = os.path.join(RUN_BASE_PIN, f"pin_{_chain}_"
                       + (f"{_pin_ap}_" if _pin_ap else "")
                       + f"snap{_sn:03d}_gal{_gid}_z{_zk}")

    _arch_ap = _pin_ap or SFH_PIN_AP
    _sfh, _extra = _sfh_file_one(_rd, _sid, _PIN_IL, _arch_ap, _age_myr, _shift)
    if _sfh is None:
        for _r in _rows:
            _skips.append({**{k: _r[k] for k in ("arm", "aperture", "incl",
                                                 "snap", "gal_id")},
                           "reason": _extra})
        continue
    # keyed by RUN NAME: it is unique, cg.collect_results already carries it
    # into the results table as 'run', and it cannot round-trip through FITS as
    # a masked empty string the way a '' aperture token does.
    _holds.append(dict(run=os.path.basename(_rd), chain=_chain, snap=_sn,
                       gal_id=_gid, sfh_ap=_arch_ap, nrows=len(_rows),
                       sfr_hold_frac=_extra))

    # nebular zgas: the group's median, snapped to the nebular grid
    _zg_vals = np.array([_r["zg_raw"] for _r in _rows], float)
    _zg = _snap_z(np.nanmedian(_zg_vals) if np.isfinite(_zg_vals).any()
                  else np.nan, _ZG_GRID, fallback=_zs)

    _objf = os.path.join(_rd, "obj.fits")
    ALLCAT[[_r["row"] for _r in _rows]].write(_objf, overwrite=True)

    _mp = {
        "sfhfromfile": {"filename": _sfh, "sfr_column": [1],
                        "age": [_age_myr], "normalise": True},
        "bc03": {"imf": 1, "metallicity": [_zs]},
        "nebular": {"zgas": [_zg]},
        "dustatt_modified_CF00": {"Av_ISM": AV_ISM_GRID,
                                  "slope_ISM": SLOPE_ISM_GRID,
                                  "mu": MU_GRID,
                                  "slope_BC": SLOPE_BC_GRID},
        "dl2014": {"qpah": QPAH_GRID, "umin": UMIN_GRID, "gamma": GAMMA_GRID},
        "restframe_parameters": {"Dn4000": True, "EW": EW_SPEC},
    }
    _mods = SED_MODULES_DUST
    if _chain == "agn":
        _mods = SED_MODULES_AGN
        _mp["skirtor2016"] = {"fracAGN": FRAC_AGN_GRID, "i": SKIRTOR_I}
    cg.prepare_run(_rd, _objf, sed_modules=_mods, module_params=_mp,
                   analysis_params={"variables": VARIABLES[_chain],
                                    "save_best_sed": True},
                   cores=CORES_PER_TASK, fit_bands=FIT_BANDS, verbose=False)
    _run_dirs.append(_rd)
    _fitted.extend(_rows)
    if len(_run_dirs) % 100 == 0:
        print(f"   ... {len(_run_dirs)} run dirs written")

if not _run_dirs:
    raise RuntimeError("no run dirs prepared — check the Part 7b catalogs and "
                       "the Part 7c SFH archive")
_run_dirs = sorted(_run_dirs)

# ── the age cap, auditable ──
print("\npcigale age cap per snapshot (t_SIMBA, t_Planck18 ceiling, dropped):")
for _sn in sorted(_agecap):
    _z, _ac, _ag2, _sh = _agecap[_sn]
    print(f"   snap {_sn:3d}  z={_z:.3f}  t_cos={_ac:6d}  t_cig={_ag2:6d} Myr"
          f"  -> age={min(_ac, _ag2):6d}, oldest {_sh:3d} Myr dropped")

# ── coverage ──
_meta = [parse_pin_run(os.path.basename(d)) for d in _run_dirs]
if any(m is None for m in _meta):
    raise RuntimeError("some run-dir names do not match PIN_RE — Part 7f's "
                       "collect_results would silently drop them")
_nrow = np.array([h["nrows"] for h in _holds], float)
print(f"\n{len(_run_dirs)} run dirs prepared under {RUN_BASE_PIN}")
for _c in sorted({m["chain"] for m in _meta}):
    _n = sum(m["chain"] == _c for m in _meta)
    print(f"   {_c:5s} chain: {_n:4d} runs x "
          f"{_n_att * _n_dust * (len(FRAC_AGN_GRID) if _c == 'agn' else 1):,} models")
print(f"   {int(_nrow.sum()):5d} fitted SEDs, {_nrow.min():.0f}-{_nrow.max():.0f} "
      f"per run (median {np.median(_nrow):.0f})")
print("\nfitted SEDs per (aperture, arm):")
_cov = {}
for _r in _fitted:
    _cov[(_r["aperture"], _r["arm"])] = _cov.get((_r["aperture"], _r["arm"]), 0) + 1
print("   " + "".join(f"{l:>10s}" for l in APERTURE_LABELS))
for _arm in FIT_ARMS:
    print(f"{_arm:>9s}" + "".join(f"{_cov.get((l, _arm), 0):>10d}"
                                  for l in APERTURE_LABELS))

SKIP_FITS = os.path.join(TABLEDIR, "cigale_pinned_skipped.fits")
if _skips:
    _st = Table(rows=_skips, names=("arm", "aperture", "incl", "snap",
                                    "gal_id", "reason"))
    _st.write(SKIP_FITS, overwrite=True)
    print(f"\n[skipped] {len(_st)} (arm, aperture, sightline, galaxy) rows "
          f"-> {SKIP_FITS}")
    _u, _c = np.unique(np.asarray(_st["reason"], str), return_counts=True)
    for _r, _n in sorted(zip(_u, _c), key=lambda x: -x[1]):
        print(f"   {_n:5d}  {_r}")
    print("   ^ these SEDs are not fitted. Under the GALAXY-level pin the SFH "
          "comes from the whole cutout, so 'no archived SFH' can no longer cut "
          "the star-poor inner apertures — what is left is photometric.")
else:
    print("\n[skipped] none")

if _holds:
    _hf = np.asarray([h["sfr_hold_frac"] for h in _holds], float)
    Table(rows=_holds).write(os.path.join(TABLEDIR, "cigale_pinned_sfrhold.fits"),
                             overwrite=True)
    print(f"[sfr_hold_frac] median {np.median(_hf):.4f}, p95 "
          f"{np.percentile(_hf, 95):.4f} of the tabulated time is edge-held "
          "past the archive's last sample (the smoothed grid ends at the last "
          "bin CENTRE). Part 7f checks it does not correlate with dAv.")

JOB_FILE = cg.write_slurm_array(
    _run_dirs, os.path.join(RUN_BASE_PIN, "submit_cigale_pin.job"),
    pcigale_cmd=PCIGALE_CMD, partition=PARTITIONS, cores=CORES_PER_TASK,
    time=WALLTIME, array_throttle=ARRAY_THROTTLE, plots=PLOT_SEDS,
    skip_if_done=SKIP_IF_DONE, job_name="cigale_pin",
    runs_per_task=RUNS_PER_TASK, max_array_tasks=MAX_ARRAY_TASKS)
print(f"\nsbatch {JOB_FILE}")
print(f"  partitions {PARTITIONS}")
print(f"  SKIP_IF_DONE={SKIP_IF_DONE} -> a resubmit only runs what is missing "
      "(False makes pcigale keep a timestamped backup of every out/)")
print("  pre-flight before the full array: cg.check(_run_dirs[0], PCIGALE_CMD) "
      "should report the model count printed above, then time one "
      "cg.run(_run_dirs[0], PCIGALE_CMD) interactively — that time x "
      f"{len(_run_dirs)} is the campaign.")


# Part 7e — aperture-matched SIMBA truth (cluster, cached once)

CIGALE's estimates describe the stars inside **one projected aperture along one sightline** —
the global caesar/history M\*/SFR/age are only the right truth for the largest aperture. One
pass over the Stage-0 region cutouts measures, per (galaxy, sightline, aperture rung
**and annulus between rungs** — `ann3kpc…ann100kpc`, for the radial profiles; only the
cumulative apertures are ever fitted):

- **M\*** — current stellar mass in the projected cylinder (matching Hyperion's peeled
  apertures: radius r perpendicular to the (θ, φ) sightline, full depth);
- **archaeological SFR** over the last 25 / 100 Myr — mass formed in the window from
  `StellarFormationTime` (current masses, so ≲10–15 % mass-loss bias — noted, not corrected);
- **mass-weighted age and total Z** of the same stars (→ `stellar.age_m_star`,
  `stellar.metallicity`);
- a **delayed+bq fit to the aperture's own archaeological SFH** (50 Myr bins,
  `simbanator.analysis.sfh_utils.fit_delayed_bq` — the same form CIGALE fits), giving the
  per-aperture `sfh.tau_main / age_main / age_bq / r_sfr` truth.

Everything is cached to `tables/aperture_truth.fits` (one row per galaxy × sightline ×
aperture/annulus; `OVERWRITE_APERTURE_TRUTH=True` rebuilds, and a cache without annulus rows
is rebuilt automatically). Part 7f joins this cache on `(snap, gal_id, incl, aperture)`.
The `tau_main / age_main / age_bq / r_sfr` columns are a leftover of the parametric-SFH era —
harmless, already cached, and no longer compared against anything. Requires the
Stage-0 particle files and the Stage-1 selection HDF5 (centres = the RT-grid `x_cent`), so it
runs on the **cluster**.

In [ ]:
# ── Part 7e — cache SIMBA properties in the SAME apertures/annuli/sightlines as the RT ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts (cluster).
from simbanator.analysis.sfh_utils import fit_delayed_bq

APERTURE_TRUTH_FITS      = os.path.join(TABLEDIR, "aperture_truth.fits")
OVERWRITE_APERTURE_TRUTH = False
SFR_WINDOWS_MYR = (25.0, 100.0)     # -> sfh.sfr / sfh.sfr100Myrs truth
ARCH_BIN_MYR    = 50.0              # archaeological-SFH bin for the delayed+bq fit
NSTAR_AP_MIN    = 20                # ages/Z/fits need at least this many star particles

APERTURE_TRUTH = None
if os.path.exists(APERTURE_TRUTH_FITS) and not OVERWRITE_APERTURE_TRUTH:
    APERTURE_TRUTH = Table.read(APERTURE_TRUTH_FITS)
    if not any(str(a).startswith("ann") for a in APERTURE_TRUTH["aperture"]):
        print("cache has no annulus rows (pre-annuli version) -> rebuilding")
        APERTURE_TRUTH = None
    else:
        print(f"cached ({len(APERTURE_TRUTH)} rows) -> {APERTURE_TRUTH_FITS}  "
              "(OVERWRITE_APERTURE_TRUTH=True rebuilds)")
if APERTURE_TRUTH is None:
    SEL, SNAPS, IDS = load_selection()
    _cen = rt_centers(SNAPS, IDS)               # RT-grid centres (code units)

    _ag = np.linspace(0.02, 1.0, 4096)          # a -> t interpolation grid
    _tg = COSMO.age(1.0 / _ag - 1.0).value                      # Gyr

    _rows = []
    _apr = APERTURE_RADII_KPC[np.asarray(WANTED_AP_IDX)]        # true rung radii [pkpc]
    for _snap in np.unique(SNAPS):
        _zs = float(sim.get_z_from_snap(int(_snap)))
        _t_obs = float(COSMO.age(_zs).value)
        _n_gal = 0
        for _gid in np.unique(IDS[SNAPS == _snap]):
            _cut = read_cutout(_snap, _gid, _cen.get((int(_snap), int(_gid))),
                               "PartType4",
                               fields=("Masses", "StellarFormationTime", "Metallicity"))
            if _cut is None:
                print(f"  [skip] snap {_snap} gal {_gid}: no cutout/centre or no stars")
                continue
            _pos = _cut["pos"]
            _mst = np.asarray(_cut["Masses"], float) * 1e10 / _cut["h"]
            _af  = np.asarray(_cut["StellarFormationTime"], float)
            _Zst = np.asarray(_cut["Metallicity"], float)
            _Zst = _Zst[:, 0] if _Zst.ndim == 2 else _Zst
            _tf = np.interp(np.clip(_af, _ag[0], 1.0), _ag, _tg)    # formation time [Gyr]
            _n_gal += 1
            for _il, _nv in zip(INCL_LABELS, NHAT):
                _rproj = projected_radius(_pos, _nv)

                def _measure(_msk, _lab, _r):
                    """Truth of the stars in one projected selection (aperture OR annulus)."""
                    _nap = int(_msk.sum())
                    _row = dict(snap=int(_snap), gal_id=int(_gid), incl=_il,
                                aperture=_lab, ap_kpc=float(_r), nstar_ap=_nap,
                                mstar=np.nan, sfr25=np.nan, sfr100=np.nan,
                                age_m_star_myr=np.nan, met_star=np.nan,
                                tau_main_myr=np.nan, age_main_myr=np.nan,
                                age_bq_myr=np.nan, r_sfr=np.nan, fit_r2=np.nan)
                    if _nap:
                        _mm, _tt, _zz = _mst[_msk], _tf[_msk], _Zst[_msk]
                        _row["mstar"] = float(_mm.sum())
                        for _w, _key in zip(SFR_WINDOWS_MYR, ("sfr25", "sfr100")):
                            _row[_key] = float(_mm[_tt >= _t_obs - _w / 1e3].sum()
                                               / (_w * 1e6))
                        if _nap >= NSTAR_AP_MIN:
                            _row["age_m_star_myr"] = float(
                                np.sum(_mm * (_t_obs - _tt)) / _mm.sum() * 1e3)
                            _row["met_star"] = float(np.sum(_mm * _zz) / _mm.sum())
                            _bins = np.arange(_tt.min(), _t_obs + ARCH_BIN_MYR / 1e3,
                                              ARCH_BIN_MYR / 1e3)
                            if _bins.size >= 8:
                                _hm, _ = np.histogram(_tt, bins=_bins, weights=_mm)
                                _tc = 0.5 * (_bins[1:] + _bins[:-1])
                                _fit = fit_delayed_bq(_tc, _hm / (ARCH_BIN_MYR * 1e6),
                                                      _t_obs)
                                if _fit:
                                    for _k2 in ("tau_main_myr", "age_main_myr",
                                                "age_bq_myr", "r_sfr"):
                                        _row[_k2] = float(_fit[_k2])
                                    _row["fit_r2"] = float(_fit["r2"])
                    return _row

                for _lab, _r in zip(APERTURE_LABELS, _apr):     # cumulative rungs
                    _rows.append(_measure(_rproj <= _r, _lab, float(_r)))
                for _ki in range(1, len(_apr)):                 # true annuli (outer rung
                    _rows.append(_measure((_rproj > _apr[_ki - 1])   # names the annulus)
                                          & (_rproj <= _apr[_ki]),
                                          ANNULUS_LABELS[_ki], float(_apr[_ki])))
        print(f"snap {_snap:3d} (z={_zs:.2f}): aperture+annulus truth for {_n_gal} galaxies")
    APERTURE_TRUTH = Table(rows=_rows)
    APERTURE_TRUTH.write(APERTURE_TRUTH_FITS, overwrite=True)
    print(f"{len(APERTURE_TRUTH)} rows ({len(INCL_LABELS)} sightlines x "
          f"{len(APERTURE_LABELS)} apertures + {len(APERTURE_LABELS) - 1} annuli) "
          f"-> {APERTURE_TRUTH_FITS}")


# Part 7f — did the fit recover $A_V$? (merged results vs SIMBA truth)

Collects every `<run_dir>/out/results.fits` under `cigale_runs_pinned/` into one table, joins it to
the aperture-matched SIMBA truth (Part 7e) and to the **true** attenuation measured per aperture
*and* per sightline from the `dust_on`/`dust_off` Johnson-V fluxes, then writes
`tables/cigale_pinned_results.fits` plus the figure set.

Run it after the array drains — `collect_results` reports how many runs are missing, which is the
"did it finish?" check.

## What is actually measured here

Pinning the SFH is what makes $A_V$ a measurand, and it is also what makes most of the other
`bayes.*` columns *not* measurements. Being explicit about which is which is the whole point:

| CIGALE output | status | truth it is checked against |
|---|---|---|
| `attenuation.Av_ISM`, `Av_BC`, `attenuation.generic.bessell.V` | **recovered — the measurand** | `A_V_true` = $-2.5\log_{10}(F^{\rm on}_V/F^{\rm off}_V)$ per (aperture, sightline) |
| `stellar.m_star` | **recovered** — the only free normalisation | `mstar` (both are the *surviving* mass: `bc03.py` returns `info_all["m_star"]`, and Part 7e sums current particle masses) |
| `dust.mass`, `dust.umean` | **recovered** from the FIR (`qpah`/`umin`/`gamma` free) | no aperture-matched SIMBA dust truth yet — distributions only |
| `agn.fracAGN` (`agn_on`) | **recovered** | `fracAGN_true_V` = $1 - F^{\rm on}_V/F^{\rm agn}_V$ |
| `dust.luminosity` | **derived, not independent** — energy balance forces it to equal $L_{\rm abs}$($M_*$, pinned SFH, $A_V$), so it is a restatement of $A_V$ | — |
| `stellar.age_m_star`, `stellar.metallicity` | **PINNED inputs** | `age_m_star_myr`, `met_star` — a **closure test** of the archive + age-cap + mfrac chain, not a recovery. Deviation here is a bug, not physics |
| `sfh.sfr`, `sfh.sfr10Myrs`, `sfh.sfr100Myrs` | **shape pinned, scale fitted** — sSFR is an input; only the absolute value inherits the fitted $M_*$ | `sfr25`, `sfr100` |
| `param.*` (Dn4000, EWs, rest-frame colours) | predictions of a pinned model | diagnostics |

## The zero-point

The `dust_off` control runs have truth $A_V \equiv 0$ by construction, so their mean recovered
`Av_ISM` is the **BC03-vs-FSPS zero-point**: the optical colour offset between the SPS library
powderday rendered with (FSPS, MIST + MILES) and the one CIGALE fits with (BC03, Padova94 + STELIB),
cashed out as spurious dust because with everything else pinned $A_V$ is the only knob that can
absorb it. It is annotated on every $A_V$ panel and reported as `AV_ZP` — quote it as a systematic,
or subtract it, but do not ignore it. `av_residual_vs_true.png` gives the independent in-sample
estimate (the intercept as $A_V^{\rm true}\to0$); the two should agree.

## Figures

`av_recovered_vs_true_by_aperture.png` is the headline. `av_sightline_spread.png` is the one only a
per-object pin makes possible: at fixed stellar population, does an SED fit track **orientation**?
`pinned_closure.png` is the sanity check on the pipeline itself. `chi2_distribution.png` is a real
goodness-of-fit statistic here — a pinned fit has ~5 effective free parameters against ~41 bands, so
a systematic $\chi^2_{\rm red}$ floor *is* the SSP-mismatch measurement.

In [ ]:
# ── Part 7f: merged CIGALE results vs the SIMBA truth ────────────────────────
# Self-contained after Part 0 (+ Part 7e's aperture_truth.fits and the Part 7
# catalogs). One row per (arm, aperture, sightline, snapshot, galaxy):
#   bayes.*  from <run_dir>/out/results.fits            (the fit)
#   mstar/sfr/age/met from tables/aperture_truth.fits   (Part 7e, same geometry)
#   A_V_true from the dust_on/dust_off Johnson-V ratio  (per ap AND sightline —
#            Part 7a only builds the fiducial sightline)
# Part 7d puts MANY sources in one run (they share the pinned grid), so a
# results.fits is a table, not a row, and identity arrives on two channels:
# what is constant over the run (chain, SFH pin, Z node) from the directory
# name via cg.collect_results' id_map, and what varies row to row (arm,
# aperture, sightline) from the source id via parse_pin_id. cg.compare_results
# still has nothing to compare against, so the join and the figures live here.
from simbanator.sed import cigale as cg
from simbanator.sed.flux_extraction import attenuation_mag

RUN_BASE_PIN = globals().get("RUN_BASE_PIN",
                             os.path.join(OUT, "cigale_runs_pinned"))
RESULTS_FITS = os.path.join(TABLEDIR, "cigale_pinned_results.fits")
AV_STATS_FITS = os.path.join(TABLEDIR, "cigale_pinned_av_stats.fits")
ARMS_PLOT = ("dust_on", "agn_on")

if "parse_pin_run" not in globals():        # kernel-restart safe (see Part 7d)
    _TAG_RE = re.compile(r"^(dust_on|dust_off|agn_on)_(ap[0-9]+kpc)_(i\d+p\d+)$")
    PIN_RE = re.compile(r"^pin_(dust|agn)_(?:(ap\d+kpc)_)?snap(\d+)_gal(\d+)"
                        r"_z(\d+)$")

    def parse_pin_run(name):
        m = PIN_RE.match(name)
        if m is None:
            return None
        return dict(chain=m.group(1), pin_ap=m.group(2) or "",
                    snap=int(m.group(3)), gal_id=int(m.group(4)),
                    zs_idx=int(m.group(5)))

    def parse_pin_id(row_id):
        sid, tag = cg.parse_stacked_id(str(row_id))
        m = _TAG_RE.match(tag)
        if m is None or "_gal" not in sid:
            return None
        return dict(id=sid, snap=int(sid.split("_gal")[0][4:]),
                    gal_id=int(sid.split("_gal")[1]), arm=m.group(1),
                    aperture=m.group(2), incl=m.group(3))


def _col(t, name, default=np.nan):
    """Column as a plain float array.

    vstack(join_type='outer') and the left join both leave MASKED columns
    wherever a run's chain did not have that parameter (e.g. agn.fracAGN in
    the dust_on runs) — comparing those directly silently propagates masks.
    """
    if name not in t.colnames:
        return np.full(len(t), default, float)
    c = t[name]
    if hasattr(c, "filled"):
        return np.asarray(c.astype(float).filled(default), float)
    return np.asarray(c, float)


# ── 1. merge every results.fits (one collect per CHAIN keeps columns uniform;
#      the dust chain carries both dust_on and dust_off rows) ──
def _run_consts(name):
    """Only what is genuinely constant over the run — the rest is per row."""
    m = parse_pin_run(name)
    return None if m is None else {k: m[k] for k in ("chain", "pin_ap",
                                                     "zs_idx")}


_parts = []
for _chain in ("dust", "agn"):
    _dirs = sorted(glob.glob(os.path.join(RUN_BASE_PIN, f"pin_{_chain}_*")))
    if not _dirs:
        print(f"[{_chain}] no run dirs — skipped")
        continue
    print(f"[{_chain}]", end=" ")
    try:
        _parts.append(cg.collect_results(_dirs, id_map=_run_consts))
    except RuntimeError as _e:
        print(f"   {_e}")
if not _parts:
    raise RuntimeError(f"no results under {RUN_BASE_PIN} — has the Part 7d "
                       "array drained? (sbatch submit_cigale_pin.job)")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    RES = vstack(_parts, join_type="outer", metadata_conflicts="silent")
print(f"\nmerged: {len(RES)} fits")

# per-ROW identity. Many sources share a run, so the arm/aperture/sightline are
# NOT in the directory name — cg.stack_cigale_inputs put them in the id.
_ident = [parse_pin_id(i) for i in np.asarray(RES["id"], str)]
_unp = [i for i, r in zip(np.asarray(RES["id"], str), _ident) if r is None]
if _unp:
    raise RuntimeError(
        f"{len(_unp)} result row id(s) do not parse, e.g. {_unp[:3]} — Part "
        "7d's stack tags and parse_pin_id have drifted apart, and every "
        "downstream join keys off these")
for _k in ("snap", "gal_id", "arm", "aperture", "incl"):
    RES[_k] = np.array([r[_k] for r in _ident])
RES["id"] = np.array([r["id"] for r in _ident])   # back to snapNNN_galID
print("   per-row identity from the ids: "
      + ", ".join(f"{a}={int((RES['arm'] == a).sum())}"
                  for a in sorted(set(np.asarray(RES["arm"], str)))))

# ── 2. aperture-matched SIMBA truth (Part 7e), cumulative apertures only ──
_aptf = os.path.join(TABLEDIR, "aperture_truth.fits")
if not os.path.exists(_aptf):
    raise RuntimeError(f"{_aptf} missing — run Part 7e first")
APT = Table.read(_aptf)
APT = APT[[not str(a).startswith("ann") for a in APT["aperture"]]]
APT = APT["snap", "gal_id", "incl", "aperture", "ap_kpc", "nstar_ap",
          "mstar", "sfr25", "sfr100", "age_m_star_myr", "met_star"]
for _c in ("incl", "aperture"):
    APT[_c] = np.char.strip(np.asarray(APT[_c], str))
    RES[_c] = np.char.strip(np.asarray(RES[_c], str))
TAB = join(RES, APT, keys=("snap", "gal_id", "incl", "aperture"),
           join_type="left")
print(f"joined to aperture truth: "
      f"{int(np.isfinite(_col(TAB, 'mstar')).sum())}/{len(TAB)} rows matched")

# ── 3. the TRUE attenuation, per aperture AND per sightline ──
#    A_lambda = -2.5 log10(F_on / F_off) from the Part 7 rest-frame catalogs.
#    (Part 7a builds this for the fiducial sightline only, and adds the ISM
#     correlations; here we need all four to compare against the fits.)
ATTEN_BANDS = {"A_V_true": "Johnson.V.V", "A_U_true": "Johnson2.U.U",
               "A_J_true": "2MASS.J.J"}


def _band_map(arm, ap, incl, band):
    f = os.path.join(CATDIR, f"catalog_{arm}_{ap}_{incl}.fits")
    if not os.path.exists(f):
        return None
    t = Table.read(f)
    if band not in t.colnames:
        return None
    return {(int(s), int(g)): float(v)
            for s, g, v in zip(t["snap"], t["gal_id"], t[band])}


_AV, _AGN = {k: {} for k in ATTEN_BANDS}, {"A_V_true_agn": {}, "fracAGN_true_V": {}}
for _ap in APERTURE_LABELS:
    for _il in INCL_LABELS:
        for _key, _band in ATTEN_BANDS.items():
            _on = _band_map("dust_on", _ap, _il, _band)
            _off = _band_map("dust_off", _ap, _il, _band)
            if not (_on and _off):
                continue
            for _k in _on.keys() & _off.keys():
                _AV[_key][(_ap, _il) + _k] = attenuation_mag(_on[_k], _off[_k])
            if _band != ATTEN_BANDS["A_V_true"]:
                continue
            _agn = _band_map("agn_on", _ap, _il, _band)
            if not _agn:
                continue
            for _k in _agn.keys() & _off.keys():
                _AGN["A_V_true_agn"][(_ap, _il) + _k] = attenuation_mag(
                    _agn[_k], _off[_k])
            for _k in _agn.keys() & _on.keys():
                _AGN["fracAGN_true_V"][(_ap, _il) + _k] = (
                    1.0 - _on[_k] / _agn[_k] if _agn[_k] > 0 else np.nan)

_keys = [(str(r["aperture"]), str(r["incl"]), int(r["snap"]), int(r["gal_id"]))
         for r in TAB]
for _name, _db in list(_AV.items()) + list(_AGN.items()):
    TAB[_name] = np.array([_db.get(k, np.nan) for k in _keys], float)
TAB["S_UV_true"] = TAB["A_U_true"] - TAB["A_V_true"]     # true curve slope
print(f"true A_V per (aperture, sightline): "
      f"{int(np.isfinite(_col(TAB, 'A_V_true')).sum())}/{len(TAB)} rows")

# ── 4. residuals + the zero-point from the dust_off control ──
AV_CIG = _col(TAB, "bayes.attenuation.Av_ISM")
TAB["dAv"] = AV_CIG - _col(TAB, "A_V_true")
TAB["dlogMstar"] = (np.log10(np.clip(_col(TAB, "bayes.stellar.m_star"), 1e-30, None))
                    - np.log10(np.clip(_col(TAB, "mstar"), 1e-30, None)))
_arm_a = np.asarray(TAB["arm"], str)
_ap_a = np.char.strip(np.asarray(TAB["aperture"], str))
_ctrl = (_arm_a == "dust_off") & np.isfinite(AV_CIG)
AV_ZP = float(np.median(AV_CIG[_ctrl])) if _ctrl.any() else np.nan
AV_ZP_NMAD = float(cg.nmad(AV_CIG[_ctrl])) if _ctrl.any() else np.nan
# dust_off rides along in the dust chain at NO extra grid cost, so the whole
# aperture x sightline set is fitted and the zero point is measurable PER
# APERTURE — the FSPS-vs-BC03 offset follows the stellar population, and the
# inner apertures do not hold the same one as the outskirts.
AV_ZP_AP = {}
for _ap in APERTURE_LABELS:
    _m = _ctrl & (_ap_a == _ap)
    if _m.sum() >= 5:
        AV_ZP_AP[_ap] = (float(np.median(AV_CIG[_m])),
                         float(cg.nmad(AV_CIG[_m])), int(_m.sum()))
TAB.meta["AV_ZP"] = AV_ZP
for _ap, (_z, _nm, _n) in AV_ZP_AP.items():
    TAB.meta[f"AV_ZP_{_ap}"] = _z
TAB["Av_zp"] = np.array([AV_ZP_AP.get(a, (AV_ZP,))[0] for a in _ap_a], float)
TAB["dAv_zpcorr"] = _col(TAB, "dAv") - np.asarray(TAB["Av_zp"], float)
if _ctrl.any():
    print(f"\n[zero-point] dust_off, truth A_V == 0: n={int(_ctrl.sum())}, "
          f"median Av_cig = {AV_ZP:+.3f} +- {AV_ZP_NMAD:.3f} (NMAD)")
    print("   the BC03(Padova94+STELIB)-vs-FSPS(MIST+MILES) colour offset that "
          "a fully pinned population can only absorb as dust — per aperture:")
    for _ap in APERTURE_LABELS:
        if _ap in AV_ZP_AP:
            _z, _nm, _n = AV_ZP_AP[_ap]
            print(f"      {_ap:>10s}  {_z:+.3f} +- {_nm:.3f}   (n={_n})")
    if len(AV_ZP_AP) > 1:
        _sp = max(v[0] for v in AV_ZP_AP.values()) - min(
            v[0] for v in AV_ZP_AP.values())
        print(f"   spread across apertures: {_sp:.3f} mag — a single global "
              "number would misstate the systematic by this much at the ends.")
    print("   'dAv_zpcorr' in the output table is dAv with the per-aperture "
          "zero point subtracted.")
else:
    print("\n[zero-point] no dust_off fits found — check FIT_ARMS in Part 7d; "
          "the A_V systematic is then uncalibrated")
TAB.write(RESULTS_FITS, overwrite=True)
print(f"-> {RESULTS_FITS}")

# ── 5. per (arm, aperture, sightline) summary ──
_rows = []
for _arm in sorted(set(_arm_a.tolist())):
    for _ap in APERTURE_LABELS:
        for _il in INCL_LABELS:
            _m = ((_arm_a == _arm) & (np.asarray(TAB["aperture"], str) == _ap)
                  & (np.asarray(TAB["incl"], str) == _il))
            _d = _col(TAB, "dAv")[_m]
            _d = _d[np.isfinite(_d)]
            if _d.size == 0:
                continue
            _dm = _col(TAB, "dlogMstar")[_m]
            _dm = _dm[np.isfinite(_dm)]
            _c2 = _col(TAB, "best.reduced_chi_square")[_m]
            _c2 = _c2[np.isfinite(_c2)]
            _rows.append(dict(
                arm=_arm, aperture=_ap, incl=_il, n=int(_d.size),
                dAv_med=float(np.median(_d)), dAv_nmad=float(cg.nmad(_d)),
                dlogM_med=float(np.median(_dm)) if _dm.size else np.nan,
                dlogM_nmad=float(cg.nmad(_dm)) if _dm.size else np.nan,
                chi2red_med=float(np.median(_c2)) if _c2.size else np.nan))
if _rows:
    Table(rows=_rows).write(AV_STATS_FITS, overwrite=True)
    print(f"-> {AV_STATS_FITS}")
    print(f"\n{'arm':9s} {'aperture':>10s}   n   median dAv   NMAD   chi2red")
    for _arm in ARMS_PLOT:
        for _ap in APERTURE_LABELS:
            _s = [r for r in _rows if r["arm"] == _arm and r["aperture"] == _ap]
            if not _s:
                continue
            _n = sum(r["n"] for r in _s)
            _md = float(np.median([r["dAv_med"] for r in _s]))
            _nm = float(np.median([r["dAv_nmad"] for r in _s]))
            _c2 = float(np.nanmedian([r["chi2red_med"] for r in _s]))
            print(f"{_arm:9s} {_ap:>10s} {_n:4d}   {_md:+9.3f}  {_nm:6.3f}  "
                  f"{_c2:8.2f}")

# ── 6. figures ────────────────────────────────────────────────────────────────
_SNAPC = {s: c for s, c in zip(sorted(set(np.asarray(TAB["snap"], int))),
                               plt.cm.viridis(np.linspace(0.1, 0.9, 4)))}
_MRK = dict(zip(INCL_LABELS, ["o", "s", "^", "D"]))


def _sel(arm=None, ap=None, finite=("A_V_true", "bayes.attenuation.Av_ISM")):
    m = np.ones(len(TAB), bool)
    if arm is not None:
        m &= _arm_a == arm
    if ap is not None:
        m &= np.asarray(TAB["aperture"], str) == ap
    for c in finite:
        m &= np.isfinite(_col(TAB, c))
    return m


def _scatter(ax, m, x, y):
    for _il in INCL_LABELS:
        for _sn, _c in _SNAPC.items():
            k = m & (np.asarray(TAB["incl"], str) == _il) \
                  & (np.asarray(TAB["snap"], int) == _sn)
            if k.any():
                ax.scatter(x[k], y[k], s=13, marker=_MRK[_il], color=_c,
                           alpha=0.7, lw=0.3, edgecolor="k")


_AVT = _col(TAB, "A_V_true")
_DAV = _col(TAB, "dAv")

# 1. HEADLINE — recovered vs true A_V, per arm x aperture
fig, axs = plt.subplots(len(ARMS_PLOT), len(APERTURE_LABELS),
                        figsize=(4.8 * len(APERTURE_LABELS), 4.8 * len(ARMS_PLOT)),
                        squeeze=False, sharex=True, sharey=True)
for _i, _arm in enumerate(ARMS_PLOT):
    for _j, _ap in enumerate(APERTURE_LABELS):
        ax = axs[_i][_j]
        m = _sel(_arm, _ap)
        _scatter(ax, m, _AVT, AV_CIG)
        _lim = [-0.05, max(1.0, np.nanpercentile(_AVT[m], 99) if m.any() else 1.0)]
        ax.plot(_lim, _lim, "k--", lw=1)
        if np.isfinite(AV_ZP):
            ax.plot(_lim, [v + AV_ZP for v in _lim], ":", color="C3", lw=1)
        if _i == 0:
            ax.set_title(_ap)
        if m.any():
            ax.text(0.04, 0.96, f"n={int(m.sum())}\n"
                    f"med {np.median(_DAV[m]):+.2f}\n"
                    f"NMAD {cg.nmad(_DAV[m]):.2f}", fontsize=9,
                    transform=ax.transAxes, va="top")
        ax.grid(alpha=0.3)
        if _j == 0:
            ax.set_ylabel(f"{_arm}\n" + r"CIGALE $A_V^{\rm ISM}$")
        if _i == len(ARMS_PLOT) - 1:
            ax.set_xlabel(r"true $A_V$")
fig.suptitle(f"recovered vs true $A_V$  (dotted: zero point {AV_ZP:+.2f})",
             y=1.005)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "av_recovered_vs_true_by_aperture.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# 2. residual vs true A_V — the in-sample zero-point estimate
fig, axs = plt.subplots(1, len(ARMS_PLOT), figsize=(8 * len(ARMS_PLOT), 6),
                        squeeze=False, sharey=True)
for _i, _arm in enumerate(ARMS_PLOT):
    ax = axs[0][_i]
    m = _sel(_arm)
    _scatter(ax, m, _AVT, _DAV)
    if m.any():
        _b = np.nanpercentile(_AVT[m], np.linspace(0, 100, 9))
        _b = np.unique(np.round(_b, 4))
        _xc, _md, _sc = [], [], []
        for _lo, _hi in zip(_b[:-1], _b[1:]):
            k = m & (_AVT >= _lo) & (_AVT < _hi)
            if k.sum() >= 5:
                _xc.append(0.5 * (_lo + _hi))
                _md.append(np.median(_DAV[k]))
                _sc.append(cg.nmad(_DAV[k]))
        if _xc:
            ax.plot(_xc, _md, "-", color="C3", lw=2, label="running median")
            ax.fill_between(_xc, np.array(_md) - np.array(_sc),
                            np.array(_md) + np.array(_sc), color="C3", alpha=0.2)
    ax.axhline(0, color="k", ls="--", lw=1)
    if np.isfinite(AV_ZP):
        ax.axhline(AV_ZP, color="C0", ls=":", lw=1.2,
                   label=f"dust_off zero-point {AV_ZP:+.2f}")
    ax.set(xlabel=r"true $A_V$", title=_arm)
    ax.grid(alpha=0.3)
    if _i == 0:
        ax.set_ylabel(r"$\Delta A_V$ (CIGALE $-$ true)")
        ax.legend(fontsize=10, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "av_residual_vs_true.png"), dpi=150,
            bbox_inches="tight")
plt.show()

# 3. residual vs aperture — does the CGM entering the aperture break it?
fig, ax = plt.subplots(figsize=(11, 6))
_w = 0.34
for _i, _arm in enumerate(ARMS_PLOT):
    _data = [_DAV[_sel(_arm, _ap)] for _ap in APERTURE_LABELS]
    _data = [d[np.isfinite(d)] for d in _data]
    _pos = np.arange(len(APERTURE_LABELS)) + (_i - 0.5) * _w
    _bp = ax.boxplot([d if d.size else [np.nan] for d in _data], positions=_pos,
                     widths=_w * 0.85, patch_artist=True, showfliers=False,
                     medianprops=dict(color="k"))
    for _p in _bp["boxes"]:
        _p.set(facecolor=f"C{_i}", alpha=0.55)
    ax.plot([], [], "s", color=f"C{_i}", label=_arm)
ax.axhline(0, color="k", ls="--", lw=1)
if np.isfinite(AV_ZP):
    ax.axhline(AV_ZP, color="C3", ls=":", lw=1.2, label="zero-point")
ax.set(xticks=np.arange(len(APERTURE_LABELS)), xticklabels=APERTURE_LABELS,
       ylabel=r"$\Delta A_V$")
ax.legend(fontsize=10, frameon=False)
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "av_residual_vs_aperture.png"), dpi=150,
            bbox_inches="tight")
plt.show()

# 4. sightline spread — only a per-object pin makes this measurable
_gkey = {}
for _i, _r in enumerate(TAB):
    if _arm_a[_i] != "dust_on":
        continue
    _gkey.setdefault((int(_r["snap"]), int(_r["gal_id"]),
                      str(_r["aperture"]).strip()), []).append(_i)
_sx, _sy, _sc2 = [], [], []
for _k, _idx in _gkey.items():
    if len(_idx) < 3:
        continue
    _a, _b = _AVT[_idx], AV_CIG[_idx]
    if np.isfinite(_a).sum() < 3 or np.isfinite(_b).sum() < 3:
        continue
    _sx.append(np.nanstd(_a))
    _sy.append(np.nanstd(_b))
    _sc2.append(APERTURE_LABELS.index(_k[2]))
fig, ax = plt.subplots(figsize=(8, 7.2))
if _sx:
    _s = ax.scatter(_sx, _sy, c=_sc2, cmap="viridis", s=22, alpha=0.8, lw=0.3,
                    edgecolor="k", vmin=-0.5, vmax=len(APERTURE_LABELS) - 0.5)
    _cb = fig.colorbar(_s, ax=ax, ticks=range(len(APERTURE_LABELS)))
    _cb.ax.set_yticklabels(APERTURE_LABELS, fontsize=7)
    _lim = [0, max(max(_sx), max(_sy)) * 1.05]
    ax.plot(_lim, _lim, "k--", lw=1)
    ax.set(xlim=_lim, ylim=_lim)
ax.set(xlabel=r"$\sigma_{\rm sightline}(A_V^{\rm true})$",
       ylabel=r"$\sigma_{\rm sightline}(A_V^{\rm CIGALE})$",
       title="sightline spread (dust_on)")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "av_sightline_spread.png"), dpi=150,
            bbox_inches="tight")
plt.show()

# 5. the zero-point control itself
if _ctrl.any():
    fig, ax = plt.subplots(figsize=(10, 6))
    _bins = np.linspace(0, max(0.6, np.nanpercentile(AV_CIG[_ctrl], 99)), 30)
    ax.hist(AV_CIG[_ctrl], bins=_bins, color="0.75", label="all apertures")
    for _ap, _c in zip(APERTURE_LABELS,
                       plt.cm.viridis(np.linspace(0.1, 0.9,
                                                  len(APERTURE_LABELS)))):
        _m = _ctrl & (_ap_a == _ap)
        if _m.sum() < 5:
            continue
        ax.hist(AV_CIG[_m], bins=_bins, histtype="step", lw=1.6, color=_c,
                label=f"{_ap} ({AV_ZP_AP[_ap][0]:+.3f})")
    ax.axvline(0, color="k", ls="--", lw=1, label="truth ($A_V \\equiv 0$)")
    ax.axvline(AV_ZP, color="C3", lw=2,
               label=f"global {AV_ZP:+.3f} $\\pm$ {AV_ZP_NMAD:.3f}")
    ax.set(xlabel=r"CIGALE $A_V^{\rm ISM}$ on dust-free photometry",
           ylabel="fits", title="zero-point control (dust_off)")
    ax.legend(fontsize=10, frameon=False, ncol=2)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTDIR, "av_zeropoint_control.png"), dpi=150,
                bbox_inches="tight")
    plt.show()

# 6. stellar mass — the one other genuine recovery, and its dust degeneracy
fig, axs = plt.subplots(1, 2, figsize=(15, 6.5))
m = _sel("dust_on", finite=("mstar", "bayes.stellar.m_star"))
_scatter(axs[0], m, np.log10(np.clip(_col(TAB, "mstar"), 1e-30, None)),
         np.log10(np.clip(_col(TAB, "bayes.stellar.m_star"), 1e-30, None)))
if m.any():
    _l = [np.nanmin(np.log10(np.clip(_col(TAB, "mstar")[m], 1e-30, None))),
          np.nanmax(np.log10(np.clip(_col(TAB, "mstar")[m], 1e-30, None)))]
    axs[0].plot(_l, _l, "k--", lw=1)
axs[0].set(xlabel=r"true $\log M_\star$ (aperture)",
           ylabel=r"CIGALE $\log M_\star$", title="dust_on")
m2 = _sel("dust_on", finite=("A_V_true", "dlogMstar"))
_scatter(axs[1], m2, _AVT, _col(TAB, "dlogMstar"))
axs[1].axhline(0, color="k", ls="--", lw=1)
axs[1].set(xlabel=r"true $A_V$", ylabel=r"$\Delta \log M_\star$",
           title="mass\u2013dust degeneracy")
for ax in axs:
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "mstar_recovery.png"), dpi=150,
            bbox_inches="tight")
plt.show()

# 7. closure test on the PINNED inputs — deviation here is a bug, not physics
fig, axs = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, (_cig, _tru, _lab) in zip(axs, [
        ("bayes.stellar.age_m_star", "age_m_star_myr",
         r"mass-weighted age [Myr]"),
        ("bayes.stellar.metallicity", "met_star", r"stellar $Z$")]):
    m = _sel("dust_on", finite=(_cig, _tru))
    _scatter(ax, m, _col(TAB, _tru), _col(TAB, _cig))
    if m.any():
        _l = [np.nanmin(_col(TAB, _tru)[m]), np.nanmax(_col(TAB, _tru)[m])]
        ax.plot(_l, _l, "k--", lw=1)
    ax.set(xlabel=f"SIMBA {_lab}", ylabel=f"CIGALE {_lab}")
    ax.grid(alpha=0.3)
fig.suptitle("pinned-input closure (not a recovery)", y=1.0)
fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "pinned_closure.png"), dpi=150,
            bbox_inches="tight")
plt.show()

# 8. the AGN's effect on the dust measurement
if (_arm_a == "agn_on").any():
    fig, axs = plt.subplots(1, 2, figsize=(15, 6.5))
    m = _sel("agn_on", finite=("fracAGN_true_V", "bayes.agn.fracAGN"))
    _scatter(axs[0], m, _col(TAB, "fracAGN_true_V"),
             _col(TAB, "bayes.agn.fracAGN"))
    axs[0].plot([0, 1], [0, 1], "k--", lw=1)
    axs[0].set(xlabel=r"true $f_{\rm AGN}$ (V band, $1-F_{\rm on}/F_{\rm agn}$)",
               ylabel=r"CIGALE $f_{\rm AGN}$", title="AGN fraction")
    # per-object dAv(agn_on) - dAv(dust_on)
    _don = {(int(r["snap"]), int(r["gal_id"]), str(r["aperture"]).strip(),
             str(r["incl"]).strip()): _DAV[i]
            for i, r in enumerate(TAB) if _arm_a[i] == "dust_on"}
    _x, _y = [], []
    for i, r in enumerate(TAB):
        if _arm_a[i] != "agn_on":
            continue
        _k = (int(r["snap"]), int(r["gal_id"]), str(r["aperture"]).strip(),
              str(r["incl"]).strip())
        if _k in _don and np.isfinite(_DAV[i]) and np.isfinite(_don[_k]):
            _x.append(_col(TAB, "fracAGN_true_V")[i])
            _y.append(_DAV[i] - _don[_k])
    axs[1].scatter(_x, _y, s=14, alpha=0.7, lw=0.3, edgecolor="k")
    axs[1].axhline(0, color="k", ls="--", lw=1)
    axs[1].set(xlabel=r"true $f_{\rm AGN}$ (V)",
               ylabel=r"$\Delta A_V$(agn_on) $-$ $\Delta A_V$(dust_on)",
               title=r"AGN effect on $\Delta A_V$")
    for ax in axs:
        ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTDIR, "agn_bias.png"), dpi=150,
                bbox_inches="tight")
    plt.show()

# 9. goodness of fit — a real statistic now (~5 free params vs ~41 bands)
_C2 = _col(TAB, "best.reduced_chi_square")
if np.isfinite(_C2).any():
    fig, ax = plt.subplots(figsize=(11, 6))
    _bins = np.logspace(np.log10(max(np.nanpercentile(_C2, 1), 1e-2)),
                        np.log10(max(np.nanpercentile(_C2, 99), 1.0)), 40)
    for _i, _arm in enumerate(("dust_on", "agn_on", "dust_off")):
        _v = _C2[(_arm_a == _arm) & np.isfinite(_C2)]
        if _v.size:
            ax.hist(_v, bins=_bins, histtype="step", lw=1.8, color=f"C{_i}",
                    label=f"{_arm} (median {np.median(_v):.2f})")
    ax.axvline(1.0, color="k", ls="--", lw=1)
    ax.set(xscale="log", xlabel=r"best $\chi^2_{\rm red}$", ylabel="fits")
    ax.legend(fontsize=10, frameon=False)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTDIR, "chi2_distribution.png"), dpi=150,
                bbox_inches="tight")
    plt.show()

# ── 7. does the edge-held recent SFH bias A_V? (Part 7d's sfr_hold_frac) ──
_holdf = os.path.join(TABLEDIR, "cigale_pinned_sfrhold.fits")
if os.path.exists(_holdf):
    # one SFH per RUN now, so the ledger is keyed by the run name
    _ht = Table.read(_holdf)
    _hd = {str(r["run"]).strip(): float(r["sfr_hold_frac"]) for r in _ht}
    _hv = np.array([_hd.get(x, np.nan)
                    for x in np.char.strip(np.asarray(TAB["run"], str))], float)
    _m = np.isfinite(_hv) & np.isfinite(_DAV)
    if not np.isfinite(_hv).any():
        print("\n[sfr_hold_frac] the ledger joined to ZERO fits — its keys and "
              "the run identity have drifted apart. Not fatal, but the "
              "edge-held-SFH check below is void, so fix Part 7d's _holds "
              "before quoting the A_V residual.")
    elif _m.sum() > 20 and np.nanstd(_hv[_m]) > 0:
        print(f"\n[sfr_hold_frac] corr(hold_frac, dAv) = "
              f"{np.corrcoef(_hv[_m], _DAV[_m])[0, 1]:+.3f} over "
              f"{int(_m.sum())} fits (|r| < 0.2 -> the edge-held recent SFH is "
              "not driving the A_V residual)")

# ── 8. what the array did not deliver ──
_skipf = os.path.join(TABLEDIR, "cigale_pinned_skipped.fits")
if os.path.exists(_skipf):
    _sk = Table.read(_skipf)
    print(f"\n[coverage] {len(_sk)} (arm, aperture, sightline, galaxy) "
          f"combinations were never prepared (see {os.path.basename(_skipf)}); "
          f"{len(TAB)} fits are in hand.")
    _u, _c = np.unique(np.asarray(_sk["reason"], str), return_counts=True)
    for _r, _n in sorted(zip(_u, _c), key=lambda x: -x[1]):
        print(f"   {_n:5d}  {_r}")

# Part 8 — Red cores: surviving ISM dust vs AGN coupling class

Can the red cores of these quenched galaxies be explained by **surviving ISM dust** — and does the
answer differ across the AGN coupling classes (`agn_class` / `xstr_quench`)? Five tests:

- **T1 (decomposition, 8c)** — is the red core dust at all? $\Delta(U\!-\!V)_{\rm dust}(R) =
  (U\!-\!V)_{\rm on} - (U\!-\!V)_{\rm off}$ isolates the dust part of the colour gradient; the
  dust_off gradient is the intrinsic (age/$Z$) part.
- **T2 (sufficiency, 8d)** — does the surviving dust column carry the attenuation? Annular $A_V$
  vs $\Sigma_{\rm dust}$ against the foreground-screen ceiling $A_{V,\rm screen} =
  1.086\,\kappa_V\,\Sigma_{\rm dust}$, with $\kappa_V$ read from the RT's own KMH94 dust model.
- **T3 (carrier, 8e)** — is the dust in *cold* surviving ISM? Central $A_V$ vs $f_{\rm cold}$,
  $\Sigma_{H_2}$, $\Sigma_{HI}$, DGR — raw, and rank-partial at fixed
  ($\log M_\star$, $\log\Sigma_{\rm dust}$).
- **T4 (classes, 8f)** — central dust/attenuation distributions by coupling class, plus the
  geometry test: $A_V$ residuals at fixed $\Sigma_{\rm dust}$ by class (an offset means coupling
  changes the dust *geometry*, not just the amount).
- **T5 (observability, 8g)** — does the pinned-CIGALE aperture $A_V$ (Part 7f) recover the same
  red-core signal and the same class ranking an observer would need?

Inputs: **8a** builds `tables/annulus_ism_truth.fits` — the per-aperture/annulus dust, gas and
cold-gas truth that Part 4b only *counted*; **8b** generalises Part 7a's annular $A_V$ (and adds
the annular $(U\!-\!V)$ colours) to all four sightlines. 8c–8g are pure reads of the caches.

**Caveats.** Annuli are never fitted with CIGALE (energy balance — closed decision, see the run
order); annular attenuation is the RT differential truth. The four sightlines of one galaxy are
not independent — every statistic below is bootstrapped over *galaxies*, and rank tests run on
per-galaxy medians.

In [ ]:
# ── Part 8a — per-aperture/annulus ISM truth: dust, gas, cold gas (cluster, cached) ──
# Self-contained after Parts 0/0b + the Stage-0 cutouts (Part 4). Part 4b only
# COUNTED particles per projected annulus; this cell sums their masses — the
# dust/gas truth the red-core tests join against. Same geometry as Parts 4b/4c/7e.
# dust/HI/H2 split + temperature come verbatim from build_profiles_job.py (repo root).
from build_profiles_job import _components, _temperature, _XH, T_COLD

ANNULUS_ISM_FITS      = os.path.join(TABLEDIR, "annulus_ism_truth.fits")
OVERWRITE_ANNULUS_ISM = False
KAPPA_V_FALLBACK = 3.0e4                            # cm^2 per g of DUST (MW-like)
KAPPA_UM = {"U": 0.365, "V": 0.551, "J": 1.235}     # rest-frame pivots [um]

def _kappa_rt():
    """Extinction opacity chi [cm^2 per g of DUST] of the RT dust model.

    Resolved exactly as parameters_master.py does: $POWDERDAY_ROOT (else ~) /
    hyperion-dust/dust_files/kmh94_3.1_hg.hdf5; chi is tabulated vs frequency.
    Falls back to a MW-like constant (Part 8d's screen then carries a ~x2 model
    uncertainty and says so).
    """
    _df = os.path.join(os.environ.get("POWDERDAY_ROOT", os.path.expanduser("~")),
                       "hyperion-dust", "dust_files", "kmh94_3.1_hg.hdf5")
    try:
        with h5py.File(_df, "r") as f:
            _nu  = np.asarray(f["optical_properties"]["nu"][:], float)
            _chi = np.asarray(f["optical_properties"]["chi"][:], float)
        _lam = 2.99792458e14 / _nu                  # c [um s^-1] / nu -> um
        _o = np.argsort(_lam)
        return ({b: float(np.interp(l, _lam[_o], _chi[_o])) for b, l in KAPPA_UM.items()},
                os.path.basename(_df))
    except (OSError, KeyError) as _e:
        print(f"[kappa] {_df} unreadable ({_e}) -> MW-like fallback "
              f"kappa_V={KAPPA_V_FALLBACK:g} cm2/g (A_U/A_V=1.55, A_J/A_V=0.28)")
        return ({"U": 1.55 * KAPPA_V_FALLBACK, "V": KAPPA_V_FALLBACK,
                 "J": 0.28 * KAPPA_V_FALLBACK}, "fallback")

ANNULUS_ISM = None
if os.path.exists(ANNULUS_ISM_FITS) and not OVERWRITE_ANNULUS_ISM:
    ANNULUS_ISM = Table.read(ANNULUS_ISM_FITS)
    print(f"cached ({len(ANNULUS_ISM)} rows) -> {ANNULUS_ISM_FITS}  "
          "(OVERWRITE_ANNULUS_ISM=True rebuilds)")
if ANNULUS_ISM is None:
    KAPPA, KAPPA_SRC = _kappa_rt()
    print(f"RT dust opacity [{KAPPA_SRC}]: "
          + "  ".join(f"kappa_{b}={v:.4g} cm2/g" for b, v in KAPPA.items()))
    SEL, SNAPS, IDS = load_selection()
    _cen = rt_centers(SNAPS, IDS)
    _G_FIELDS = ("Masses", "Dust_Masses", "Metallicity", "NeutralHydrogenAbundance",
                 "FractionH2", "InternalEnergy", "ElectronAbundance", "StarFormationRate")
    _rows, _skipped, _no_thermo = [], [], 0
    for _s, _g in zip(SNAPS, IDS):
        _cut = read_cutout(_s, _g, _cen.get((int(_s), int(_g))), "PartType0",
                           fields=_G_FIELDS)
        if _cut is None:
            _skipped.append((int(_s), int(_g)))
            continue
        _hh = _cut["h"]
        _m  = np.asarray(_cut["Masses"], float) * 1e10 / _hh
        _mdp = (np.asarray(_cut["Dust_Masses"], float) * 1e10 / _hh
                if _cut["Dust_Masses"] is not None else np.zeros_like(_m))
        _Z    = (np.asarray(_cut["Metallicity"], float)
                 if _cut["Metallicity"] is not None else None)
        _fnt  = (np.asarray(_cut["NeutralHydrogenAbundance"], float)
                 if _cut["NeutralHydrogenAbundance"] is not None else None)
        _fmol = (np.asarray(_cut["FractionH2"], float)
                 if _cut["FractionH2"] is not None else None)
        _sfr  = (np.asarray(_cut["StarFormationRate"], float)
                 if _cut["StarFormationRate"] is not None else np.zeros_like(_m))
        _mdust, _mHI, _mH2 = _components(_m, _mdp, _Z, _fnt, _fmol)
        if _cut["InternalEnergy"] is not None and _cut["ElectronAbundance"] is not None:
            _T = _temperature(np.asarray(_cut["InternalEnergy"], float),
                              np.asarray(_cut["ElectronAbundance"], float),
                              _XH(_Z, len(_m)))
        else:                                       # cold cut degrades to the SFR gate
            _T, _no_thermo = np.full(len(_m), np.nan), _no_thermo + 1
        # SF gas sits on the effective EOS (its T is not physical) -> count it
        # cold; M_sf is ledgered separately so the choice stays auditable
        _cold = (_T < T_COLD) | (_sfr > 0)
        for _j, _il in enumerate(INCL_LABELS):
            _R = projected_radius(_cut["pos"], NHAT[_j])

            def _measure(_msk, _lab, _rin, _rout):
                _area = np.pi * (_rout**2 - _rin**2)
                _mg, _mdu = float(_m[_msk].sum()), float(np.nansum(_mdust[_msk]))
                _mhi, _mh2 = float(np.nansum(_mHI[_msk])), float(np.nansum(_mH2[_msk]))
                _row = dict(snap=int(_s), gal_id=int(_g), incl=_il, aperture=_lab,
                            r_in_kpc=float(_rin), r_out_kpc=float(_rout),
                            area_kpc2=float(_area),
                            ngas=int(_msk.sum()),
                            ndust=int((_mdp[_msk] > 0).sum()),   # Part 4b convention
                            ncold=int(_cold[_msk].sum()),
                            M_gas=_mg, M_dust=_mdu, M_HI=_mhi, M_H2=_mh2,
                            M_cold=float(_m[_msk & _cold].sum()),
                            M_sf=float(_m[_msk & (_sfr > 0)].sum()),
                            Sigma_dust=_mdu / _area, Sigma_gas=_mg / _area,
                            Sigma_HI=_mhi / _area, Sigma_H2=_mh2 / _area)
                _row["DGR"]    = _mdu / _mg if _mg > 0 else np.nan
                _row["f_cold"] = _row["M_cold"] / _mg if _mg > 0 else np.nan
                _row["f_mol_ann"] = _mh2 / (_mh2 + _mhi) if (_mh2 + _mhi) > 0 else np.nan
                return _row

            for _k, _lab in enumerate(APERTURE_LABELS):              # cumulative rungs
                _rows.append(_measure(_R <= R_EDGES[_k + 1], _lab, 0.0, R_EDGES[_k + 1]))
            for _k, _lab in enumerate(ANNULUS_LABELS[1:], start=1):  # true annuli
                _rows.append(_measure((_R > R_EDGES[_k]) & (_R <= R_EDGES[_k + 1]),
                                      _lab, R_EDGES[_k], R_EDGES[_k + 1]))
    ANNULUS_ISM = Table(rows=_rows)
    ANNULUS_ISM.meta["R_EDGES"] = list(np.round(R_EDGES, 3))
    ANNULUS_ISM.meta["T_COLD"]  = T_COLD
    ANNULUS_ISM.meta["KAP_SRC"] = KAPPA_SRC
    for _b, _v in KAPPA.items():
        ANNULUS_ISM.meta[f"KAPPA_{_b}"] = _v
    ANNULUS_ISM.write(ANNULUS_ISM_FITS, overwrite=True)
    print(f"{len(ANNULUS_ISM)} rows ({len(ANNULUS_ISM) // (N_INCL * (2 * N_AP - 1))} galaxies x "
          f"{N_INCL} sightlines x {2 * N_AP - 1} labels) -> {ANNULUS_ISM_FITS}")
    if _skipped:
        print(f"[WARN] {len(_skipped)} galaxies without cutout/centre, skipped: {_skipped}")
    if _no_thermo:
        print(f"[WARN] {_no_thermo} galaxies lack InternalEnergy/ElectronAbundance -> "
              "their cold cut is the SFR>0 gate only")

# ── QC: sampling per label + galaxy-level DGR against Part 7a ──
_labs_qc = list(APERTURE_LABELS) + ANNULUS_LABELS[1:]
print(f"\n{'label':>10s} {'med Sig_dust':>13s} {'ngas<10':>8s} {'Mdust=0':>8s}")
for _lab in _labs_qc:
    _t = ANNULUS_ISM[np.asarray(ANNULUS_ISM["aperture"], str) == _lab]
    print(f"{_lab:>10s} {np.nanmedian(np.asarray(_t['Sigma_dust'], float)):13.3g} "
          f"{np.mean(np.asarray(_t['ngas'], int) < 10) * 100:7.0f}% "
          f"{np.mean(np.asarray(_t['M_dust'], float) == 0) * 100:7.0f}%")
# membership differs (100 pkpc cutout sphere vs caesar glist): ~<0.1 dex is
# expected agreement, >0.3 dex means a units bug, not physics
_avf = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")
if os.path.exists(_avf):
    _at = Table.read(_avf)
    _dgr7a = {(int(s), int(g)): float(d)
              for s, g, d in zip(_at["snap"], _at["gal_id"], _at["DGR"])}
    _t100 = ANNULUS_ISM[(np.asarray(ANNULUS_ISM["aperture"], str) == "ap100kpc")
                        & (np.asarray(ANNULUS_ISM["incl"], str) == INCL_LABELS[0])]
    with np.errstate(all="ignore"):
        _dl = np.array([abs(np.log10(float(_r["DGR"]))
                            - np.log10(_dgr7a.get((int(_r["snap"]), int(_r["gal_id"])),
                                                  np.nan)))
                        for _r in _t100])
    print(f"\n[DGR cross-check vs Part 7a] median |dlog DGR| = {np.nanmedian(_dl):.3f} dex "
          f"over {int(np.isfinite(_dl).sum())} galaxies (expect <~0.1; >0.3 = units bug)")
else:
    print("\n[DGR cross-check] attenuation_vs_ism.fits not found — run Part 7a to enable it")

In [ ]:
# ── Part 8b — annular A_V and (U-V) colours for ALL four sightlines (cached) ──
# Part 7a builds the annular A_V for the fiducial sightline only; the red-core
# tests need it per (label, sightline) to join the 8a ISM truth row-for-row.
# Same differential construction: F_ann = F(<r_out) - F(<r_in), band by band
# (filter convolution is linear in flux). Catalog fluxes are mJy f_nu, so
# -2.5 log10(F_U/F_V) is directly an AB colour.
from simbanator.sed.flux_extraction import attenuation_mag

ANNULUS_AV_FITS      = os.path.join(TABLEDIR, "annulus_av_allincl.fits")
OVERWRITE_ANNULUS_AV = False
_AV_BANDS = {"U": "Johnson2.U.U", "V": "Johnson.V.V", "J": "2MASS.J.J"}  # rest-frame

ANNULUS_AV = None
if os.path.exists(ANNULUS_AV_FITS) and not OVERWRITE_ANNULUS_AV:
    ANNULUS_AV = Table.read(ANNULUS_AV_FITS)
    print(f"cached ({len(ANNULUS_AV)} rows) -> {ANNULUS_AV_FITS}  "
          "(OVERWRITE_ANNULUS_AV=True rebuilds)")
if ANNULUS_AV is None:
    SEL, SNAPS, IDS = load_selection()
    _keys = [(int(s), int(g)) for s, g in zip(SNAPS, IDS)]
    _rows = []
    for _il in INCL_LABELS:
        # (n_gal, n_ap) cumulative flux matrices, one catalog read per (arm, rung)
        _M, _miss = {}, []
        for _arm in ("dust_on", "dust_off"):
            for _k, _lab in enumerate(APERTURE_LABELS):
                _f = os.path.join(CATDIR, f"catalog_{_arm}_{_lab}_{_il}.fits")
                if not os.path.exists(_f):
                    _miss.append(os.path.basename(_f))
                    continue
                _t = Table.read(_f)
                _idx = {(int(s), int(g)): _j
                        for _j, (s, g) in enumerate(zip(_t["snap"], _t["gal_id"]))}
                for _b, _col in _AV_BANDS.items():
                    _mat = _M.setdefault((_arm, _b),
                                         np.full((len(_keys), N_AP), np.nan))
                    if _col not in _t.colnames:
                        continue
                    _v = np.asarray(_t[_col], float)
                    for _i, _key in enumerate(_keys):
                        _j = _idx.get(_key)
                        if _j is not None:
                            _mat[_i, _k] = _v[_j]
        if _miss:
            print(f"[{_il}] {len(_miss)} catalogs missing (e.g. {_miss[0]}) -> partial")
        if not _M:
            continue
        # cumulative -> annular; column 0 (the 0->1 kpc disc) IS ann1kpc == ap1kpc
        _D = {k: np.column_stack([v[:, :1], np.diff(v, axis=1)]) for k, v in _M.items()}
        for _which, _F in (("ap", _M), ("ann", _D)):
            for _k, _lab in enumerate(APERTURE_LABELS):
                if _which == "ann" and _k == 0:
                    continue                       # ann1kpc == ap1kpc: keep one copy
                _lab_out = _lab if _which == "ap" else ANNULUS_LABELS[_k]
                _fon  = {b: _F[("dust_on", b)][:, _k] for b in _AV_BANDS}
                _foff = {b: _F[("dust_off", b)][:, _k] for b in _AV_BANDS}
                with np.errstate(all="ignore"):
                    _uv_on  = np.where((_fon["U"] > 0) & (_fon["V"] > 0),
                                       -2.5 * np.log10(_fon["U"] / _fon["V"]), np.nan)
                    _uv_off = np.where((_foff["U"] > 0) & (_foff["V"] > 0),
                                       -2.5 * np.log10(_foff["U"] / _foff["V"]), np.nan)
                _A = {b: attenuation_mag(_fon[b], _foff[b]) for b in _AV_BANDS}
                for _i, (_sn, _gd) in enumerate(_keys):
                    _rows.append(dict(snap=_sn, gal_id=_gd, incl=_il, aperture=_lab_out,
                                      A_U=float(_A["U"][_i]), A_V=float(_A["V"][_i]),
                                      A_J=float(_A["J"][_i]),
                                      S_UV=float(_A["U"][_i] - _A["V"][_i]),
                                      UV_on=float(_uv_on[_i]), UV_off=float(_uv_off[_i]),
                                      F_V_on=float(_fon["V"][_i]),
                                      F_V_off=float(_foff["V"][_i])))
    if not _rows:
        raise FileNotFoundError(f"no dust_on/dust_off catalogs in {CATDIR}; run Part 7 first")
    ANNULUS_AV = Table(rows=_rows)
    ANNULUS_AV.meta["R_EDGES"] = list(np.round(R_EDGES, 3))
    ANNULUS_AV.write(ANNULUS_AV_FITS, overwrite=True)
    print(f"{len(ANNULUS_AV)} rows -> {ANNULUS_AV_FITS}")

# ── closure: the fiducial sightline must reproduce Part 7a's stored A_V_ann_* ──
_avf = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")
if os.path.exists(_avf):
    _at = Table.read(_avf)
    _fid = ANNULUS_AV[np.char.strip(np.asarray(ANNULUS_AV["incl"], str)) == INCL_LABELS[0]]
    _mine = {(int(_r["snap"]), int(_r["gal_id"]), str(_r["aperture"]).strip()):
             float(_r["A_V"]) for _r in _fid}
    _dmax, _ncmp = 0.0, 0
    for _r in _at:
        for _k, _lab in enumerate(APERTURE_LABELS):
            if f"A_V_ann_{_lab}" not in _at.colnames:
                continue
            _lab_o = _lab if _k == 0 else ANNULUS_LABELS[_k]     # 7a names by the rung
            _v7a = float(_r[f"A_V_ann_{_lab}"])
            _v8b = _mine.get((int(_r["snap"]), int(_r["gal_id"]), _lab_o), np.nan)
            if np.isfinite(_v7a) and np.isfinite(_v8b):
                _dmax, _ncmp = max(_dmax, abs(_v7a - _v8b)), _ncmp + 1
    assert _ncmp == 0 or _dmax < 1e-6, \
        f"8b drifted from Part 7a's annular A_V (max |dA_V| = {_dmax:.2e} over {_ncmp})"
    print(f"closure vs Part 7a ({INCL_LABELS[0]}): max |dA_V_ann| = {_dmax:.2e} "
          f"over {_ncmp} annuli — OK")

# NaN'd annuli (non-positive differential flux = MC noise / empty annulus) are a
# mild TRANSPARENCY selection: report per label so T2 can quote it
_ap8b = np.char.strip(np.asarray(ANNULUS_AV["aperture"], str))
print(f"\n{'label':>10s} {'A_V NaN':>8s} {'(U-V)_on NaN':>13s}")
for _lab in list(APERTURE_LABELS) + ANNULUS_LABELS[1:]:
    _t = ANNULUS_AV[_ap8b == _lab]
    if len(_t):
        print(f"{_lab:>10s} {np.mean(~np.isfinite(np.asarray(_t['A_V'], float))) * 100:7.0f}% "
              f"{np.mean(~np.isfinite(np.asarray(_t['UV_on'], float))) * 100:12.0f}%")

In [ ]:
# ── Part 8c — master red-core table + T1: is the red core dust at all? ──
# Pure read of the 8a/8b caches + selection metadata. The dust_off colour profile
# is the INTRINSIC (age/Z) gradient; dust_on - dust_off is the dust contribution.
# f_dust_color = the fraction of the central (U-V) excess that vanishes without dust.
from scipy.stats import mannwhitneyu

NGAS_ANN_MIN  = 10          # Sigma/DGR/f_cold need >= this many gas particles
N_BOOT        = 2000        # galaxy-level bootstrap resamples
CORE_LABEL    = "ap3kpc"    # headline core: the 0-3.16 kpc disc (best sampled)
CORE_ECHO     = "ap1kpc"    # echo: the innermost disc (~3-4 softenings, noisy)
OUT_REF_LABEL = "ann32kpc"  # outer reference annulus (10-31.6 kpc)
RED_CORE_MIN  = 0.05        # mag; a 'red core' must exceed this central excess
AGN_COLORS = {"strong": "#c0392b", "intermediate": "#e67e22", "weak": "#2980b9",
              "no_AGN": "#27ae60", "no_event": "#7f8c8d", "unclassified": "#bdc3c7"}
CLS4 = ["strong", "intermediate", "weak", "no_AGN"]

_A8 = Table.read(os.path.join(TABLEDIR, "annulus_ism_truth.fits"))
_B8 = Table.read(os.path.join(TABLEDIR, "annulus_av_allincl.fits"))
for _t in (_A8, _B8):
    for _c in ("incl", "aperture"):
        _t[_c] = np.char.strip(np.asarray(_t[_c], str))
P8 = join(_A8, _B8, keys=("snap", "gal_id", "incl", "aperture"),
          metadata_conflicts="silent")
print(f"join on (snap, gal_id, incl, aperture): {len(P8)} rows "
      f"(ISM {len(_A8)}, A_V {len(_B8)})"
      + ("" if len(P8) == min(len(_A8), len(_B8)) else "  [rows lost — check labels]"))

# galaxy metadata (class labels, mass, size flags) from the selection catalog
SEL, _, _ = load_selection()
_md = {(int(_r["snap"]), int(_r["gal_id"])): _r for _r in SEL}
_pk = list(zip(np.asarray(P8["snap"], int), np.asarray(P8["gal_id"], int)))

def _selnum(row, col):
    # tolerate masked values and columns absent from older selection builds
    if col not in row.colnames:
        return np.nan
    _v = row[col]
    return np.nan if np.ma.is_masked(_v) else float(_v)

_want = ("log_mstar", "xstr_quench", "r50_star_kpc", "flag_too_large")
_absent = [c for c in _want if c not in SEL.colnames]
if _absent:
    print(f"[meta] selection FITS lacks {_absent} -> NaN "
          "(Part 3b was not re-run after the last selection rebuild)")
for _c in _want:
    P8[_c] = np.array([_selnum(_md[_k], _c) if _k in _md else np.nan for _k in _pk])
P8["agn_class"] = np.array(
    [(lambda a: a.decode() if isinstance(a, (bytes, np.bytes_)) else str(a))
     (_md[_k]["agn_class"]) if _k in _md else "unclassified" for _k in _pk])
_z_of_snap = {int(s): float(sim.get_z_from_snap(int(s))) for s in np.unique(P8["snap"])}
P8["z_target"] = np.array([_z_of_snap[int(s)] for s in P8["snap"]])

# QC mask: surface densities and ratios are shot-noise garbage below NGAS_ANN_MIN
_lowN = np.asarray(P8["ngas"], int) < NGAS_ANN_MIN
for _c in ("Sigma_dust", "Sigma_gas", "Sigma_HI", "Sigma_H2", "DGR", "f_cold", "f_mol_ann"):
    _v = np.asarray(P8[_c], float)
    _v[_lowN] = np.nan
    P8[_c] = _v
_ap8 = np.asarray(P8["aperture"], str)
print(f"ngas < {NGAS_ANN_MIN}: Sigma/DGR/f_cold masked on {int(_lowN.sum())}/{len(P8)} rows — "
      + "  ".join(f"{_l}:{np.mean(_lowN[_ap8 == _l]) * 100:.0f}%"
                  for _l in ["ap1kpc"] + ANNULUS_LABELS[1:]))

# ── shared inference helpers (used by 8c-8g) ──
_GKEY = np.array([f"{s}_{g}" for s, g in zip(P8["snap"], P8["gal_id"])])

def _gboot_idx(keys, statfn, n=N_BOOT, seed=0):
    """Cluster bootstrap over GALAXIES: statfn(row_indices) -> (point, lo16, hi84).
    All sightline rows of a resampled galaxy ride along together."""
    keys = np.asarray(keys)
    _uk = np.unique(keys)
    _where = {k: np.where(keys == k)[0] for k in _uk}
    _rng = np.random.default_rng(seed)
    _pt = statfn(np.arange(len(keys)))
    _s = np.array([statfn(np.concatenate(
        [_where[k] for k in _rng.choice(_uk, len(_uk), replace=True)])) for _ in range(n)])
    return float(_pt), float(np.nanpercentile(_s, 16)), float(np.nanpercentile(_s, 84))

def _gboot_med(vals, keys, **kw):
    vals = np.asarray(vals, float)
    return _gboot_idx(keys, lambda i: (np.nanmedian(vals[i])
                                       if np.isfinite(vals[i]).any() else np.nan), **kw)

def _gal_median(vals, keys):
    """Per-galaxy median over sightlines -> (per-gal values, per-gal keys)."""
    vals, keys = np.asarray(vals, float), np.asarray(keys)
    _uk = np.unique(keys)
    return np.array([np.nanmedian(vals[keys == k]) for k in _uk]), _uk

def _cls_of_keys(keys):
    _c = {k: c for k, c in zip(_GKEY, P8["agn_class"])}
    return np.array([_c[k] for k in keys])

# ── T1: decompose the central colour excess into dust vs intrinsic ──
def _lab_map(col, lab):
    _m = _ap8 == lab
    return {(k, str(i)): float(v)
            for k, i, v in zip(_GKEY[_m], P8["incl"][_m], np.asarray(P8[col], float)[_m])}

_T1 = {}
for _core in (CORE_LABEL, CORE_ECHO):
    _con, _cof = _lab_map("UV_on", _core), _lab_map("UV_off", _core)
    _oon, _oof = _lab_map("UV_on", OUT_REF_LABEL), _lab_map("UV_off", OUT_REF_LABEL)
    _rows = []
    for _k in _con:
        _dt = _con[_k] - _oon.get(_k, np.nan)          # total central excess (dusty view)
        _di = _cof[_k] - _oof.get(_k, np.nan)          # intrinsic (age/Z) part
        _rows.append((_k[0], _dt, _dt - _di,
                      (_dt - _di) / _dt if (np.isfinite(_dt) and _dt > RED_CORE_MIN) else np.nan))
    _gk = np.array([r[0] for r in _rows])
    _T1[_core] = dict(gkey=_gk,
                      D_tot=np.array([r[1] for r in _rows]),
                      D_dust=np.array([r[2] for r in _rows]),
                      f_dust=np.array([r[3] for r in _rows]),
                      cls=_cls_of_keys(_gk))

_t1 = _T1[CORE_LABEL]
_nred = int(np.isfinite(_t1["f_dust"]).sum())
print(f"\nT1 [{CORE_LABEL} - {OUT_REF_LABEL}]: {_nred}/{len(_t1['f_dust'])} "
      f"(gal x sightline) rows have a red core (D_tot > {RED_CORE_MIN:g} mag)")
print(f"{'class':>14s} {'n_gal':>5s} {'f_dust_color [16-84]':>24s} {'D_tot med':>10s}")
_fd_gal, _fd_keys = _gal_median(_t1["f_dust"], _t1["gkey"])
_fd_cls = _cls_of_keys(_fd_keys)
for _cl in CLS4 + [c for c in set(_t1["cls"]) if c not in CLS4]:
    _m = _t1["cls"] == _cl
    if not _m.any():
        continue
    _v, _lo, _hi = _gboot_med(_t1["f_dust"][_m], _t1["gkey"][_m])
    print(f"{_cl:>14s} {len(set(_t1['gkey'][_m])):5d} {_v:12.2f} [{_lo:+.2f},{_hi:+.2f}] "
          f"{np.nanmedian(_t1['D_tot'][_m]):10.2f}")
_a, _b = (_fd_gal[(_fd_cls == c) & np.isfinite(_fd_gal)] for c in ("strong", "no_AGN"))
if len(_a) >= 3 and len(_b) >= 3:
    _u, _p = mannwhitneyu(_a, _b, alternative="two-sided")
    print(f"Mann-Whitney f_dust_color strong vs no_AGN (per-galaxy medians): p = {_p:.3f}")

# ── figure: colour profiles, dust contribution, decomposition by class ──
_PROF_LABS = ["ap1kpc"] + ANNULUS_LABELS[1:]          # radial sequence of annuli
_R_MID = np.where(R_EDGES[:-1] > 0, np.sqrt(R_EDGES[:-1] * R_EDGES[1:]), R_EDGES[1:] / 2.0)
_cls_present = [c for c in CLS4 + ["no_event", "unclassified"] if c in set(P8["agn_class"])]

def _prof_matrix(col):
    """(n_gal*n_incl, n_annuli) matrix of `col` along the radial annulus sequence."""
    _maps = [_lab_map(col, _l) for _l in _PROF_LABS]
    _rk = sorted(set().union(*[set(m) for m in _maps]))
    return (np.array([[m.get(k, np.nan) for m in _maps] for k in _rk]),
            np.array([k[0] for k in _rk]))

_UVon_M, _pk_on = _prof_matrix("UV_on")
_UVoff_M, _ = _prof_matrix("UV_off")
_pcls = _cls_of_keys(_pk_on)
fig, axs = plt.subplots(1, 3, figsize=(21, 6.5))
for _row in _UVon_M:                                   # context spaghetti
    axs[0].plot(_R_MID, _row, color="0.85", lw=0.5, alpha=0.5, zorder=1)
for _cl in _cls_present:
    _m = _pcls == _cl
    if _m.sum() < 4:
        continue
    axs[0].plot(_R_MID, np.nanmedian(_UVon_M[_m], axis=0), "o-",
                color=AGN_COLORS[_cl], lw=2, zorder=3, label=_cl)
    axs[0].plot(_R_MID, np.nanmedian(_UVoff_M[_m], axis=0), "--",
                color=AGN_COLORS[_cl], lw=1.4, zorder=2)
axs[0].set_xscale("log")
axs[0].set_xlabel("radius [pkpc]")
axs[0].set_ylabel(r"$(U-V)$ [mag]  (solid: dust_on, dashed: dust_off)")
axs[0].legend(fontsize=10, frameon=False)
_DUV = _UVon_M - _UVoff_M                              # dust contribution profile
for _cl in _cls_present:
    _m = _pcls == _cl
    if _m.sum() < 4:
        continue
    _med = np.nanmedian(_DUV[_m], axis=0)
    _ci = np.array([_gboot_med(_DUV[_m][:, _j], _pk_on[_m], n=400)[1:]
                    for _j in range(len(_PROF_LABS))])
    axs[1].plot(_R_MID, _med, "o-", color=AGN_COLORS[_cl], lw=2, label=_cl)
    axs[1].fill_between(_R_MID, _ci[:, 0], _ci[:, 1], color=AGN_COLORS[_cl], alpha=0.15)
axs[1].axhline(0, color="0.5", lw=0.8, ls=":")
axs[1].set_xscale("log")
axs[1].set_xlabel("radius [pkpc]")
axs[1].set_ylabel(r"$\Delta(U-V)_{\rm dust}$ [mag]")
axs[1].legend(fontsize=10, frameon=False)
for _i, _cl in enumerate(_cls_present):                # decomposition strip
    _m = (_fd_cls == _cl) & np.isfinite(_fd_gal)
    if not _m.any():
        continue
    _xj = _i + np.random.default_rng(_i).uniform(-0.16, 0.16, int(_m.sum()))
    axs[2].scatter(_xj, _fd_gal[_m], c=AGN_COLORS[_cl], edgecolor="k",
                   linewidth=0.3, s=34)
    axs[2].hlines(np.nanmedian(_fd_gal[_m]), _i - 0.3, _i + 0.3, color="k", lw=2)
for _y, _lab in ((0, "no dust"), (1, "all dust")):
    axs[2].axhline(_y, color="0.6", lw=0.8, ls="--")
    axs[2].text(len(_cls_present) - 0.4, _y + 0.03, _lab, fontsize=8, color="0.4")
axs[2].set_xticks(range(len(_cls_present)))
axs[2].set_xticklabels(_cls_present, rotation=30, ha="right", fontsize=10)
axs[2].set_ylabel(r"$f_{\rm dust}$ of the central $(U-V)$ excess")
fig.tight_layout()
_f = os.path.join(PLOTDIR, "p8_t1_color_decomposition.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)

In [ ]:
# ── Part 8d — T2: does the surviving dust column carry the attenuation? ──
# Self-contained after Part 8c (P8 in memory). The foreground screen
#     A_V,screen = 1.086 * kappa_V * Sigma_dust
# is the CEILING: stars mixed with (or in front of) the dust attenuate less per
# unit column, so 'surviving ISM dust suffices' predicts a tight A_V-Sigma_dust
# correlation with measured/screen in ~[0.1, 1]. Ratios > 1 falsify it locally.
from scipy.stats import spearmanr

MSUN_KPC2_TO_G_CM2 = 2.089e-10        # 1 Msun/kpc^2 = 2.089e-10 g/cm^2
_ism_meta = Table.read(os.path.join(TABLEDIR, "annulus_ism_truth.fits")).meta
KAPPA = {b: float(_ism_meta[f"KAPPA_{b}"]) for b in ("U", "V", "J")}
print(f"screen opacities [{_ism_meta.get('KAP_SRC', '?')}]: "
      + "  ".join(f"kappa_{b}={v:.4g} cm2/g" for b, v in KAPPA.items()))

_sig = np.asarray(P8["Sigma_dust"], float)
_avv = np.asarray(P8["A_V"], float)
P8["A_V_screen"] = 1.086 * KAPPA["V"] * _sig * MSUN_KPC2_TO_G_CM2
with np.errstate(all="ignore"):
    P8["screen_ratio"] = _avv / np.asarray(P8["A_V_screen"], float)

# convention guard: a per-GAS-mass chi would shift the ratio by ~x150
_ok0 = np.isfinite(np.asarray(P8["screen_ratio"], float)) & (_sig > 0) & (_avv > 0.02)
_rmed = float(np.nanmedian(np.asarray(P8["screen_ratio"], float)[_ok0]))
print(f"median measured/screen (A_V > 0.02 rows): {_rmed:.3g}")
assert 1e-3 < _rmed < 1e3, \
    f"screen ratio {_rmed:.2g} out of range — kappa convention broken (per gas vs per dust?)"

# ── per-annulus correlation + ratio (galaxy-bootstrap CIs) ──
_T2_LABS = ["ap1kpc"] + ANNULUS_LABELS[1:]

def _rho_stat(idx, x, y):
    _f = np.isfinite(x[idx]) & np.isfinite(y[idx])
    return spearmanr(x[idx][_f], y[idx][_f])[0] if _f.sum() >= 5 else np.nan

print(f"\n{'annulus':>10s} {'n':>4s} {'rho(A_V, Sig_dust) [16-84]':>28s} "
      f"{'med ratio [16-84]':>22s} {'>1.2':>5s}")
_t2_sum = {}
for _lab in _T2_LABS:
    _m = (_ap8 == _lab) & np.isfinite(_sig) & np.isfinite(_avv)
    if _m.sum() < 8:
        print(f"{_lab:>10s} {int(_m.sum()):4d}  — too few finite rows")
        continue
    _x, _y, _k = _sig[_m], _avv[_m], _GKEY[_m]
    _r, _rlo, _rhi = _gboot_idx(_k, lambda i: _rho_stat(i, _x, _y))
    _rat = np.asarray(P8["screen_ratio"], float)[_m]
    _q, _qlo, _qhi = _gboot_med(_rat, _k)
    _t2_sum[_lab] = (_r, _q)
    print(f"{_lab:>10s} {int(_m.sum()):4d} {_r:+12.2f} [{_rlo:+.2f},{_rhi:+.2f}] "
          f"{_q:10.3g} [{_qlo:.2g},{_qhi:.2g}] {int(np.nansum(_rat > 1.2)):5d}")

# screen-slope consistency: the SAME dust column must predict A_U/A_V and A_J/A_V
print("\nscreen-slope check (median measured vs KMH94 screen; agreement = "
      "geometry, not opacity, sets the ratio):")
for _b in ("U", "J"):
    _ab = np.asarray(P8[f"A_{_b}"], float)
    _f = np.isfinite(_ab) & np.isfinite(_avv) & (_avv > 0.05)
    if _f.sum() >= 8:
        print(f"   A_{_b}/A_V: measured {np.nanmedian(_ab[_f] / _avv[_f]):.2f}   "
              f"screen {KAPPA[_b] / KAPPA['V']:.2f}")

# falsifiers: attenuation the local dust column cannot supply even as a screen
_bad = _ok0 & (np.asarray(P8["screen_ratio"], float) > 1.2)
if _bad.any():
    print(f"\n[falsifiers] {int(_bad.sum())} rows with A_V > 1.2x the screen ceiling — "
          "per class: " + "  ".join(f"{c}:{int((_bad & (P8['agn_class'] == c)).sum())}"
                                    for c in CLS4 if (_bad & (P8["agn_class"] == c)).any()))
else:
    print("\n[falsifiers] no rows exceed 1.2x the screen ceiling — the surviving "
          "column is always sufficient")

# ── figure: A_V vs Sigma_dust per annulus + the ratio profile ──
fig, axs = plt.subplots(2, 3, figsize=(21, 12), sharey=False)
_sgrid = np.logspace(2.5, 8, 60)
for _axi, _lab in zip(axs.flat[:len(_T2_LABS)], _T2_LABS):
    _m = (_ap8 == _lab) & np.isfinite(_sig) & np.isfinite(_avv) & (_sig > 0)
    for _cl in _cls_present:
        _s = _m & (P8["agn_class"] == _cl)
        _axi.scatter(_sig[_s], _avv[_s], s=22, c=AGN_COLORS[_cl], alpha=0.55,
                     edgecolor="k", linewidth=0.25, label=_cl)
    _scr = 1.086 * KAPPA["V"] * _sgrid * MSUN_KPC2_TO_G_CM2
    _axi.plot(_sgrid, _scr, "k-", lw=1.4)
    _axi.fill_between(_sgrid, 0.1 * _scr, _scr, color="0.5", alpha=0.15)
    if _lab in _t2_sum:
        _axi.text(0.04, 0.95, f"$\\rho$={_t2_sum[_lab][0]:+.2f}\n"
                  f"ratio={_t2_sum[_lab][1]:.2g}", transform=_axi.transAxes,
                  va="top", fontsize=9,
                  bbox=dict(fc="white", ec="0.7", alpha=0.85, pad=1.6))
    _axi.set_xscale("log")
    _axi.set_yscale("log")
    _axi.set_xlim(3e2, 1e8)
    _axi.set_ylim(1e-3, 5)
    _axi.set_title(_lab, fontsize=11)
    _axi.set_xlabel(r"$\Sigma_{\rm dust}$ [M$_\odot$ kpc$^{-2}$]")
    _axi.set_ylabel(r"$A_V^{\rm ann}$ [mag]")
axs.flat[0].legend(fontsize=9, loc="lower right", framealpha=0.9)
_axr = axs.flat[-1]                                    # summary: ratio vs radius
_R_MID8 = np.where(R_EDGES[:-1] > 0, np.sqrt(R_EDGES[:-1] * R_EDGES[1:]), R_EDGES[1:] / 2.0)
for _cl in _cls_present:
    _med = []
    for _lab in _T2_LABS:
        _s = (_ap8 == _lab) & (P8["agn_class"] == _cl) & _ok0
        _med.append(np.nanmedian(np.asarray(P8["screen_ratio"], float)[_s])
                    if _s.sum() >= 4 else np.nan)
    _axr.plot(_R_MID8, _med, "o-", color=AGN_COLORS[_cl], lw=2, label=_cl)
_axr.axhline(1.0, color="k", lw=1.2)
_axr.axhspan(0.1, 1.0, color="0.5", alpha=0.15)
_axr.set_xscale("log")
_axr.set_yscale("log")
_axr.set_xlabel("radius [pkpc]")
_axr.set_ylabel("measured / screen $A_V$")
_axr.legend(fontsize=9, frameon=False)
fig.tight_layout()
_f = os.path.join(PLOTDIR, "p8_t2_av_vs_sigma_dust.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)

In [ ]:
# ── Part 8e — T3: is the attenuating dust in COLD surviving ISM? ──
# Self-contained after Part 8c. Raw Spearman says the phases correlate with A_V;
# the rank-partial at fixed (log M*, log Sigma_dust) is the discriminating one —
# if f_cold / Sigma_H2 lose all signal at fixed Sigma_dust, the 'carrier' is just
# 'where the dust is' and the cold phase adds nothing beyond the column.
from scipy.stats import spearmanr, rankdata, pearsonr

def _partial_spearman(y, x, C):
    """Rank partial correlation (no statsmodels on the cluster kernel):
    rank-transform y, x and every control column, residualise the ranks of y and
    x on the controls via lstsq, Pearson on the residuals. -> (rho, used_mask)"""
    ok = np.isfinite(y) & np.isfinite(x) & np.all(np.isfinite(C), axis=1)
    if ok.sum() < 10:
        return np.nan, ok
    RC = np.column_stack([np.ones(int(ok.sum()))] + [rankdata(c) for c in C[ok].T])
    ey = rankdata(y[ok]) - RC @ np.linalg.lstsq(RC, rankdata(y[ok]), rcond=None)[0]
    ex = rankdata(x[ok]) - RC @ np.linalg.lstsq(RC, rankdata(x[ok]), rcond=None)[0]
    return float(pearsonr(ey, ex)[0]), ok

_T3_X = [("f_cold", r"$f_{\rm cold}$", False),
         ("Sigma_H2", r"$\Sigma_{H_2}$", True),
         ("Sigma_HI", r"$\Sigma_{HI}$", True),
         ("DGR", "DGR", True)]

print(f"{'label':>8s} {'x':>10s} {'n':>4s} {'raw rho [16-84]':>18s} "
      f"{'partial | M*,Sig_d [16-84]':>28s}")
_t3_resid = {}
for _lab in (CORE_LABEL, "ann3kpc"):
    _m = _ap8 == _lab
    _y = np.asarray(P8["A_V"], float)[_m]
    _k = _GKEY[_m]
    with np.errstate(all="ignore"):
        _C = np.column_stack([np.asarray(P8["log_mstar"], float)[_m],
                              np.log10(np.asarray(P8["Sigma_dust"], float)[_m])])
    for _xc, _xl, _xlog in _T3_X:
        with np.errstate(all="ignore"):
            _x = np.asarray(P8[_xc], float)[_m]
            _xr = np.log10(_x) if _xlog else _x
        _f = np.isfinite(_y) & np.isfinite(_xr)
        if _f.sum() < 10:
            continue
        _r0, _lo0, _hi0 = _gboot_idx(_k, lambda i: (
            spearmanr(_y[i][np.isfinite(_y[i]) & np.isfinite(_xr[i])],
                      _xr[i][np.isfinite(_y[i]) & np.isfinite(_xr[i])])[0]
            if (np.isfinite(_y[i]) & np.isfinite(_xr[i])).sum() >= 5 else np.nan))
        _rp, _lop, _hip = _gboot_idx(_k, lambda i: _partial_spearman(
            _y[i], _xr[i], _C[i])[0])
        print(f"{_lab:>8s} {_xc:>10s} {int(_f.sum()):4d} "
              f"{_r0:+7.2f} [{_lo0:+.2f},{_hi0:+.2f}] "
              f"{_rp:+9.2f} [{_lop:+.2f},{_hip:+.2f}]")
        if _lab == CORE_LABEL:                 # keep the residuals for the figure
            _rr, _okr = _partial_spearman(_y, _xr, _C)
            RC = np.column_stack([np.ones(int(_okr.sum()))]
                                 + [rankdata(c) for c in _C[_okr].T])
            _t3_resid[_xc] = (
                rankdata(_xr[_okr]) - RC @ np.linalg.lstsq(
                    RC, rankdata(_xr[_okr]), rcond=None)[0],
                rankdata(_y[_okr]) - RC @ np.linalg.lstsq(
                    RC, rankdata(_y[_okr]), rcond=None)[0],
                np.asarray(P8["agn_class"])[_m][_okr], _rr)

# ── figure: raw (top) and rank-residual (bottom) views of the core ──
_mC = _ap8 == CORE_LABEL
_yC = np.asarray(P8["A_V"], float)[_mC]
fig, axs = plt.subplots(2, 4, figsize=(24, 11))
for _j, (_xc, _xl, _xlog) in enumerate(_T3_X):
    _ax = axs[0][_j]
    with np.errstate(all="ignore"):
        _x = np.asarray(P8[_xc], float)[_mC]
    for _cl in _cls_present:
        _s = np.asarray(P8["agn_class"])[_mC] == _cl
        _ax.scatter(_x[_s], _yC[_s], s=24, c=AGN_COLORS[_cl], alpha=0.6,
                    edgecolor="k", linewidth=0.25, label=_cl if _j == 0 else None)
    if _xlog:
        _ax.set_xscale("log")
    _ax.set_xlabel(_xl)
    _ax.set_ylabel(rf"$A_V$({CORE_LABEL}) [mag]")
    _ax = axs[1][_j]
    if _xc in _t3_resid:
        _ex, _ey, _cls_r, _rr = _t3_resid[_xc]
        for _cl in _cls_present:
            _s = _cls_r == _cl
            _ax.scatter(_ex[_s], _ey[_s], s=24, c=AGN_COLORS[_cl], alpha=0.6,
                        edgecolor="k", linewidth=0.25)
        _ax.text(0.04, 0.95, f"$\\rho_{{\\rm partial}}$={_rr:+.2f}",
                 transform=_ax.transAxes, va="top", fontsize=9,
                 bbox=dict(fc="white", ec="0.7", alpha=0.85, pad=1.6))
    _ax.axhline(0, color="0.6", lw=0.8, ls=":")
    _ax.axvline(0, color="0.6", lw=0.8, ls=":")
    _ax.set_xlabel(f"rank {_xl} | $\\log M_\\star$, $\\log\\Sigma_{{\\rm dust}}$")
    _ax.set_ylabel(r"rank $A_V$ | controls")
axs[0][0].legend(fontsize=9, loc="upper left", framealpha=0.9)
fig.tight_layout()
_f = os.path.join(PLOTDIR, "p8_t3_av_vs_cold_ism.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)

In [ ]:
# ── Part 8f — T4: do the AGN coupling classes differ, and HOW? ──
# Self-contained after Parts 8c/8d. Two different questions:
#   (1) amounts — central A_V / Sigma_dust / DGR / f_cold distributions by class;
#   (2) geometry — A_V residuals at FIXED Sigma_dust (Theil-Sen fit): a class
#       offset here means coupling changes clumpiness/covering, not dust mass.
# All rank tests run on per-galaxy medians over sightlines (kills the x4
# pseudo-replication); anchors are pooled — N is too small for sub-binning.
from scipy.stats import mannwhitneyu, spearmanr, theilslopes

_T4_Q = [("A_V", r"$A_V$ [mag]", False),
         ("Sigma_dust", r"$\Sigma_{\rm dust}$ [M$_\odot$ kpc$^{-2}$]", True),
         ("DGR", "DGR", True),
         ("f_cold", r"$f_{\rm cold}$", False)]
_mC4 = _ap8 == CORE_LABEL

# explicit small-N ledger: this is the resolution limit of every claim below
print(f"galaxies per class per anchor ({CORE_LABEL}):")
_snaps_u = sorted(set(np.asarray(P8["snap"], int)))
print(f"{'class':>14s} " + " ".join(f"s{_s:03d}" for _s in _snaps_u) + "  total")
for _cl in CLS4:
    _n = [len(set(_GKEY[_mC4 & (P8["agn_class"] == _cl)
                        & (np.asarray(P8["snap"], int) == _s)])) for _s in _snaps_u]
    print(f"{_cl:>14s} " + " ".join(f"{v:4d}" for v in _n) + f"  {sum(_n):5d}")

# ── (1) class distributions of the central quantities ──
print(f"\n{'quantity':>11s} {'class':>14s} {'n':>3s} {'median [16-84 boot]':>22s}")
_t4_gal = {}
for _qc, _ql, _qlog in _T4_Q:
    _v, _k = _gal_median(np.asarray(P8[_qc], float)[_mC4], _GKEY[_mC4])
    _c = _cls_of_keys(_k)
    _t4_gal[_qc] = (_v, _k, _c)
    for _cl in CLS4:
        _m = (_c == _cl) & np.isfinite(_v)
        if not _m.any():
            continue
        _pt, _lo, _hi = _gboot_med(_v[_m], _k[_m])
        print(f"{_qc:>11s} {_cl:>14s} {int(_m.sum()):3d} "
              f"{_pt:10.3g} [{_lo:.3g},{_hi:.3g}]")
    for _c1, _c2 in (("strong", "no_AGN"), ("strong", "weak")):
        _a = _v[(_c == _c1) & np.isfinite(_v)]
        _b = _v[(_c == _c2) & np.isfinite(_v)]
        if len(_a) >= 3 and len(_b) >= 3:
            _u, _p = mannwhitneyu(_a, _b, alternative="two-sided")
            print(f"{'':>11s} MW {_c1} vs {_c2}: p = {_p:.3f}   "
                  f"d(median) = {np.median(_a) - np.median(_b):+.3g}")

# ── (2) geometry: A_V at fixed Sigma_dust (pooled Theil-Sen, residuals by class) ──
with np.errstate(all="ignore"):
    _lsd = np.log10(np.asarray(P8["Sigma_dust"], float)[_mC4])
_av4 = np.asarray(P8["A_V"], float)[_mC4]
_ok4 = np.isfinite(_lsd) & np.isfinite(_av4)
_ts = theilslopes(_av4[_ok4], _lsd[_ok4])
_resid = np.full(len(_av4), np.nan)
_resid[_ok4] = _av4[_ok4] - (_ts[1] + _ts[0] * _lsd[_ok4])
print(f"\nTheil-Sen A_V vs log Sigma_dust ({CORE_LABEL}, n={int(_ok4.sum())}): "
      f"slope = {_ts[0]:+.3f} mag/dex, intercept = {_ts[1]:+.3f}")
_rv, _rk = _gal_median(_resid, _GKEY[_mC4])
_rc = _cls_of_keys(_rk)
for _cl in CLS4:
    _m = (_rc == _cl) & np.isfinite(_rv)
    if _m.any():
        _pt, _lo, _hi = _gboot_med(_rv[_m], _rk[_m])
        print(f"   resid {_cl:>14s}: {_pt:+.3f} [{_lo:+.3f},{_hi:+.3f}]  (n={int(_m.sum())})")
_a = _rv[(_rc == "strong") & np.isfinite(_rv)]
_b = _rv[(_rc == "no_AGN") & np.isfinite(_rv)]
if len(_a) >= 3 and len(_b) >= 3:
    print(f"   MW resid strong vs no_AGN: p = "
          f"{mannwhitneyu(_a, _b, alternative='two-sided')[1]:.3f} — a class offset "
          "at fixed column = geometry/clumpiness, not dust amount")

# ── (3) continuous: everything vs the coupling strength xstr_quench ──
_xsg, _xsk = _gal_median(np.asarray(P8["xstr_quench"], float)[_mC4], _GKEY[_mC4])
_xs_of = dict(zip(_xsk, _xsg))
print("\nSpearman vs xstr_quench (per-galaxy medians):")
for _qc in [q[0] for q in _T4_Q] + ["screen_ratio"]:
    _v, _k = _gal_median(np.asarray(P8[_qc], float)[_mC4], _GKEY[_mC4])
    _x = np.array([_xs_of.get(k, np.nan) for k in _k])
    _f = np.isfinite(_v) & np.isfinite(_x)
    if _f.sum() >= 8:
        _r, _p = spearmanr(_x[_f], _v[_f])
        print(f"   {_qc:>12s}: rho = {_r:+.2f}  (p = {_p:.3f}, n = {int(_f.sum())})")

# ── figures ──
fig, axs = plt.subplots(1, 4, figsize=(24, 6.2))
for _ax, (_qc, _ql, _qlog) in zip(axs, _T4_Q):
    _v, _k, _c = _t4_gal[_qc]
    for _i, _cl in enumerate(CLS4):
        _m = (_c == _cl) & np.isfinite(_v)
        if not _m.any():
            continue
        _xj = _i + np.random.default_rng(_i).uniform(-0.16, 0.16, int(_m.sum()))
        _ax.scatter(_xj, _v[_m], c=AGN_COLORS[_cl], edgecolor="k", linewidth=0.3, s=34)
        _ax.hlines(np.nanmedian(_v[_m]), _i - 0.3, _i + 0.3, color="k", lw=2)
    if _qlog:
        _ax.set_yscale("log")
    _ax.set_xticks(range(len(CLS4)))
    _ax.set_xticklabels(CLS4, rotation=30, ha="right", fontsize=10)
    _ax.set_ylabel(_ql)
fig.suptitle(f"central ({CORE_LABEL}) ISM by coupling class — per-galaxy medians", y=1.0)
fig.tight_layout()
_f = os.path.join(PLOTDIR, "p8_t4_class_distributions.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)

fig, axs = plt.subplots(1, 2, figsize=(16, 6.5))
for _i, _cl in enumerate(CLS4):                       # residual strips
    _m = (_rc == _cl) & np.isfinite(_rv)
    if not _m.any():
        continue
    _xj = _i + np.random.default_rng(_i).uniform(-0.16, 0.16, int(_m.sum()))
    axs[0].scatter(_xj, _rv[_m], c=AGN_COLORS[_cl], edgecolor="k", linewidth=0.3, s=34)
    axs[0].hlines(np.nanmedian(_rv[_m]), _i - 0.3, _i + 0.3, color="k", lw=2)
axs[0].axhline(0, color="0.6", lw=0.8, ls="--")
axs[0].set_xticks(range(len(CLS4)))
axs[0].set_xticklabels(CLS4, rotation=30, ha="right", fontsize=10)
axs[0].set_ylabel(r"$A_V$ resid at fixed $\Sigma_{\rm dust}$ [mag]")
for _cl in CLS4:                                      # vs coupling strength
    _m = (_rc == _cl) & np.isfinite(_rv)
    _x = np.array([_xs_of.get(k, np.nan) for k in _rk])
    axs[1].scatter(_x[_m], _rv[_m], c=AGN_COLORS[_cl], edgecolor="k",
                   linewidth=0.3, s=34, label=_cl)
axs[1].axhline(0, color="0.6", lw=0.8, ls="--")
axs[1].set_xlabel(r"$x_{\rm str}$ (coupling strength over the quench window)")
axs[1].set_ylabel(r"$A_V$ resid at fixed $\Sigma_{\rm dust}$ [mag]")
axs[1].legend(fontsize=9, frameon=False)
fig.tight_layout()
_f = os.path.join(PLOTDIR, "p8_t4_residuals_xstr.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)

In [ ]:
# ── Part 8g — T5: can an observer SEE the red-core dust with resolved SED fits? ──
# Self-contained after Part 8c (+ Part 7f's cigale_pinned_results.fits). The
# observable red-core proxy is the aperture DIFFERENCE of the pinned-CIGALE fits,
#     dAv_obs = Av_corr(inner) - Av_corr(ap100kpc),   Av_corr = Av_ISM - Av_zp
# (per-aperture dust_off zero point, Part 7f), against the RT truth
#     dAv_true = A_V(inner) - A_V(ap100kpc).
# Remember the standing caveat: the pinned fit is a BEST case — a real free-SFH
# fit will do worse, so a null here is fatal, a success is only necessary.
from scipy.stats import spearmanr, kendalltau, mannwhitneyu

_resf = os.path.join(TABLEDIR, "cigale_pinned_results.fits")
if not os.path.exists(_resf):
    raise FileNotFoundError(f"{_resf} missing — run Parts 7d + 7f first")
_TC = Table.read(_resf)
_TC = _TC[np.char.strip(np.asarray(_TC["arm"], str)) == "dust_on"]
for _c in ("aperture", "incl"):
    _TC[_c] = np.char.strip(np.asarray(_TC[_c], str))
_avc = (np.asarray(_TC["bayes.attenuation.Av_ISM"], float)
        - np.asarray(_TC["Av_zp"], float))            # zero-point-corrected

def _obs_map(ap):
    _m = np.asarray(_TC["aperture"], str) == ap
    return {(int(s), int(g), str(i)): float(v) for s, g, i, v in
            zip(_TC["snap"][_m], _TC["gal_id"][_m], _TC["incl"][_m], _avc[_m])}

def _true_map(ap):
    _m = _ap8 == ap
    return {(int(s), int(g), str(i)): float(v) for s, g, i, v in
            zip(P8["snap"][_m], P8["gal_id"][_m], P8["incl"][_m],
                np.asarray(P8["A_V"], float)[_m])}

_o100, _t100 = _obs_map("ap100kpc"), _true_map("ap100kpc")
_mX5 = _ap8 == CORE_LABEL                             # xstr is per galaxy; any label works
_xsg5, _xsk5 = _gal_median(np.asarray(P8["xstr_quench"], float)[_mX5], _GKEY[_mX5])
_xs_of = dict(zip(_xsk5, _xsg5))
print("inner-aperture fit completeness per class (a class-dependent hole would "
      "bias the comparison):")
_T5 = {}
for _inner in ("ap1kpc", "ap3kpc"):
    _oi, _ti = _obs_map(_inner), _true_map(_inner)
    _keys = sorted(set(_t100) | set(_ti))
    _rows = []
    for _k in _keys:
        _do = _oi.get(_k, np.nan) - _o100.get(_k, np.nan)
        _dt = _ti.get(_k, np.nan) - _t100.get(_k, np.nan)
        _rows.append((f"{_k[0]}_{_k[1]}", _do, _dt))
    _gk = np.array([r[0] for r in _rows])
    _T5[_inner] = dict(gkey=_gk, cls=_cls_of_keys(_gk),
                       dAv_obs=np.array([r[1] for r in _rows]),
                       dAv_true=np.array([r[2] for r in _rows]))
    _t5 = _T5[_inner]
    for _cl in CLS4:
        _m = _t5["cls"] == _cl
        if _m.any():
            _fin = np.isfinite(_t5["dAv_obs"][_m])
            print(f"   {_inner} {_cl:>14s}: {int(_fin.sum())}/{int(_m.sum())} "
                  f"({np.mean(_fin) * 100:.0f}%)")

print()
for _inner in ("ap1kpc", "ap3kpc"):
    _t5 = _T5[_inner]
    _x, _y, _k = _t5["dAv_true"], _t5["dAv_obs"], _t5["gkey"]
    _f = np.isfinite(_x) & np.isfinite(_y)
    if _f.sum() < 8:
        print(f"[{_inner}] only {int(_f.sum())} rows with both obs and truth — skipped")
        continue
    _r, _lo, _hi = _gboot_idx(_k[_f], lambda i: (
        spearmanr(_x[_f][i], _y[_f][i])[0] if len(i) >= 5 else np.nan))
    print(f"[{_inner}] Spearman(dAv_obs, dAv_true) = {_r:+.2f} [{_lo:+.2f},{_hi:+.2f}] "
          f"(n = {int(_f.sum())})")
    # does the truth class RANKING survive in the observable?
    _mt, _mo = [], []
    for _cl in CLS4:
        _m = (_t5["cls"] == _cl)
        _vt, _kt = _gal_median(_t5["dAv_true"][_m], _k[_m])
        _vo, _ko = _gal_median(_t5["dAv_obs"][_m], _k[_m])
        _mt.append(np.nanmedian(_vt))
        _mo.append(np.nanmedian(_vo))
        print(f"      {_cl:>14s}: dAv_true med = {np.nanmedian(_vt):+.3f}   "
              f"dAv_obs med = {np.nanmedian(_vo):+.3f}")
    _tau, _ptau = kendalltau(_mt, _mo)
    print(f"      class-median ranking truth vs observable: Kendall tau = {_tau:+.2f}")
    _vo_all, _ko_all = _gal_median(_t5["dAv_obs"], _k)
    _co = _cls_of_keys(_ko_all)
    _a = _vo_all[(_co == "strong") & np.isfinite(_vo_all)]
    _b = _vo_all[(_co == "no_AGN") & np.isfinite(_vo_all)]
    if len(_a) >= 3 and len(_b) >= 3:
        print(f"      MW dAv_obs strong vs no_AGN: p = "
              f"{mannwhitneyu(_a, _b, alternative='two-sided')[1]:.3f}")

# ── figure ──
_t5 = _T5["ap1kpc"]
fig, axs = plt.subplots(1, 3, figsize=(21, 6.5))
for _cl in _cls_present:                               # truth vs observable
    _m = _t5["cls"] == _cl
    axs[0].scatter(_t5["dAv_true"][_m], _t5["dAv_obs"][_m], s=24,
                   c=AGN_COLORS[_cl], alpha=0.6, edgecolor="k", linewidth=0.25,
                   label=_cl)
_lim = np.nanpercentile(np.concatenate([_t5["dAv_true"], _t5["dAv_obs"]]), [1, 99])
axs[0].plot(_lim, _lim, "k--", lw=1)
axs[0].axhline(0, color="0.7", lw=0.7)
axs[0].axvline(0, color="0.7", lw=0.7)
axs[0].set_xlabel(r"$\Delta A_V^{\rm true}$ (ap1kpc $-$ ap100kpc) [mag]")
axs[0].set_ylabel(r"$\Delta A_V^{\rm CIGALE}$ (zp-corr) [mag]")
axs[0].legend(fontsize=9, frameon=False)
for _j, (_col, _ttl) in enumerate((("dAv_true", "truth"),
                                   ("dAv_obs", "observable (pinned CIGALE)"))):
    _ax = axs[1]
    _v_all, _k_all = _gal_median(_t5[_col], _t5["gkey"])
    _c_all = _cls_of_keys(_k_all)
    for _i, _cl in enumerate(CLS4):
        _m = (_c_all == _cl) & np.isfinite(_v_all)
        if not _m.any():
            continue
        _xj = 2 * _i + _j * 0.7 + np.random.default_rng(_i + _j).uniform(
            -0.12, 0.12, int(_m.sum()))
        _ax.scatter(_xj, _v_all[_m], c=AGN_COLORS[_cl], edgecolor="k",
                    linewidth=0.3, s=30, alpha=0.9 if _j else 0.45,
                    marker="o" if _j else "s")
        _ax.hlines(np.nanmedian(_v_all[_m]), 2 * _i + _j * 0.7 - 0.25,
                   2 * _i + _j * 0.7 + 0.25, color="k", lw=2)
axs[1].axhline(0, color="0.6", lw=0.8, ls="--")
axs[1].set_xticks([2 * _i + 0.35 for _i in range(len(CLS4))])
axs[1].set_xticklabels(CLS4, rotation=30, ha="right", fontsize=10)
axs[1].set_ylabel(r"central $\Delta A_V$ [mag]")
axs[1].set_title("squares: truth   circles: observable", fontsize=10)
_v_all, _k_all = _gal_median(_t5["dAv_obs"], _t5["gkey"])
_x_all = np.array([_xs_of.get(k, np.nan) for k in _k_all])
_c_all = _cls_of_keys(_k_all)
for _cl in CLS4:                                       # observable vs coupling strength
    _m = _c_all == _cl
    axs[2].scatter(_x_all[_m], _v_all[_m], c=AGN_COLORS[_cl], edgecolor="k",
                   linewidth=0.3, s=34, label=_cl)
axs[2].axhline(0, color="0.6", lw=0.8, ls="--")
axs[2].set_xlabel(r"$x_{\rm str}$ (coupling strength)")
axs[2].set_ylabel(r"$\Delta A_V^{\rm CIGALE}$ [mag]")
fig.tight_layout()
_f = os.path.join(PLOTDIR, "p8_t5_observability.png")
fig.savefig(_f, dpi=140, bbox_inches="tight")
plt.show()
print("saved", _f)

# Run order (cheat sheet)

1. **cluster** — `BUILD_MULTI_Z=True` → run Parts 0–1 (histories); then `BUILD_BH=True` → Part 1
   BH cell. Flip both back to `False` afterwards.
2. Parts 2–3 (selection, SFT/QT, AGN split, statistics → `powderday_quenched_selection.fits`) —
   needs only the HDF5s from step 1. Then **Part 3b** (mass–size QC): `flag_too_large` /
   `flag_unresolved` into `SELECTION_FITS` (carried into every Part 7 catalog).
3. **cluster** — Part 4 (Stage 0 particle files), apply the **powderday aperture patch** (Part 5
   markdown), Part 5 cell, then `bash submit_all_snaps.sh` in **all three** run trees under
   `output/cis25/sed_quenched_regions/<run_tag>/powderday_sed_out/` (`dusty_simdust`,
   `nodust_1e-12`, `dusty_simdust_agn`). Any time after Stage 0: **Part 4b** (star/gas/dust counts
   per projected annulus × sightline) and **Part 4c** (mass-weighted Z_star/Z_gas per aperture →
   `tables/aperture_metallicities.fits`, the Part 7d metallicity pins).
4. When `.rtout.sed` files exist: Part 6 QC (must show 5 apertures × 4 inclinations + MC
   uncertainties), then Part 7 → the rest-frame per-aperture catalogs. Then:
   - **Part 7a (required)** — the TRUE attenuation $A_V=-2.5\log_{10}F_{\rm on}/F_{\rm off}$ vs the
     ISM and quench/AGN diagnostics (`tables/attenuation_vs_ism.fits`, fiducial sightline; Parts
     4b/7f reuse its dusty flag). This is the only consumer of `dust_off` and the origin of the
     $A_V$ reference — no CIGALE needed.
   - **Part 7b (required)** — the observed-frame CIGALE inputs, all three arms × 5 cumulative
     apertures × 4 sightlines.
5. **Part 7c (required)** — the aperture-matched **formed-mass** SFH archive
   (`cigale/sfh_smoothed_aperture.h5`) that Part 7d injects one column of per run. Needs `fsps` in
   the kernel; confirm `sfh_mass_kind == 'formed'` in the printed attrs, or Part 7d refuses to
   build. Then **Part 7c2 (required reading)** — the shape cost of the galaxy-level pin
   (`tables/cigale_sfh_pin_shape.fits`): it raises if the `ap100kpc` SFH is not sightline-degenerate
   (which is what licenses Part 7d's sightline collapse), and it reports the inner-aperture age
   offset that the pin imposes.
6. **cluster** — **Part 7d**: one run dir per (snapshot, galaxy, chain, Z node) under
   `output/cis25/cigale_runs_pinned/` — a few hundred dirs, each fitting 20–40 SEDs against one
   shared pinned grid, and ONE SLURM array. All three arms are fitted: `dust_on` + `dust_off` share
   the dust chain (identical module chain → same run), `agn_on` gets its own (`skirtor2016`).
   Pre-flight with `PILOT=True` (one galaxy → 2 dirs) + `cg.check()` + one interactive `cg.run()`,
   then `sbatch cigale_runs_pinned/submit_cigale_pin.job`. `SKIP_IF_DONE=True` is the default, so a
   resubmit only runs the gaps.
7. **cluster** — **Part 7e**: `tables/aperture_truth.fits` (one pass over the Stage-0 cutouts, per
   galaxy × sightline × aperture/annulus). Then **Part 7f**: collect every `results.fits`, join the
   truth and the per-(aperture, sightline) $A_V$, write `tables/cigale_pinned_results.fits` +
   `cigale_pinned_av_stats.fits` and the figure set.

8. **Part 8 (red cores)** — **cluster**: **Part 8a** (`tables/annulus_ism_truth.fits`, one
   pass over the Stage-0 cutouts; first verify one cutout carries the PartType0 thermo
   fields: `h5ls -r output/cis25/filtered_particles/snap_*/m25n512_snap*_gal*.h5 | head`),
   then **Part 8b** (`tables/annulus_av_allincl.fits`, needs the Part 7 catalogs; asserts
   closure against Part 7a's fiducial-sightline `A_V_ann_*`). **Parts 8c–8f** are pure
   reads of the caches; **8g** additionally needs `cigale_pinned_results.fits` (step 7).

Old → new part labels (2026-08-10 prune): `7d2→7c`, `7e→7d`, `7f→7e`, `7g→7f`, plus the new `7c2`;
`7a-agn`, the old `7c` (annular CIGALE inputs), `7d` (delayed+bq priors), `7e2`, `7e3`, `7h`, `7i`,
`7j`, `7k` are deleted — the previous notebook is kept as `powderday_flux_quenched_m25.ipynb.pre-cleanup.bak`.
The useful half of `7a-agn` (per-band true $f_{\rm AGN}$ and the AGN-diluted $A_V$) is four lines
inside Part 7f; `7e2`/`7e3` are answered by construction (there is no SFH family, and `dust_on` has
no AGN module).

**Caveats.**
- **The pinned fit is a best case, not an observation.** With the SFH, the mass-weighted age and
  the metallicity all fixed from the simulation, the recovered $A_V$ is the *floor* of the
  systematic: a real fit with a free SFH will do worse. That is the point — it isolates dust from
  the age–metallicity–dust degeneracy — but the paper text has to say so.
- **The SPS-library mismatch is now the dominant systematic.** powderday renders with FSPS
  (MIST + MILES, Chabrier), CIGALE fits with BC03 (Padova94 + STELIB). At fixed age and $Z$ those
  differ by ~0.05–0.15 mag in optical colour, and $A_V$ is the only free knob left to absorb it.
  The `dust_off` set (truth $A_V\equiv0$) measures the resulting zero-point, now over the **full**
  aperture × sightline grid because it rides along in the dust chain at no extra grid cost — so
  Part 7f quotes it **per aperture** and stores `dAv_zpcorr`. Do not quote an $A_V$ offset smaller
  than it, and do not use the global number where the per-aperture ones disagree.
- **The SFH pin is per GALAXY, not per aperture.** `normalise=True` discards the amplitude, so
  apertures whose histories differ by a scale factor share the pin exactly; a difference in *shape*
  does not. Part 7c2 measures that shape difference — read
  `tables/cigale_sfh_pin_shape.fits` before quoting anything. A positive $\Delta t_{\rm mw}$ in the
  inner apertures means the pin is too blue there and the fit pays for it with dust, i.e. $A_V$
  biased **high**. `SFH_PIN_LEVEL = "aperture"` in Part 7d buys the strict per-aperture pin back at
  ~5× the runs (the sightline and arm collapses are exact and survive either way).
- **Coverage is now photometric, not stellar.** `SFH_NSTAR_MIN = 20` still excludes an aperture from
  the Part 7c archive, but the galaxy-level pin reads the SFH from the whole cutout, so a star-poor
  `ap1kpc` no longer loses its fit — only rows with fewer than `MIN_FIT_BANDS` finite fluxes drop.
  Read `tables/cigale_pinned_skipped.fits`; `ap1kpc` spans only ~3–4 softening lengths at m25n512
  and is still hit hardest.
- **`bc03.metallicity` is coarse where these galaxies live** (0.008 / 0.02 / 0.05, a factor 2.5
  apart) and a single pinned node marginalises over nothing. Rows are sub-grouped by their own node,
  so an aperture that straddles one costs an extra run rather than a wrong pin — Part 7d prints that
  census, and `Z_PIN_LEVEL = "galaxy"` collapses it if the split is not worth paying for. If
  $\Delta A_V$ comes out bimodal, colour it by `zs_idx`.
- Stage 0 writes **100 pkpc region cutouts** (CGM + satellites, periodic-wrap safe); the RT grid is
  ±100 kpc (`zoom_box_len`), and the 5 hyperion-log-spaced apertures (1, 3.16, 10, 31.6, 100 kpc)
  sample central → outskirts. Only the outermost is slightly depth-truncated at its edge (sphere
  inscribed in the cube). `N_AP/AP_MIN_KPC/AP_MAX_KPC` + `THETA_DEG/PHI_DEG` here must match
  `SED_APERTURE_*` / `THETA/PHI` in `simbanator/sed/parameters_master*.py` at RT time.
- Annuli are **never** fitted: an annulus does not contain the dust heated by the light it emits, so
  CIGALE's energy balance — the very pathway that sets $A_V$ — is ill-posed there. Part 4b/4c and
  Part 7e still measure annular quantities for the radial profiles.
- The **`agn_on` run** is `dust_on` + AGN point sources (`parameters_master-agn.py`: `BH_SED=True`,
  Hopkins+2007 intrinsic quasar template, `BH_var=False` so $L_{\rm bol}=0.1\,\dot M_{\rm BH}c^2$
  follows the SIMBA accretion rates; attenuated by the same live-dust grid). It needs `PartType5` in
  the Stage-0 cutouts — re-run Part 4 first. Only its chain carries `skirtor2016`, so a
  `dust_on`↔`agn_on` difference mixes real AGN light with the extra model freedom; Part 7f's
  `agn_bias.png` shows both together.
- `<filter>_err` is the Hyperion **Monte-Carlo photon noise** propagated through the filter
  convolution — an RT-convergence error, not a mock observational depth. All-NaN error columns mean
  the run stored no uncertainties (patch not applied / `set_uncertainties` missing).
- Part 7 fluxes are rest-frame convolved; Part 7b re-extracts observed-frame for CIGALE (same
  `extract_flux_set` helper, `redshift=True`). Part 7a's $A_V$ is therefore a **rest-frame**
  attenuation, directly comparable across anchors.
- The dust_off run uses 1 dust-RT photon (`parameters_master-nodust.py`) — some galaxies can
  crash/truncate; the Part 7 cross-check + `missing_sources_*.txt` make any loss explicit.
- CIGALE error budget: the input files carry raw MC errors; the fit adds `additionalerror = 0.1`
  (10 %) in quadrature via `prepare_run` — change it there, not in Part 7b.
- **Negative errors** = "upper limit" to CIGALE. Part 7b catalogs written before the
  `convolveFilterWithSED` sign fix are all-negative → all-NaN fits; Part 7d's preflight
  (`cigale.sanitize_input_errors`) repairs them in place.
- An **all-zero SFH column** would make `sfhfromfile`'s `normalise=True` divide by zero and produce
  an all-NaN `results.fits` that looks converged. `cigale.validate_sfh_file` (called from
  `prepare_run`) and Part 7d's own guard catch it; the affected objects appear in
  `cigale_pinned_skipped.fits` rather than silently vanishing.

- Part 8's $\Sigma$/DGR/$f_{\rm cold}$ are masked where a projected annulus holds fewer
  than `NGAS_ANN_MIN = 10` gas particles (Part 8a prints the fraction); the 1 kpc rung is
  the worst sampled, so every T2/T3 headline number is quoted at `ap3kpc` with the 1 kpc
  disc as the echo. The T2 screen uses the RT's own KMH94 opacity read from
  `$POWDERDAY_ROOT/hyperion-dust/dust_files/kmh94_3.1_hg.hdf5` — if Part 8a printed the
  MW-like fallback instead, the ratios carry a ~×2 opacity uncertainty.
